# **Decision Support System for Drug Distribution Risk Analysis Using Expert Knowledge in Bayesian Parameters**

Notebook ini adalah kode analisis artikel ("Decision Support System for Drug Distribution Risk Analysis Using Expert Knowledge in Bayesian Parameters").

Data: transaksi penyaluran e-WAS BBPOM Surabaya (208.162 baris mentah, 161.728 setelah pembersihan).

Model: Bernoulli Naive Bayes dengan prior 0,15 dan indikator E1, E2, E3 yang rasio kemungkinannya (LR) berasal dari pengetahuan pakar.

## **Peta bagian**
| Bagian | Isi | Keluaran utama |
|---|---|---|
| Persiapan | urutan menjalankan, versi pustaka, saklar, cek berkas wajib | versi dan SHA berkas |
| Pengumpulan data mentah | kode pengambilan data mentah pada gsheet | `data_mentah_bbpom.csv` |
| Pembersihan data | profil, flag kualitas, duplikat, audit penyaringan | `data_clean_bbpom.csv`, berkas audit |
| Audit data | satuan, jenis sarana, kategori obat, vial/ampul | `data_baru_clean.csv` |
| EDA dan lokasi E5 | profil, frekuensi, tanggal, flag, indikator lokasi | tabel EDA, `e5_lokasi.xlsx` |
| Model Bayesian | indikator, matriks bukti, kalibrasi LR, posterior, skema A/B | kategori Rendah/Sedang/Tinggi |
| Analisis OAT | sensitivitas satu parameter per kali | tabel OAT |
| Monte Carlo | simulasi ketidakpastian parameter | ringkasan Monte Carlo |
| Validasi pakar | pemetaan transaksi pakar, V1, V2, baseline, kappa | tabel kecocokan |
| Tabel dan gambar artikel | tabel dan gambar untuk naskah | tabel dan gambar |
| Tabel suplemen | S1-S4 | `tabel_suplemen_artikel.xlsx` |

## **Berkas yang dibutuhkan (satu folder dengan notebook)**
`data_mentah_bbpom.csv`, `kunci_sampel.xlsx`, `kunci_validasi_ulang.xlsx`, `lembar_penilaian_pakar_terisi.xlsx`, `lembar_validasi_ulang_terisi.xlsx`, `data_mentah_bbpom.csv`.

## **Catatan**
- Jalankan dari atas ke bawah. Bagian generator sampel berstrata sengaja tidak disertakan agar sampel yang sudah dinilai pakar tidak berubah.
- Data mentah memuat nama sarana dan pemasok sehingga tidak ikut dipublikasikan.

**URUTAN MENJALANKAN**
1. Letakkan `data_mentah_bbpom.csv`, `kunci_sampel.xlsx`, `kunci_validasi_ulang.xlsx`, `lembar_penilaian_pakar_terisi.xlsx`, dan `lembar_validasi_ulang_terisi.xlsx` di folder yang sama dengan notebook.
2. Jalankan semua sel dari atas (Run all). Alur: pembersihan, pengayaan, konversi vial/ampul, indikator, model, OAT, Monte Carlo, baseline, validasi pakar, Bagian 10 (tabel suplemen S1-S4).
3. **Jangan jalankan ulang sel generator sampel berstrata (Bagian 5)** bila `kunci_sampel.xlsx` dan `kunci_validasi_ulang.xlsx` sudah ada: sel itu membuat sampel dan kunci baru.

### **Persiapan**
Mencetak versi Python dan pustaka serta sidik jari (SHA-256) berkas data agar hasil dapat direproduksi.

In [ ]:
import sys, platform, hashlib, os
import numpy, pandas, matplotlib
try:
    import scipy
    _ver_scipy = scipy.__version__
except Exception:
    _ver_scipy = "tidak terpasang"
try:
    import openpyxl
    _ver_openpyxl = openpyxl.__version__
except Exception:
    _ver_openpyxl = "tidak terpasang"

print("Python      :", sys.version.split()[0], "|", platform.platform())
print("numpy       :", numpy.__version__)
print("pandas      :", pandas.__version__)
print("matplotlib  :", matplotlib.__version__)
print("scipy       :", _ver_scipy)
print("openpyxl    :", _ver_openpyxl)

for _berkas in ["data_baru_clean.csv", "kunci_sampel.xlsx", "kunci_validasi_ulang.xlsx",
                "lembar_penilaian_pakar_terisi.xlsx", "lembar_validasi_ulang_terisi.xlsx"]:
    if os.path.exists(_berkas):
        with open(_berkas, "rb") as _f:
            print(f"SHA-256 {_berkas}: {hashlib.sha256(_f.read()).hexdigest()[:16]}...")
    else:
        print(f"{_berkas}: tidak ditemukan di folder ini")

Saklar dan pemeriksaan berkas wajib: memastikan empat berkas validasi tersedia, dengan jumlah baris dan SHA yang sesuai.

In [ ]:
import os, hashlib
import pandas as pd

PAKAI_SNAPSHOT = True

BERKAS_WAJIB = [
    ("kunci_sampel.xlsx",                "Kunci",     112, 100, "f10f99e6e1a60a1a"),
    ("kunci_validasi_ulang.xlsx",        "Kunci",      75,  60, "90a4d6bf3d6dd108"),
    ("lembar_penilaian_pakar_terisi.xlsx","Penilaian", 112, None, "62cc107136b3e010"),
    ("lembar_validasi_ulang_terisi.xlsx", "Penilaian",  75, None, "1a6134c118bc4611"),
]
masalah = []
for nama, sheet, n_baris, n_unik, awalan in BERKAS_WAJIB:
    if not os.path.exists(nama):
        masalah.append(f"{nama}: TIDAK ADA di folder {os.getcwd()}")
        continue
    with open(nama, "rb") as f:
        h = hashlib.sha256(f.read()).hexdigest()
    t = pd.read_excel(nama, sheet_name=sheet)
    info = f"{nama}: {len(t)} baris"
    if n_unik is not None:
        info += f", {int((t['Duplikat uji ulang'] != 'ya').sum())} unik"
    ok = (len(t) == n_baris) and (n_unik is None or int((t['Duplikat uji ulang'] != 'ya').sum()) == n_unik)
    if h[:16] != awalan:
        ok = False
    print(("OK   " if ok else "SALAH"), info, "| SHA", h[:16])
    if not ok:
        masalah.append(f"{nama}: isi berbeda dari berkas yang dipakai di artikel (harus {n_baris} baris, SHA {awalan}...)")
for nama in ["data_mentah_bbpom.csv"] + (["data_baru_clean.csv"] if PAKAI_SNAPSHOT else []):
    if not os.path.exists(nama):
        masalah.append(f"{nama}: TIDAK ADA")
    else:
        print("OK   ", nama)
if masalah:
    raise FileNotFoundError("Perbaiki dahulu:\n- " + "\n- ".join(masalah))
print("\nSemua berkas wajib lengkap dan sesuai.")

MENGUMPULKAN DATA MENTAH

Bagian ini untuk pengumpulan data mentah dari sumber.

In [ ]:
import pandas as pd
from google.colab import files
import io
import os

KOLOM_FINAL = [
    "Jenis Transaksi",
    "Tujuan Penyaluran",
    "Alamat Tujuan",
    "Nama Kota/Kab Tujuan",
    "Nama Provinsi Tujuan",
    "Nama Zat Aktif",
    "Nama Obat Jadi",
    "Produsen Obat Jadi",
    "Nama PBF",
    "Provinsi",
    "Kabupaten/Kota",
    "No. Faktur",
    "Tanggal Penyaluran",
    "Batch",
    "Jumlah",
    "Satuan",
    "Tanggal Kedaluwarsa",
    "Keterangan"
]


RENAME_COLUMNS = {
    "Nama Provinsi TUjuan": "Nama Provinsi Tujuan",
    "Nama Provinsi tujuan": "Nama Provinsi Tujuan",
    "Nama Provinsi TUJUAN": "Nama Provinsi Tujuan",
    "Nama Provinsi Tujuan ": "Nama Provinsi Tujuan",
    " Nama Provinsi Tujuan": "Nama Provinsi Tujuan"
}

print("=" * 70)
print("PENGGABUNGAN 5 FILE CSV")
print("=" * 70)

print("\nSilakan pilih tepat 5 file CSV sekaligus.")

uploaded = files.upload()

if len(uploaded) != 5:
    raise ValueError(
        f"File yang dipilih ada {len(uploaded)}. "
        "Silakan upload tepat 5 file CSV."
    )

print(f"\n✓ {len(uploaded)} file berhasil dipilih.")

dataframes = []
laporan = []

for nama_file, isi_file in uploaded.items():

    print("\n" + "-" * 70)
    print(f"Memproses: {nama_file}")

    try:
        df = pd.read_csv(
            io.BytesIO(isi_file),
            encoding="utf-8-sig"
        )

        jumlah_baris_awal = len(df)
        jumlah_kolom_awal = len(df.columns)

        print(
            f"Jumlah baris awal  : {jumlah_baris_awal:,}"
        )

        print(
            f"Jumlah kolom awal  : {jumlah_kolom_awal}"
        )

        df.columns = df.columns.str.strip()

        df.rename(
            columns=RENAME_COLUMNS,
            inplace=True
        )

        kolom_sarana = (
            "Jenis Sarana "
            "(Apt, KL, RS, PKM, Dinkes, PBF)"
        )

        if kolom_sarana in df.columns:

            df.drop(
                columns=[kolom_sarana],
                inplace=True
            )

            print(
                "✓ Kolom 'Jenis Sarana' dihapus"
            )

        kolom_hilang = [
            kolom
            for kolom in KOLOM_FINAL
            if kolom not in df.columns
        ]

        if kolom_hilang:

            print("\n⚠ Kolom yang tidak ditemukan:")

            for kolom in kolom_hilang:
                print(f"   - {kolom}")
                df[kolom] = pd.NA

        kolom_tambahan = [
            kolom
            for kolom in df.columns
            if kolom not in KOLOM_FINAL
        ]

        if kolom_tambahan:

            print("\n⚠ Kolom tambahan:")

            for kolom in kolom_tambahan:
                print(f"   - {kolom}")

        df = df[KOLOM_FINAL]

        dataframes.append(df)

        laporan.append({
            "Nama File": nama_file,
            "Jumlah Baris": jumlah_baris_awal,
            "Jumlah Kolom Awal": jumlah_kolom_awal,
            "Kolom Hilang": (
                ", ".join(kolom_hilang)
                if kolom_hilang
                else "-"
            ),
            "Kolom Tambahan": (
                ", ".join(kolom_tambahan)
                if kolom_tambahan
                else "-"
            )
        })

        print(
            f"✓ File berhasil diproses"
        )

    except Exception as e:

        raise Exception(
            f"Gagal memproses {nama_file}: {e}"
        )


print("\n" + "=" * 70)
print("MENGGABUNGKAN DATA")
print("=" * 70)

hasil = pd.concat(
    dataframes,
    ignore_index=True
)

hasil = hasil[KOLOM_FINAL]

total_baris_sumber = sum(
    item["Jumlah Baris"]
    for item in laporan
)

total_baris_hasil = len(hasil)

print(f"\nJumlah file        : {len(dataframes)}")
print(f"Total baris sumber : {total_baris_sumber:,}")
print(f"Total baris hasil  : {total_baris_hasil:,}")
print(f"Total kolom hasil  : {len(hasil.columns)}")

if total_baris_sumber == total_baris_hasil:
    print("✓ Tidak ada baris yang hilang.")
else:
    print("⚠ Jumlah baris berbeda. Perlu pemeriksaan.")

if list(hasil.columns) == KOLOM_FINAL:
    print("✓ Struktur 18 kolom sesuai standar.")
else:
    print("✗ Struktur kolom tidak sesuai.")

OUTPUT_FILE = "data_mentah_bbpom.csv"

hasil.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

laporan_df = pd.DataFrame(laporan)

LAPORAN_FILE = "Laporan.csv"

laporan_df.to_csv(
    LAPORAN_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 70)
print("PENGGABUNGAN SELESAI")
print("=" * 70)

print(f"File hasil   : {OUTPUT_FILE}")
print(f"Laporan      : {LAPORAN_FILE}")
print(f"Jumlah baris : {len(hasil):,}")
print(f"Jumlah kolom : {len(hasil.columns)}")

print("\nKolom hasil:")

for i, kolom in enumerate(
    hasil.columns,
    start=1
):
    print(f"{i:2}. {kolom}")

print("\nMenyiapkan file untuk didownload...")

files.download(OUTPUT_FILE)

CLEANING DATA

### **Pembersihan data**
Konfigurasi jalur berkas dan kolom sumber.

In [ ]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

BASE_DIR = Path.cwd()

SOURCE_PATH = BASE_DIR / "data_mentah_bbpom.csv"
OUTPUT_DIR = BASE_DIR / "hasil_cleaning_bbpom"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Konfigurasi siap.")
print("Direktori proyek :", BASE_DIR)
print("Sumber           :", SOURCE_PATH)
print("Output           :", OUTPUT_DIR)

SLASH_ORDER = "mdy"

SOURCE_COLUMNS = [
    "Jenis Transaksi",
    "Tujuan Penyaluran",
    "Alamat Tujuan",
    "Nama Kota/Kab Tujuan",
    "Nama Provinsi Tujuan",
    "Nama Zat Aktif",
    "Nama Obat Jadi",
    "Produsen Obat Jadi",
    "Nama PBF",
    "Provinsi",
    "Kabupaten/Kota",
    "No. Faktur",
    "Tanggal Penyaluran",
    "Batch",
    "Jumlah",
    "Satuan",
    "Tanggal Kedaluwarsa",
    "Keterangan",
]

RENAME_COLUMNS = {
    "Nama Provinsi TUjuan": "Nama Provinsi Tujuan",
    "Nama Provinsi tujuan": "Nama Provinsi Tujuan",
    "Nama Provinsi TUJUAN": "Nama Provinsi Tujuan",
    "Nama Provinsi Tujuan ": "Nama Provinsi Tujuan",
    " Nama Provinsi Tujuan": "Nama Provinsi Tujuan",
}

NULL_TOKENS = {
    "", "nan", "none", "null", "-", "@", "n/a", "na", "<na>"
}

print("Konfigurasi siap.")
print("Sumber :", SOURCE_PATH)
print("Output :", OUTPUT_DIR)

if not SOURCE_PATH.exists():
    raise FileNotFoundError(
        f"File dataset tidak ditemukan: {SOURCE_PATH}"
    )

print("\nFile dataset berhasil ditemukan.")

Membaca CSV mentah sebagai teks agar bentuk aslinya tidak berubah (mencoba utf-8-sig, utf-8, latin1), lalu merapikan nama kolom.

In [ ]:
def baca_csv_robust(path):
    """Baca CSV sebagai string agar representasi mentah tidak hilang."""
    encodings = ["utf-8-sig", "utf-8", "latin1"]
    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(
                path,
                dtype="string",
                keep_default_na=False,
                encoding=encoding,
                low_memory=False,
            )
        except UnicodeDecodeError as exc:
            last_error = exc

    raise last_error

if not SOURCE_PATH.exists():
    raise FileNotFoundError(
        f"""
File dataset tidak ditemukan!

Lokasi yang dicari:
{SOURCE_PATH}

Pastikan file data_mentah_bbpom.csv berada
di dalam folder proyek VS Code.
"""
    )

print("File dataset ditemukan.")
print("Lokasi:", SOURCE_PATH)

df_raw = baca_csv_robust(SOURCE_PATH)

df_raw.columns = [str(c).strip() for c in df_raw.columns]

df_raw = df_raw.rename(columns=RENAME_COLUMNS)

missing_columns = [
    c for c in SOURCE_COLUMNS
    if c not in df_raw.columns
]

if missing_columns:
    raise ValueError(
        "Kolom wajib tidak ditemukan:\n- "
        + "\n- ".join(missing_columns)
    )

df_raw = df_raw[SOURCE_COLUMNS].copy()

print("=" * 70)
print("DATA MENTAH BERHASIL DIBACA")
print("=" * 70)

print(f"Jumlah baris : {len(df_raw):,}")
print(f"Jumlah kolom : {df_raw.shape[1]}")

print("\n5 baris pertama:")

print(df_raw.head().to_string(index=False))

Profil kondisi awal data mentah.

In [ ]:
def is_missing_like(series):
    teks = series.astype("string").fillna("").str.strip().str.casefold()
    return teks.isin(NULL_TOKENS)

profil_awal = pd.DataFrame({
    "Kolom": SOURCE_COLUMNS,
    "Jumlah Kosong/Null-like": [
        int(is_missing_like(df_raw[c]).sum()) for c in SOURCE_COLUMNS
    ],
})

profil_awal["Persentase (%)"] = (
    profil_awal["Jumlah Kosong/Null-like"] / len(df_raw) * 100
).round(3)

duplicate_raw_exact = df_raw.duplicated(keep="first")

print(f"Baris mentah                    : {len(df_raw):,}")
print(f"Duplikat mentah exact           : {duplicate_raw_exact.sum():,}")
print("\nProfil missing/null-like:")
display(profil_awal)

Fungsi pembersihan: normalisasi teks serta pembaca nilai jumlah dan tanggal.

In [ ]:
import numpy as np
import pandas as pd

def normalize_text(series):
    """
    Standardisasi ringan:
    - menjadi pandas StringDtype
    - trim kiri/kanan
    - multiple whitespace -> satu spasi
    - token kosong umum -> <NA>

    Tidak mengubah isi menjadi lowercase.
    """

    s = (
        series.astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    missing = s.fillna("").str.casefold().isin(NULL_TOKENS)

    return s.mask(missing, pd.NA)

def canonical_text(series):
    """
    Versi pembanding untuk filter/duplicate;
    tidak mengganti nilai final.
    """

    return (
        normalize_text(series)
        .fillna("")
        .str.casefold()
    )

def is_integer_numeric(values):
    """
    Memeriksa apakah nilai numerik merupakan bilangan bulat.

    Aman untuk Pandas nullable Float64 dan nilai missing.
    """

    arr = values.to_numpy(
        dtype="float64",
        na_value=np.nan
    )

    hasil = (
        np.isfinite(arr)
        & np.isclose(
            arr,
            np.round(arr),
            atol=1e-12
        )
    )

    return pd.Series(
        hasil,
        index=values.index,
        dtype="bool"
    )

def parse_jumlah(series):
    """
    Aturan cleaning kolom Jumlah:

    KEEP:
    - 1       -> 1
    - 1000    -> 1000
    - 1.0     -> 1
    - 2.0     -> 2
    - 1000.0  -> 1000
    - 1,0     -> 1
    - 1,000   -> 1000
    - 12,000  -> 12000
    - 123,456 -> 123456
    - 1e3     -> 1000 jika hasil akhirnya integer

    DROP:
    - 0
    - 0.0
    - nilai negatif
    - 0.3
    - 0,3
    - 0.33
    - 0,33
    - 1.33
    - 1,33
    - seluruh pecahan non-zero

    REVIEW:
    - format yang benar-benar tidak dapat dikenali
    - missing/null token

    Return:
    hasil  : Int64
    status : string
    """

    raw = (
        series
        .astype("string")
        .fillna("")
        .str.strip()
        .str.replace(r"\s+", "", regex=True)
    )

    hasil = pd.Series(
        pd.NA,
        index=series.index,
        dtype="Int64"
    )

    status = pd.Series(
        "invalid_format",
        index=series.index,
        dtype="string"
    )

    missing_tokens = {
        "",
        "-",
        "@",
        "na",
        "n/a",
        "nan",
        "null",
        "none",
        "<na>"
    }

    mask_missing = (
        raw.str.casefold()
        .isin(missing_tokens)
    )

    status.loc[mask_missing] = "missing"

    mask_thousand_comma = raw.str.fullmatch(
        r"[+-]?[1-9]\d{0,2}(?:,\d{3})+",
        na=False
    )

    nilai_thousand = pd.to_numeric(
        raw.where(mask_thousand_comma)
        .str.replace(",", "", regex=False),
        errors="coerce"
    )

    mask_thousand_positive = (
        mask_thousand_comma
        & nilai_thousand.notna()
        & (nilai_thousand > 0)
    )

    mask_thousand_zero = (
        mask_thousand_comma
        & nilai_thousand.notna()
        & (nilai_thousand == 0)
    )

    mask_thousand_negative = (
        mask_thousand_comma
        & nilai_thousand.notna()
        & (nilai_thousand < 0)
    )

    hasil.loc[
        mask_thousand_comma & nilai_thousand.notna()
    ] = (
        nilai_thousand.loc[
            mask_thousand_comma & nilai_thousand.notna()
        ]
        .round()
        .astype("int64")
    )

    status.loc[
        mask_thousand_positive
    ] = "valid_thousands_comma"

    status.loc[
        mask_thousand_zero
    ] = "zero_thousands_comma"

    status.loc[
        mask_thousand_negative
    ] = "negative_thousands_comma"

    mask_integer = (
        raw.str.fullmatch(
            r"[+-]?\d+",
            na=False
        )
        & ~mask_missing
    )

    nilai_integer = pd.to_numeric(
        raw.where(mask_integer),
        errors="coerce"
    )

    mask_integer_positive = (
        mask_integer
        & nilai_integer.notna()
        & (nilai_integer > 0)
    )

    mask_integer_zero = (
        mask_integer
        & nilai_integer.notna()
        & (nilai_integer == 0)
    )

    mask_integer_negative = (
        mask_integer
        & nilai_integer.notna()
        & (nilai_integer < 0)
    )

    hasil.loc[
        mask_integer & nilai_integer.notna()
    ] = (
        nilai_integer.loc[
            mask_integer & nilai_integer.notna()
        ]
        .round()
        .astype("int64")
    )

    status.loc[
        mask_integer_positive
    ] = "valid_integer"

    status.loc[
        mask_integer_zero
    ] = "zero_integer"

    status.loc[
        mask_integer_negative
    ] = "negative_integer"

    mask_decimal_dot = raw.str.fullmatch(
        r"[+-]?\d+\.\d+",
        na=False
    )

    nilai_dot = pd.to_numeric(
        raw.where(mask_decimal_dot),
        errors="coerce"
    )

    mask_dot_integer = (
        mask_decimal_dot
        & nilai_dot.notna()
        & is_integer_numeric(nilai_dot)
    )

    mask_dot_fractional = (
        mask_decimal_dot
        & nilai_dot.notna()
        & ~mask_dot_integer
    )

    mask_dot_positive_integer = (
        mask_dot_integer
        & (nilai_dot > 0)
    )

    mask_dot_zero = (
        mask_dot_integer
        & (nilai_dot == 0)
    )

    mask_dot_negative_integer = (
        mask_dot_integer
        & (nilai_dot < 0)
    )

    hasil.loc[mask_dot_integer] = (
        nilai_dot.loc[mask_dot_integer]
        .round()
        .astype("int64")
    )

    status.loc[
        mask_dot_positive_integer
    ] = "valid_integer_decimal_dot"

    status.loc[
        mask_dot_zero
    ] = "zero_decimal_dot"

    status.loc[
        mask_dot_negative_integer
    ] = "negative_integer_decimal_dot"

    status.loc[
        mask_dot_fractional
    ] = "fractional_decimal_dot"

    mask_decimal_comma = (
        raw.str.fullmatch(
            r"[+-]?\d+,\d+",
            na=False
        )
        & ~mask_thousand_comma
    )

    nilai_comma = pd.to_numeric(
        raw.where(mask_decimal_comma)
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )

    mask_comma_integer = (
        mask_decimal_comma
        & nilai_comma.notna()
        & is_integer_numeric(nilai_comma)
    )

    mask_comma_fractional = (
        mask_decimal_comma
        & nilai_comma.notna()
        & ~mask_comma_integer
    )

    mask_comma_positive_integer = (
        mask_comma_integer
        & (nilai_comma > 0)
    )

    mask_comma_zero = (
        mask_comma_integer
        & (nilai_comma == 0)
    )

    mask_comma_negative_integer = (
        mask_comma_integer
        & (nilai_comma < 0)
    )

    hasil.loc[mask_comma_integer] = (
        nilai_comma.loc[mask_comma_integer]
        .round()
        .astype("int64")
    )

    status.loc[
        mask_comma_positive_integer
    ] = "valid_integer_decimal_comma"

    status.loc[
        mask_comma_zero
    ] = "zero_decimal_comma"

    status.loc[
        mask_comma_negative_integer
    ] = "negative_integer_decimal_comma"

    status.loc[
        mask_comma_fractional
    ] = "fractional_decimal_comma"

    mask_scientific = raw.str.fullmatch(
        r"[+-]?\d+(?:\.\d+)?[eE][+-]?\d+",
        na=False
    )

    nilai_scientific = pd.to_numeric(
        raw.where(mask_scientific),
        errors="coerce"
    )

    mask_scientific_integer = (
        mask_scientific
        & nilai_scientific.notna()
        & is_integer_numeric(nilai_scientific)
    )

    mask_scientific_fractional = (
        mask_scientific
        & nilai_scientific.notna()
        & ~mask_scientific_integer
    )

    mask_scientific_positive = (
        mask_scientific_integer
        & (nilai_scientific > 0)
    )

    mask_scientific_zero = (
        mask_scientific_integer
        & (nilai_scientific == 0)
    )

    mask_scientific_negative = (
        mask_scientific_integer
        & (nilai_scientific < 0)
    )

    hasil.loc[mask_scientific_integer] = (
        nilai_scientific.loc[
            mask_scientific_integer
        ]
        .round()
        .astype("int64")
    )

    status.loc[
        mask_scientific_positive
    ] = "valid_integer_scientific"

    status.loc[
        mask_scientific_zero
    ] = "zero_scientific"

    status.loc[
        mask_scientific_negative
    ] = "negative_integer_scientific"

    status.loc[
        mask_scientific_fractional
    ] = "fractional_scientific"

    return hasil, status

def parse_tanggal_mdy(series):
    """
    Parse tanggal dengan prioritas MDY untuk slash-date.
    Nilai tidak terbaca menjadi NaT.
    """

    raw = normalize_text(series)

    try:
        parsed = pd.to_datetime(
            raw,
            format="mixed",
            dayfirst=False,
            errors="coerce"
        )

    except (TypeError, ValueError):
        parsed = pd.to_datetime(
            raw,
            dayfirst=False,
            errors="coerce"
        )

    return parsed

contoh = pd.Series(
    [
        "1,000",
        "1000.0",
        "0.3",
        "0,3",
        "0",
        "10",
        "-1",
        "abc",
        "",
        "1.0",
        "1,0",
        "1e3",
        "1.5e1",
        "1.2e0"
    ],
    dtype="string"
)

uji_jumlah, uji_status = parse_jumlah(contoh)

hasil_uji = pd.DataFrame({
    "Input": contoh,
    "Hasil": uji_jumlah,
    "Status": uji_status
})

display(hasil_uji)

Standardisasi dan pembentukan flag kualitas: bukan transaksi dalam negeri, transaksi nihil, jumlah tidak valid, dan jumlah ≤ 0.

In [ ]:
df = df_raw.copy()

df.insert(0, "row_id_sumber", np.arange(2, len(df) + 2))

df["Jumlah Raw"] = df["Jumlah"].astype("string")
df["Tanggal Penyaluran Raw"] = df["Tanggal Penyaluran"].astype("string")
df["Tanggal Kedaluwarsa Raw"] = df["Tanggal Kedaluwarsa"].astype("string")

for col in SOURCE_COLUMNS:
    if col not in ["Jumlah", "Tanggal Penyaluran", "Tanggal Kedaluwarsa"]:
        df[col] = normalize_text(df[col])

df["Jumlah"], df["status_jumlah"] = parse_jumlah(df["Jumlah Raw"])

df["Tanggal Penyaluran"] = parse_tanggal_mdy(df["Tanggal Penyaluran Raw"])
df["Tanggal Kedaluwarsa"] = parse_tanggal_mdy(df["Tanggal Kedaluwarsa Raw"])

raw_tgl_penyaluran_missing = is_missing_like(df["Tanggal Penyaluran Raw"])
raw_tgl_kedaluwarsa_missing = is_missing_like(df["Tanggal Kedaluwarsa Raw"])

df["flag_tanggal_penyaluran_invalid"] = (
    ~raw_tgl_penyaluran_missing & df["Tanggal Penyaluran"].isna()
)

df["flag_tanggal_kedaluwarsa_invalid"] = (
    ~raw_tgl_kedaluwarsa_missing & df["Tanggal Kedaluwarsa"].isna()
)

df["flag_kedaluwarsa_sebelum_penyaluran"] = (
    df["Tanggal Penyaluran"].notna()
    & df["Tanggal Kedaluwarsa"].notna()
    & (df["Tanggal Kedaluwarsa"] < df["Tanggal Penyaluran"])
)

df["selisih_hari_kedaluwarsa"] = (
    df["Tanggal Kedaluwarsa"] - df["Tanggal Penyaluran"]
).dt.days.astype("Int64")

jenis_norm = canonical_text(df["Jenis Transaksi"])
ket_norm = canonical_text(df["Keterangan"])

df["flag_bukan_dalam_negeri"] = ~jenis_norm.eq("dalam negeri")
df["flag_transaksi_nihil"] = ket_norm.eq("transaksi nihil")
df["flag_jumlah_invalid"] = df["Jumlah"].isna()
df["flag_jumlah_nonpositif"] = (
    df["Jumlah"].notna() & (df["Jumlah"] <= 0)
)

print("Standardisasi dan flag kualitas selesai.")
display(
    df[
        [
            "row_id_sumber",
            "Jumlah Raw",
            "Jumlah",
            "status_jumlah",
            "Tanggal Penyaluran Raw",
            "Tanggal Penyaluran",
            "Tanggal Kedaluwarsa Raw",
            "Tanggal Kedaluwarsa",
        ]
    ].head(10)
)

Deteksi duplikat setelah standardisasi atribut (18 atribut sumber).

In [ ]:
def build_duplicate_key(frame):
    """
    Membentuk key duplicate dari 18 atribut sumber.
    """
    key = pd.DataFrame(index=frame.index)

    for col in SOURCE_COLUMNS:
        if col == "Jumlah":
            valid = frame["Jumlah"].notna()
            key[col] = frame["Jumlah"].astype("string")
            key.loc[~valid, col] = normalize_text(
                frame.loc[~valid, "Jumlah Raw"]
            ).fillna("")

        elif col == "Tanggal Penyaluran":
            valid = frame["Tanggal Penyaluran"].notna()
            key[col] = frame["Tanggal Penyaluran"].dt.strftime("%Y-%m-%d")
            key.loc[~valid, col] = normalize_text(
                frame.loc[~valid, "Tanggal Penyaluran Raw"]
            ).fillna("")

        elif col == "Tanggal Kedaluwarsa":
            valid = frame["Tanggal Kedaluwarsa"].notna()
            key[col] = frame["Tanggal Kedaluwarsa"].dt.strftime("%Y-%m-%d")
            key.loc[~valid, col] = normalize_text(
                frame.loc[~valid, "Tanggal Kedaluwarsa Raw"]
            ).fillna("")

        else:
            key[col] = normalize_text(frame[col]).fillna("")

    return key

base_keep = ~(
    df["flag_bukan_dalam_negeri"]
    | df["flag_transaksi_nihil"]
    | df["flag_jumlah_invalid"]
    | df["flag_jumlah_nonpositif"]
)

dup_key = build_duplicate_key(df)

df["flag_duplicate"] = False
df.loc[base_keep, "flag_duplicate"] = (
    dup_key.loc[base_keep].duplicated(keep="first")
)

print(f"Kandidat yang lolos filter dasar : {base_keep.sum():,}")
print(f"Duplikat yang akan dihapus       : {df['flag_duplicate'].sum():,}")

Audit penyaringan bertahap dan pembentukan data bersih (dasar Tabel 1 artikel).

In [ ]:
FILTER_STAGES = [
    ("Bukan Jenis Transaksi Dalam Negeri", "flag_bukan_dalam_negeri"),
    ("Keterangan = transaksi nihil", "flag_transaksi_nihil"),
    ("Jumlah invalid / pecahan / tidak terbaca", "flag_jumlah_invalid"),
    ("Jumlah <= 0", "flag_jumlah_nonpositif"),
    ("Duplikat setelah standardisasi", "flag_duplicate"),
]

ringkasan_independen = pd.DataFrame([
    {
        "Alasan": label,
        "Jumlah Baris": int(df[flag].sum()),
        "Persentase dari Raw (%)": round(df[flag].mean() * 100, 4),
    }
    for label, flag in FILTER_STAGES
])

active = pd.Series(True, index=df.index)

audit_rows = [{
    "Tahap": "Data mentah",
    "Sebelum": len(df),
    "Dihapus pada Tahap Ini": 0,
    "Sesudah": len(df),
}]

for label, flag in FILTER_STAGES:
    before = int(active.sum())
    removed_now = active & df[flag]
    active = active & ~df[flag]
    after = int(active.sum())

    audit_rows.append({
        "Tahap": label,
        "Sebelum": before,
        "Dihapus pada Tahap Ini": int(removed_now.sum()),
        "Sesudah": after,
    })

audit_df = pd.DataFrame(audit_rows)

flag_to_reason = {
    "flag_bukan_dalam_negeri": "bukan_dalam_negeri",
    "flag_transaksi_nihil": "transaksi_nihil",
    "flag_jumlah_invalid": "jumlah_invalid_atau_pecahan",
    "flag_jumlah_nonpositif": "jumlah_nonpositif",
    "flag_duplicate": "duplicate",
}

def gabung_alasan(row):
    alasan = [
        reason
        for flag, reason in flag_to_reason.items()
        if bool(row[flag])
    ]
    return "; ".join(alasan)

df["alasan_dikeluarkan"] = df.apply(gabung_alasan, axis=1)

df_excluded = df.loc[~active].copy()
df_clean_audit = df.loc[active].copy()

df_clean = df_clean_audit[
    ["row_id_sumber"] + SOURCE_COLUMNS
].copy()

print("=" * 70)
print("AUDIT CLEANING")
print("=" * 70)
display(audit_df)

print("\nJumlah per alasan (independen; dapat overlap):")
display(ringkasan_independen)

print(f"\nFINAL DATA CLEAN : {len(df_clean):,} baris")

Pemeriksaan format jumlah dengan pembaca nilai yang sama.

In [ ]:
jumlah_bersih_check, status_jumlah_check = parse_jumlah(
    df_raw["Jumlah"]
)

profil_jumlah = pd.DataFrame({
    "Jumlah Asli": (
        df_raw["Jumlah"]
        .astype("string")
        .str.strip()
    ),
    "Jumlah Setelah Konversi": jumlah_bersih_check,
    "Status": status_jumlah_check
})

profil_jumlah["Keputusan"] = "REVIEW"

mask_keep = (
    profil_jumlah["Jumlah Setelah Konversi"].notna()
    & (
        profil_jumlah["Jumlah Setelah Konversi"] > 0
    )
)

profil_jumlah.loc[
    mask_keep,
    "Keputusan"
] = "KEEP"

mask_nonpositive = (
    profil_jumlah["Jumlah Setelah Konversi"].notna()
    & (
        profil_jumlah["Jumlah Setelah Konversi"] <= 0
    )
)

profil_jumlah.loc[
    mask_nonpositive,
    "Keputusan"
] = "DROP"

mask_fractional = (
    profil_jumlah["Status"]
    .str.startswith("fractional", na=False)
)

profil_jumlah.loc[
    mask_fractional,
    "Keputusan"
] = "DROP"

mask_review = profil_jumlah["Status"].isin([
    "missing",
    "invalid_format"
])

profil_jumlah.loc[
    mask_review,
    "Keputusan"
] = "REVIEW"

label_status = {

    "valid_integer":
        "Bilangan bulat positif",

    "valid_integer_decimal_dot":
        "Bilangan bulat tertulis sebagai desimal titik",

    "valid_integer_decimal_comma":
        "Bilangan bulat tertulis sebagai desimal koma",

    "valid_thousands_comma":
        "Format ribuan koma",

    "valid_integer_scientific":
        "Bilangan bulat dalam notasi ilmiah",

    "zero_integer":
        "Nol — bilangan bulat",

    "zero_decimal_dot":
        "Nol — tertulis sebagai desimal titik",

    "zero_decimal_comma":
        "Nol — tertulis sebagai desimal koma",

    "zero_scientific":
        "Nol — notasi ilmiah",

    "negative_integer":
        "Bilangan negatif",

    "negative_integer_decimal_dot":
        "Bilangan negatif tertulis desimal titik",

    "negative_integer_decimal_comma":
        "Bilangan negatif tertulis desimal koma",

    "negative_thousands_comma":
        "Bilangan negatif format ribuan koma",

    "negative_integer_scientific":
        "Bilangan negatif notasi ilmiah",

    "fractional_decimal_dot":
        "Pecahan desimal titik",

    "fractional_decimal_comma":
        "Pecahan desimal koma",

    "fractional_scientific":
        "Pecahan notasi ilmiah",

    "missing":
        "Kosong / missing",

    "invalid_format":
        "Format tidak dikenali"
}

profil_jumlah["Interpretasi"] = (
    profil_jumlah["Status"]
    .map(label_status)
    .fillna(profil_jumlah["Status"])
)

ringkasan_jumlah = (
    profil_jumlah
    .groupby(
        [
            "Interpretasi",
            "Keputusan"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="Jumlah Baris")
)

ringkasan_jumlah["Persentase"] = (
    ringkasan_jumlah["Jumlah Baris"]
    / len(profil_jumlah)
    * 100
).round(3)

ringkasan_jumlah = (
    ringkasan_jumlah
    .sort_values(
        ["Keputusan", "Jumlah Baris"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

print("=" * 80)
print("RINGKASAN FORMAT JUMLAH")
print("=" * 80)

display(ringkasan_jumlah)

ringkasan_keputusan = (
    profil_jumlah["Keputusan"]
    .value_counts()
    .rename_axis("Keputusan")
    .reset_index(name="Jumlah Baris")
)

ringkasan_keputusan["Persentase"] = (
    ringkasan_keputusan["Jumlah Baris"]
    / len(profil_jumlah)
    * 100
).round(3)

print("\n" + "=" * 80)
print("RINGKASAN KEPUTUSAN CLEANING")
print("=" * 80)

display(ringkasan_keputusan)

mask_valid_nonstandard = (
    profil_jumlah["Status"].isin([
        "valid_integer_decimal_dot",
        "valid_integer_decimal_comma",
        "valid_thousands_comma",
        "valid_integer_scientific"
    ])
)

rincian_valid_nonstandard = (
    profil_jumlah.loc[
        mask_valid_nonstandard,
        [
            "Jumlah Asli",
            "Jumlah Setelah Konversi",
            "Interpretasi"
        ]
    ]
    .groupby(
        [
            "Jumlah Asli",
            "Jumlah Setelah Konversi",
            "Interpretasi"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="Jumlah Baris")
    .sort_values(
        "Jumlah Baris",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("FORMAT VALID NON-STANDAR — TETAP DIPERTAHANKAN")
print("=" * 80)

display(
    rincian_valid_nonstandard.head(100)
)

rincian_drop = (
    profil_jumlah.loc[
        profil_jumlah["Keputusan"] == "DROP",
        [
            "Jumlah Asli",
            "Jumlah Setelah Konversi",
            "Interpretasi"
        ]
    ]
    .groupby(
        [
            "Jumlah Asli",
            "Jumlah Setelah Konversi",
            "Interpretasi"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="Jumlah Baris")
    .sort_values(
        "Jumlah Baris",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("RINCIAN NILAI YANG AKAN DI-DROP")
print("=" * 80)

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None
):
    display(rincian_drop)

rincian_review = (
    profil_jumlah.loc[
        profil_jumlah["Keputusan"] == "REVIEW",
        [
            "Jumlah Asli",
            "Status",
            "Interpretasi"
        ]
    ]
    .groupby(
        [
            "Jumlah Asli",
            "Status",
            "Interpretasi"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="Jumlah Baris")
    .sort_values(
        "Jumlah Baris",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("FORMAT YANG BENAR-BENAR PERLU DITINJAU MANUAL")
print("=" * 80)

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None
):
    display(rincian_review)

Pemeriksaan mutu data setelah pembersihan.

In [ ]:
assert df_clean["Jumlah"].notna().all(), "Masih ada Jumlah invalid."
assert (df_clean["Jumlah"] > 0).all(), "Masih ada Jumlah <= 0."

assert (
    canonical_text(df_clean["Jenis Transaksi"]) == "dalam negeri"
).all(), "Masih ada transaksi bukan Dalam Negeri."

assert ~(
    canonical_text(df_clean["Keterangan"]) == "transaksi nihil"
).any(), "Masih ada transaksi nihil."

final_for_key = df_clean_audit.copy()
final_dup_key = build_duplicate_key(final_for_key)
jumlah_duplicate_final = int(final_dup_key.duplicated(keep="first").sum())

assert jumlah_duplicate_final == 0, (
    f"Masih ditemukan {jumlah_duplicate_final} duplikat pada data final."
)

profil_akhir = pd.DataFrame({
    "Kolom": SOURCE_COLUMNS,
    "Jumlah Missing": [
        int(df_clean[c].isna().sum()) for c in SOURCE_COLUMNS
    ]
})
profil_akhir["Persentase (%)"] = (
    profil_akhir["Jumlah Missing"] / len(df_clean) * 100
).round(3)

ringkasan_kualitas = pd.DataFrame({
    "Metrik": [
        "Jumlah baris raw",
        "Jumlah baris final",
        "Total baris dikeluarkan",
        "Tanggal Penyaluran invalid dipertahankan",
        "Tanggal Kedaluwarsa invalid dipertahankan",
        "Kedaluwarsa sebelum penyaluran dipertahankan",
        "Duplikat final",
    ],
    "Nilai": [
        len(df_raw),
        len(df_clean),
        len(df_raw) - len(df_clean),
        int(df_clean_audit["flag_tanggal_penyaluran_invalid"].sum()),
        int(df_clean_audit["flag_tanggal_kedaluwarsa_invalid"].sum()),
        int(df_clean_audit["flag_kedaluwarsa_sebelum_penyaluran"].sum()),
        jumlah_duplicate_final,
    ]
})

print("QUALITY GATE: LULUS")
display(ringkasan_kualitas)

print("\nProfil missing setelah cleaning:")
display(profil_akhir)

Kandidat duplikat bisnis (No. Faktur, tanggal, obat, batch, jumlah). Hanya untuk ditinjau, tidak dihapus.

In [ ]:
BUSINESS_KEY = [
    "No. Faktur",
    "Tanggal Penyaluran",
    "Nama Obat Jadi",
    "Batch",
    "Jumlah",
]

business_key_df = df_clean[BUSINESS_KEY].copy()
key_complete = business_key_df.notna().all(axis=1)

candidate_business_dup = pd.Series(False, index=df_clean.index)
candidate_business_dup.loc[key_complete] = (
    business_key_df.loc[key_complete]
    .duplicated(keep=False)
)

business_duplicates = (
    df_clean.loc[candidate_business_dup]
    .sort_values(BUSINESS_KEY, na_position="last")
    .copy()
)

print(
    "Kandidat business duplicate (tidak dihapus otomatis):",
    f"{len(business_duplicates):,}"
)

display(business_duplicates.head(30))

Ekspor seluruh hasil pembersihan.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(
    OUTPUT_DIR / "data_clean_bbpom.csv",
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d",
)

df_clean_audit.to_csv(
    OUTPUT_DIR / "data_clean_bbpom_audit.csv",
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d",
)

df_excluded.to_csv(
    OUTPUT_DIR / "baris_dikeluarkan.csv",
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d",
)

audit_df.to_csv(
    OUTPUT_DIR / "log_preprocessing.csv",
    index=False,
    encoding="utf-8-sig",
)

ringkasan_independen.to_csv(
    OUTPUT_DIR / "ringkasan_alasan_independen.csv",
    index=False,
    encoding="utf-8-sig",
)

profil_awal.to_csv(
    OUTPUT_DIR / "profil_missing_sebelum.csv",
    index=False,
    encoding="utf-8-sig",
)

profil_akhir.to_csv(
    OUTPUT_DIR / "profil_missing_sesudah.csv",
    index=False,
    encoding="utf-8-sig",
)

ringkasan_kualitas.to_csv(
    OUTPUT_DIR / "ringkasan_kualitas.csv",
    index=False,
    encoding="utf-8-sig",
)

business_duplicates.to_csv(
    OUTPUT_DIR / "kandidat_business_duplicate.csv",
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d",
)

ringkasan_txt = f"""
RINGKASAN CLEANING BBPOM
========================
Sumber                    : {SOURCE_PATH.name}
Jumlah raw                : {len(df_raw):,}
Jumlah final              : {len(df_clean):,}
Total dikeluarkan         : {len(df_raw) - len(df_clean):,}
Slash date                : MDY (MM/DD/YYYY)
Duplicate final           : {jumlah_duplicate_final:,}

Aturan utama:
- Jenis Transaksi harus Dalam Negeri.
- Keterangan 'transaksi nihil' dikeluarkan.
- Jumlah ribuan berkoma seperti 1,000 dikonversi menjadi 1000.
- Jumlah pecahan non-zero seperti 0.3 / 0,3 dikeluarkan.
- Jumlah <= 0 dikeluarkan.
- Tanggal invalid diubah menjadi NaT dan dipertahankan dengan quality flag.
- Duplicate setelah standardisasi dasar dihapus, keep first.
- Kandidat business duplicate hanya dilaporkan, tidak dihapus otomatis.
""".strip()

(OUTPUT_DIR / "ringkasan_cleaning.txt").write_text(
    ringkasan_txt,
    encoding="utf-8"
)

zip_path = shutil.make_archive(
    str(BASE_DIR / "hasil_cleaning_bbpom"),
    "zip",
    root_dir=OUTPUT_DIR
)

print("=" * 70)
print("EXPORT SELESAI")
print("=" * 70)

print("Folder :", OUTPUT_DIR)
print("ZIP    :", zip_path)

print("\nFile yang dibuat:")

for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)

print("\nLokasi lengkap ZIP:")
print(zip_path)

**AUDIT DATA**

Bagian ini memuat data bersih lalu mengonversi satuan, menentukan jenis sarana dan kategori obat, serta menangani vial dan ampul.

Memuat data bersih hasil pembersihan.

In [ ]:
from pathlib import Path
import pandas as pd

_folder = Path(OUTPUT_DIR) if "OUTPUT_DIR" in globals() else Path("hasil_cleaning_bbpom")
df_clean = pd.read_csv(_folder / "data_clean_bbpom.csv")
print(df_clean.shape)

In [ ]:
import pandas as pd
import re
import unicodedata

df_clean["Jumlah Asli"] = (
    df_clean["Jumlah"]
    .astype("string")
    .str.strip()
)

df_clean["Tanggal Penyaluran Asli"] = (
    df_clean["Tanggal Penyaluran"]
    .astype("string")
    .str.strip()
)

df_clean["Tanggal Kedaluwarsa Asli"] = (
    df_clean["Tanggal Kedaluwarsa"]
    .astype("string")
    .str.strip()
)

df_clean["No. Faktur Asli"] = (
    df_clean["No. Faktur"]
    .astype("string")
)

jumlah_asli = df_clean["Jumlah Asli"]

format_ribuan_valid = jumlah_asli.str.fullmatch(
    r"-?\d{1,3}(,\d{3})+",
    na=False
)

tanpa_koma = ~jumlah_asli.str.contains(
    ",",
    na=False,
    regex=False
)

numeric_tanpa_koma = pd.to_numeric(
    jumlah_asli.where(tanpa_koma),
    errors="coerce"
)

integer_tanpa_koma = (
    numeric_tanpa_koma.notna()
    & (numeric_tanpa_koma % 1 == 0)
)

mask_jumlah_valid = (
    format_ribuan_valid
    | integer_tanpa_koma
)

mask_jumlah_tidak_valid = ~mask_jumlah_valid

jumlah_bersih = jumlah_asli.copy()

jumlah_bersih.loc[format_ribuan_valid] = (
    jumlah_bersih.loc[format_ribuan_valid]
    .str.replace(",", "", regex=False)
)

df_clean["Jumlah Setelah Konversi"] = (
    pd.to_numeric(
        jumlah_bersih.where(mask_jumlah_valid),
        errors="coerce"
    )
    .astype("Int64")
)

df_clean["Jumlah"] = df_clean["Jumlah Setelah Konversi"]

df_clean["Tanggal Penyaluran"] = pd.to_datetime(
    df_clean["Tanggal Penyaluran Asli"],
    format="mixed",
    dayfirst=False,
    errors="coerce"
)

df_clean["Tanggal Kedaluwarsa"] = pd.to_datetime(
    df_clean["Tanggal Kedaluwarsa Asli"],
    format="mixed",
    dayfirst=False,
    errors="coerce"
)

mask_tanggal_penyaluran_tidak_valid = (
    df_clean["Tanggal Penyaluran"].isna()
)

mask_tanggal_kedaluwarsa_tidak_valid = (
    df_clean["Tanggal Kedaluwarsa"].isna()
)

mask_tanggal_tidak_valid = (
    mask_tanggal_penyaluran_tidak_valid
    | mask_tanggal_kedaluwarsa_tidak_valid
)

def cek_nomor_faktur(value):
    alasan = []

    if pd.isna(value) or str(value).strip() == "":
        return "Nomor faktur kosong"

    teks = str(value)
    teks_strip = teks.strip()

    if any(tanda in teks for tanda in ["â€", "Â", "�"]):
        alasan.append("Karakter encoding rusak")

    if any(
        unicodedata.category(karakter) in {"Cc", "Cf"}
        for karakter in teks
    ):
        alasan.append("Karakter tersembunyi")

    if re.fullmatch(
        r"\d+(\.\d+)?[eE][+-]?\d+",
        teks_strip
    ):
        alasan.append("Format scientific notation")

    if not re.fullmatch(
        r"[A-Za-z0-9 ./_-]+",
        teks_strip
    ):
        alasan.append("Mengandung simbol tidak umum")

    return "; ".join(alasan) if alasan else "Normal"

df_clean["Keterangan Validasi Faktur"] = (
    df_clean["No. Faktur Asli"]
    .apply(cek_nomor_faktur)
    .astype("string")
)

mask_faktur_tidak_valid = (
    df_clean["Keterangan Validasi Faktur"] != "Normal"
)

df_clean["Keterangan Validasi"] = ""

def tambah_keterangan(mask, keterangan):
    kondisi_kosong = (
        df_clean.loc[mask, "Keterangan Validasi"] == ""
    )

    indeks_kosong = kondisi_kosong[
        kondisi_kosong
    ].index

    indeks_terisi = kondisi_kosong[
        ~kondisi_kosong
    ].index

    df_clean.loc[
        indeks_kosong,
        "Keterangan Validasi"
    ] = keterangan

    df_clean.loc[
        indeks_terisi,
        "Keterangan Validasi"
    ] = (
        df_clean.loc[
            indeks_terisi,
            "Keterangan Validasi"
        ]
        + "; "
        + keterangan
    )

tambah_keterangan(
    mask_jumlah_tidak_valid,
    "Jumlah tidak valid"
)

tambah_keterangan(
    mask_tanggal_penyaluran_tidak_valid,
    "Tanggal penyaluran tidak valid"
)

tambah_keterangan(
    mask_tanggal_kedaluwarsa_tidak_valid,
    "Tanggal kedaluwarsa tidak valid"
)

tambah_keterangan(
    mask_faktur_tidak_valid,
    "Nomor faktur tidak valid"
)

df_clean.loc[
    df_clean["Keterangan Validasi"] == "",
    "Keterangan Validasi"
] = "Valid"

df_clean["Keterangan Validasi"] = (
    df_clean["Keterangan Validasi"]
    .astype("string")
)

kolom_datetime = [
    "Tanggal Penyaluran",
    "Tanggal Kedaluwarsa"
]

kolom_integer = [
    "Jumlah",
    "Jumlah Setelah Konversi"
]

kolom_string = [
    kolom for kolom in df_clean.columns
    if kolom not in kolom_datetime + kolom_integer
]

df_clean[kolom_string] = (
    df_clean[kolom_string]
    .astype("string")
)

print("RINGKASAN VALIDASI")
print("=" * 50)
print(f"Total data                       : {len(df_clean):,}")
print(f"Jumlah tidak valid               : {mask_jumlah_tidak_valid.sum():,}")
print(
    "Tanggal penyaluran tidak valid  :",
    f"{mask_tanggal_penyaluran_tidak_valid.sum():,}"
)
print(
    "Tanggal kedaluwarsa tidak valid :",
    f"{mask_tanggal_kedaluwarsa_tidak_valid.sum():,}"
)
print(f"Nomor faktur tidak valid         : {mask_faktur_tidak_valid.sum():,}")

print("\nKeterangan validasi gabungan:")
print(
    df_clean["Keterangan Validasi"]
    .value_counts(dropna=False)
)

print("\nMasalah nomor faktur:")
print(
    df_clean["Keterangan Validasi Faktur"]
    .value_counts(dropna=False)
)

kolom_tinjauan = [
    "Jumlah Asli",
    "Jumlah Setelah Konversi",
    "Tanggal Penyaluran Asli",
    "Tanggal Penyaluran",
    "Tanggal Kedaluwarsa Asli",
    "Tanggal Kedaluwarsa",
    "No. Faktur Asli",
    "Keterangan Validasi Faktur",
    "Keterangan Validasi"
]

data_perlu_ditinjau = df_clean[
    df_clean["Keterangan Validasi"] != "Valid"
].copy()

print(
    f"\nJumlah baris yang perlu ditinjau: "
    f"{len(data_perlu_ditinjau):,}"
)

display(data_perlu_ditinjau[kolom_tinjauan])

In [ ]:
import pandas as pd
import re

def kategori_satuan(nilai):
    if pd.isna(nilai):
        return "TIDAK TERISI"

    teks = " ".join(str(nilai).upper().split())

    if not teks:
        return "TIDAK TERISI"

    if re.match(r"^(?:\d+\s+)?(?:DUS|BOX)\b", teks):
        return "DUS"

    if re.match(r"^BOTOL\b", teks):
        return "BOTOL"

    return "PERLU DITINJAU"

kategori = df_clean["Satuan"].apply(kategori_satuan)

if "Kategori Satuan" in df_clean.columns:
    df_clean.drop(columns="Kategori Satuan", inplace=True)

df_clean.insert(
    df_clean.columns.get_loc("Satuan") + 1,
    "Kategori Satuan",
    kategori
)

hasil = (
    df_clean.groupby(["Satuan", "Kategori Satuan"], dropna=False)
    .size()
    .reset_index(name="Jumlah Baris")
    .sort_values("Jumlah Baris", ascending=False)
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None
):
    display(hasil)

print("Ringkasan kategori:")
display(df_clean["Kategori Satuan"].value_counts(dropna=False))

In [ ]:
import pandas as pd
from IPython.display import display

KOLOM_JUMLAH = "Jumlah"
KOLOM_SATUAN = "Satuan"
KOLOM_TARGET = "Kategori Satuan"
LABEL_TARGET = "PERLU DITINJAU"

PILIHAN = {
    "1": "DUS",
    "dus": "DUS",
    "2": "BOTOL",
    "botol": "BOTOL"
}

if "df_clean" not in globals():
    raise NameError("DataFrame df_clean belum tersedia. Jalankan sel pemuatan data terlebih dahulu.")

for kolom in [KOLOM_JUMLAH, KOLOM_SATUAN, KOLOM_TARGET]:
    if kolom not in df_clean.columns:
        raise KeyError(f"Kolom '{kolom}' tidak ditemukan di df_clean.")

if not df_clean.index.is_unique:
    df_clean = df_clean.reset_index(drop=True)

mask = (
    df_clean[KOLOM_TARGET]
    .astype("string")
    .str.strip()
    .str.upper()
    .eq(LABEL_TARGET)
    .fillna(False)
)

posisi_target = mask.to_numpy().nonzero()[0]

print(f"Total baris PERLU DITINJAU: {len(posisi_target):,}")

if len(posisi_target) == 0:
    print("Tidak ada data yang perlu ditinjau.")

else:
    data_tinjau = df_clean.iloc[posisi_target][
        [KOLOM_JUMLAH, KOLOM_SATUAN]
    ].reset_index(drop=True)

    kelompok = data_tinjau.groupby(
        [KOLOM_JUMLAH, KOLOM_SATUAN],
        dropna=False,
        sort=False
    ).indices

    daftar = sorted(
        kelompok.items(),
        key=lambda item: len(item[1]),
        reverse=True
    )

    posisi_kolom_target = df_clean.columns.get_loc(KOLOM_TARGET)

    kelompok_dijawab = 0
    baris_berubah = 0
    riwayat = []

    print("""
Petunjuk:
1 = DUS
2 = BOTOL
Enter = Lewati
stop = Hentikan sementara
""")

    try:
        for nomor, ((jumlah, satuan), posisi_grup) in enumerate(
            daftar, start=1
        ):
            posisi_asli = posisi_target[list(posisi_grup)]

            print("\n" + "=" * 45)
            print(f"Kelompok {nomor}/{len(daftar)}")
            print(f"Jumlah : {jumlah}")
            print(f"Satuan : {satuan}")
            print(f"Jumlah transaksi : {len(posisi_asli):,}")

            while True:
                jawaban = input(
                    "Kategori Satuan [1=DUS / 2=BOTOL / Enter / stop]: "
                ).strip().casefold()

                if jawaban in ("", "lewati", "skip", "stop"):
                    break

                if jawaban in PILIHAN:
                    kategori_baru = PILIHAN[jawaban]

                    # Perbarui hanya kolom Kategori Satuan
                    df_clean.iloc[
                        posisi_asli, posisi_kolom_target
                    ] = kategori_baru

                    kelompok_dijawab += 1
                    baris_berubah += len(posisi_asli)

                    riwayat.append({
                        "Jumlah": jumlah,
                        "Satuan": satuan,
                        "Kategori Satuan Baru": kategori_baru,
                        "Jumlah Transaksi": len(posisi_asli)
                    })

                    print(f"Berhasil dikategorikan: {kategori_baru}")
                    break

                print("Input tidak valid. Masukkan 1, 2, Enter, atau stop.")

            if jawaban == "stop":
                break

    except (KeyboardInterrupt, EOFError):
        print("\nProses dihentikan. Perubahan sebelumnya tetap tersimpan.")

    print("\n===== RINGKASAN PENINJAUAN =====")
    print(f"Kelompok dikategorikan : {kelompok_dijawab:,}")
    print(f"Baris diperbarui       : {baris_berubah:,}")

    if riwayat:
        print("\nRiwayat peninjauan:")
        display(pd.DataFrame(riwayat))

    frekuensi = (
        df_clean[KOLOM_TARGET]
        .astype("string")
        .fillna("(kosong)")
        .value_counts()
    )

    ringkasan = pd.DataFrame({
        "Kategori Satuan": frekuensi.index,
        "Jumlah Baris": frekuensi.values
    })

    ringkasan["Persentase (%)"] = (
        ringkasan["Jumlah Baris"] / len(df_clean) * 100
    ).round(3)

    print("\n===== DISTRIBUSI KATEGORI SATUAN =====")
    display(ringkasan.reset_index(drop=True))

    sisa = (
        df_clean[KOLOM_TARGET]
        .astype("string")
        .str.strip()
        .str.upper()
        .eq(LABEL_TARGET)
        .fillna(False)
    )

    print(f"Sisa PERLU DITINJAU: {int(sisa.sum()):,} baris")


In [ ]:
import pandas as pd
import re
import unicodedata

def normalize_text(nilai):
    teks = str(nilai)

    teks = teks.replace("ï¿½", " ").replace("\ufffd", " ")
    teks = unicodedata.normalize("NFKC", teks).casefold()

    perbaikan = [
        (r"\br\s*\.\s*s\s*\.\s*a\s*\.\s*h\b\.?", "rsah"),
        (r"\br\s*\.\s*s\s*\.\s*u\b\.?", "rsu"),
        (r"\br\s*\.\s*s\b\.?", "rs"),
        (r"\ba(?:\s*\.\s*|\s+)p\b\.?", "ap"),
        (r"\bk\s*\.\s*p\b\.?", "kp"),
    ]

    for pola, pengganti in perbaikan:
        teks = re.sub(pola, pengganti, teks)

    return re.sub(r"\s+", " ", teks).strip()

def klasifikasi_sarana(nama):
    if pd.isna(nama):
        return "PERLU DITINJAU"

    nama = normalize_text(nama)

    if not nama:
        return "PERLU DITINJAU"

    if (
        re.search(
            r"\b(rs|rsu|rsud|rumkit|rsa|rsia|rsi|rsk|rsb|"
            r"hospital|hospitals|rsal|rsd|rsau|rsup|rumkital|"
            r"rsab|pku|rkz|rsui|rsj|rb|rspal|rsm|rsmm|rskia|"
            r"rst|rskb|rspau|rsgm|rsah|rsad|rumkitban|rumkitalmar|"
            r"ifrsia|yayasan|pers|perk|yay)\b",
            nama
        )
        or "rumah sakit" in nama
    ):
        return "Rumah Sakit"

    if re.search(r"\b(puskesmas|pkm)\b", nama):
        return "Puskesmas"

    if re.search(r"\bkftd\b", nama):
        return "PBF"

    if (
        re.search(
            r"\b(apotek|apotik|ap0tek|apt|ap|viva|farma|va|vg|"
            r"pharmacy|pharmacies|kf|AOITIK)\b",
            nama
        )
        or re.search(r"\bk\s*-?\s*24\b", nama)
        or "apotek" in nama
    ):
        return "Apotek"

    if re.search(
        r"\b(instalasi farmasi|ifrs|ins|inst|farmasi|ifk)\b",
        nama
    ):
        return "Rumah Sakit"

    if re.search(r"\b(pt|pbf|cv|aam|epm)\b", nama):
        return "PBF"

    if (
        re.search(
            r"\b(klinik|clinic|kurj|kp|kprj|kpri|kl)\b",
            nama
        )
        or "klinik" in nama
    ):
        return "Klinik"

    if re.search(r"\b(dinkes|dinas kesehatan|dnk)\b", nama):
        return "PBF"

    return "PERLU DITINJAU"

jenis_sarana = df_clean["Tujuan Penyaluran"].apply(klasifikasi_sarana)

if "Jenis Sarana" in df_clean.columns:
    df_clean.drop(columns="Jenis Sarana", inplace=True)

df_clean.insert(
    df_clean.columns.get_loc("Tujuan Penyaluran") + 1,
    "Jenis Sarana",
    jenis_sarana
)

ringkasan = (
    df_clean.groupby("Jenis Sarana", dropna=False)
    .agg(
        Jumlah_Baris=("Tujuan Penyaluran", "size"),
        Jumlah_Tujuan_Unik=("Tujuan Penyaluran", "nunique")
    )
    .reset_index()
    .sort_values("Jumlah_Baris", ascending=False)
    .reset_index(drop=True)
)

ringkasan["Persentase_Baris"] = (
    (ringkasan["Jumlah_Baris"] / len(df_clean) * 100).round(2)
    if len(df_clean) else 0.0
)

print("RINGKASAN JENIS SARANA")
print(f"Total baris       : {len(df_clean):,}")
print(f"Total tujuan unik : {df_clean['Tujuan Penyaluran'].nunique():,}")
display(ringkasan)

mask = df_clean["Jenis Sarana"].eq("PERLU DITINJAU")
tujuan_tinjau = df_clean.loc[mask, "Tujuan Penyaluran"]

print("\nRINGKASAN PERLU DITINJAU")
print(f"Baris perlu ditinjau : {len(tujuan_tinjau):,}")
print(f"Tujuan unik         : {tujuan_tinjau.nunique():,}")
print(f"Baris null          : {tujuan_tinjau.isna().sum():,}")

rincian_tinjau = (
    tujuan_tinjau.value_counts(dropna=False)
    .rename_axis("Tujuan Penyaluran")
    .reset_index(name="Jumlah Baris")
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None
):
    display(rincian_tinjau)

In [ ]:
import pandas as pd

pilihan = {
    "1": "Apotek",
    "apotek": "Apotek",
    "apotik": "Apotek",

    "2": "PBF",
    "pbf": "PBF",

    "3": "Rumah Sakit",
    "rs": "Rumah Sakit",
    "rumah sakit": "Rumah Sakit",
    "yayasan": "Rumah Sakit",

    "4": "Puskesmas",
    "puskesmas": "Puskesmas",
    "pkm": "Puskesmas",

    "5": "Klinik",
    "klinik": "Klinik",

    "6": "Balai Kesehatan",
    "balai kesehatan": "Balai Kesehatan",
    "balkes": "Balai Kesehatan",
    "balkesmas": "Balai Kesehatan",

    "7": "Dinas Kesehatan",
    "dinas kesehatan": "Dinas Kesehatan",
    "dinkes": "Dinas Kesehatan",

    "8": "TUJUAN PENYALURAN TIDAK JELAS",
    "tidak jelas": "TUJUAN PENYALURAN TIDAK JELAS",
}

if "Jenis Sarana Manual" not in df_clean.columns:
    df_clean["Jenis Sarana Manual"] = pd.Series(
        pd.NA, index=df_clean.index, dtype="string"
    )

manual_ada = (
    df_clean["Jenis Sarana Manual"].notna()
    & df_clean["Jenis Sarana Manual"]
        .astype("string").str.strip().ne("").fillna(False)
)

df_clean.loc[manual_ada, "Jenis Sarana"] = (
    df_clean.loc[manual_ada, "Jenis Sarana Manual"]
)

pending = df_clean["Jenis Sarana"].eq("PERLU DITINJAU")
nama_valid = (
    df_clean["Tujuan Penyaluran"]
    .astype("string").str.strip().ne("").fillna(False)
)

kelompok = df_clean.loc[pending & nama_valid].groupby(
    "Tujuan Penyaluran", sort=False
).indices

posisi_pending = (
    (pending & nama_valid).fillna(False).to_numpy().nonzero()[0]
)

daftar = sorted(
    kelompok.items(),
    key=lambda item: len(item[1]),
    reverse=True
)

kolom_lokasi = [
    k for k in [
        "Alamat Tujuan",
        "Nama Kota/Kab Tujuan",
        "Nama Provinsi Tujuan"
    ]
    if k in df_clean.columns
]

kolom_jenis = df_clean.columns.get_loc("Jenis Sarana")
kolom_manual = df_clean.columns.get_loc("Jenis Sarana Manual")

print(
    "PILIHAN:\n"
    "1 Apotek | 2 PBF | 3 Rumah Sakit | 4 Puskesmas\n"
    "5 Klinik | 6 Balai Kesehatan | 7 Dinas Kesehatan\n"
    "8 Tujuan penyaluran tidak jelas\n"
    "\nEnter = lewati | detail = lihat semua lokasi | stop = berhenti"
)

selesai = 0

try:
    for nomor, (nama, posisi_grup) in enumerate(daftar, 1):
        posisi = posisi_pending[posisi_grup]

        print(f"\n[{nomor}/{len(daftar)}] {nama}", flush=True)
        print(f"Jumlah transaksi: {len(posisi):,}", flush=True)

        lokasi = (
            df_clean.iloc[posisi][kolom_lokasi]
            .fillna("")
            .drop_duplicates()
        )

        def tampilkan_lokasi(tabel):
            for baris in tabel.itertuples(index=False, name=None):
                teks = " | ".join(str(x) for x in baris if str(x).strip())
                print("Lokasi:", teks or "(kosong)", flush=True)

        tampilkan_lokasi(lokasi.head(1))

        if len(lokasi) > 1:
            print(
                f"Ada {len(lokasi)} variasi lokasi. "
                "Ketik detail untuk memeriksanya.",
                flush=True
            )

        while True:
            jawaban = " ".join(
                input("Kategori [1–8 / nama / detail / Enter / stop]: ")
                .casefold().split()
            )

            if jawaban == "detail":
                tampilkan_lokasi(lokasi)
                continue

            if jawaban in ("", "lewati", "skip", "stop"):
                break

            if jawaban in pilihan:
                kategori = pilihan[jawaban]

                df_clean.iloc[posisi, kolom_jenis] = kategori
                df_clean.iloc[posisi, kolom_manual] = kategori

                if "Status Tinjauan" in df_clean.columns:
                    kolom_status = df_clean.columns.get_loc("Status Tinjauan")
                    df_clean.iloc[posisi, kolom_status] = "DITINJAU MANUAL"

                selesai += 1
                print(f"Tercatat: {kategori}", flush=True)
                break

            print("Pilihan belum dikenali. Coba lagi.", flush=True)

        if jawaban == "stop":
            break

except (KeyboardInterrupt, EOFError):
    print("\nTinjauan dihentikan.")

print(f"\nNama yang diperbarui pada sesi ini: {selesai}")
print("\nRingkasan:")
print(df_clean["Jenis Sarana"].value_counts(dropna=False).to_string())

In [ ]:
import pandas as pd

ringkasan = (
    df_clean.groupby("Jenis Sarana", dropna=False)
    .agg(
        Jumlah_Baris=("Tujuan Penyaluran", "size"),
        Jumlah_Tujuan_Unik=("Tujuan Penyaluran", "nunique")
    )
    .reset_index()
    .sort_values("Jumlah_Baris", ascending=False)
)

ringkasan["Persentase"] = (
    (ringkasan["Jumlah_Baris"] / len(df_clean) * 100).round(2)
    if len(df_clean) else 0.0
)

print("RINGKASAN JENIS SARANA")
display(ringkasan)

manual = df_clean["Jenis Sarana Manual"].astype("string")
jenis = df_clean["Jenis Sarana"].astype("string")

mask_manual = manual.notna() & manual.str.strip().ne("").fillna(False)

mask_berbeda = (
    mask_manual
    & ~jenis.eq(manual).fillna(False)
)

print("\nPEMERIKSAAN HASIL MANUAL")
print(f"Baris ditinjau manual : {mask_manual.sum():,}")
print(
    "Tujuan unik manual   :",
    f"{df_clean.loc[mask_manual, 'Tujuan Penyaluran'].nunique():,}"
)
print(f"Hasil manual berbeda dengan Jenis Sarana: {mask_berbeda.sum():,}")

kolom_cek = [
    "Tujuan Penyaluran",
    "Jenis Sarana",
    "Jenis Sarana Manual"
]

hasil_manual = (
    df_clean.loc[mask_manual, kolom_cek]
    .groupby(kolom_cek, dropna=False)
    .size()
    .reset_index(name="Jumlah Baris")
    .sort_values("Jumlah Baris", ascending=False)
)

print("\nDAFTAR HASIL TINJAUAN MANUAL")
display(hasil_manual)

if mask_berbeda.any():
    print("\nHASIL MANUAL YANG BELUM SAMA DENGAN JENIS SARANA")
    display(
        df_clean.loc[mask_berbeda, kolom_cek]
        .drop_duplicates()
    )

jenis_cek = jenis.str.strip().str.upper()

mask_pending = (
    jenis_cek.isna()
    | jenis_cek.eq("").fillna(False)
    | jenis_cek.isin(["PERLU DITINJAU", "LAINNYA"])
)

mask_tidak_jelas = jenis_cek.eq(
    "TUJUAN PENYALURAN TIDAK JELAS"
).fillna(False)

def tampilkan_daftar(mask, judul):
    tujuan = df_clean.loc[mask, "Tujuan Penyaluran"]

    rincian = (
        tujuan.value_counts(dropna=False)
        .rename_axis("Tujuan Penyaluran")
        .reset_index(name="Jumlah Baris")
    )

    print(f"\n{judul}")
    print(f"Jumlah baris : {len(tujuan):,}")
    print(f"Tujuan unik  : {tujuan.nunique():,}")
    print(f"Baris null   : {tujuan.isna().sum():,}")

    with pd.option_context(
        "display.max_rows", None,
        "display.max_colwidth", None
    ):
        display(rincian)

    return rincian

sisa_tinjauan = tampilkan_daftar(
    mask_pending,
    "SISA PERLU DITINJAU"
)

tujuan_tidak_jelas = tampilkan_daftar(
    mask_tidak_jelas,
    "TUJUAN PENYALURAN TIDAK JELAS"
)

In [ ]:
import pandas as pd

if "Jenis Sarana Manual" in df_clean.columns:
    manual = df_clean["Jenis Sarana Manual"].astype("string")

    mask_manual = (
        manual.notna()
        & manual.str.strip().ne("").fillna(False)
    )

    df_clean["Jenis Sarana"] = df_clean["Jenis Sarana"].astype("string")

    df_clean.loc[mask_manual, "Jenis Sarana"] = (
        manual.loc[mask_manual].str.strip()
    )

    df_clean.drop(columns="Jenis Sarana Manual", inplace=True)

    print(f"Baris menggunakan hasil manual: {mask_manual.sum():,}")
    print("Penggabungan selesai.")
else:
    print("Kolom Jenis Sarana Manual sudah tidak tersedia.")

display(df_clean["Jenis Sarana"].value_counts(dropna=False))

Klasifikasi kategori obat.

In [ ]:
import re
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

KOLOM_NAMA_OBAT = "Nama Obat Jadi"
KOLOM_ZAT_AKTIF = "Nama Zat Aktif"

OBAT_KATEGORI = {

    "Cefixime": [
        "cefixime",
        "cefixime trihydrate",
        "cefixim",
        "sefiksim",

        "sporetik",
        "cefila",
        "helixim",
        "ceptik",
        "befixim",
        "taxime",
        "simfix",
        "cefarox",
        "lanfix",
        "fixacep",
        "pyxime",
        "fixatic",
        "cefacef",
        "maxpro",
        "inbacef",
        "tocef",
        "fixam",
        "anfix",
        "simcef",
        "starcef",
        "ximecef",
        "pharmafix"
    ],

    "TBC": [
        "rifampicin",
        "rifampisin",
        "isoniazid",
        "pyrazinamide",
        "pirazinamid",
        "ethambutol",
        "etambutol",

        "pro tb",
        "rifastat",
        "rifamtibi",
        "tb vit",
        "sanazet",
        "corifam",
        "pyratibi",
        "tibigon",
        "rifastar",
        "santibi",
        "erabutol",
        "metham",
        "arsitam",
        "tibitol",
        "rizatol",
        "pulna",
        "kalbutol",
        "pehadoxin",
        "inha",
        "pyravit",
        "inadoxin",
        "rifanh",
        "risontin",
        "tb zet",
        "siramid",
        "neotibi",
        "propulmo",
        "pynamed",
        "rifabiotic",
        "merimac",
        "tb rif"
    ],

    "Meropenem": [
        "meropenem",

        "merosan",
        "merobat",
        "merotika",
        "merocef",
        "merem",
        "merofen",
        "meropex",
        "merotik",
        "simpenem",
        "opimer",
        "eradix",
        "caprenem",
        "granem",
        "infinem"
    ],

    "Orlistat": [
        "orlistat",

        "obeslim",
        "xenical",
        "vistat",
        "orlibes",
        "orlivell",
        "maxslim",
        "orlislim"
    ],

    "Ketamine": [
        "ketamine",
        "ketamine hydrochloride",
        "ketamin",

        "ktm",
        "kemidex",
        "ketanaest",
        "ivanes",
        "tamivell",
        "ketages"
    ]
}

if "df_clean" not in globals():
    raise NameError(
        "df_clean belum tersedia. "
        "Jalankan terlebih dahulu proses preprocessing sampai df_clean terbentuk."
    )

kolom_wajib = [
    KOLOM_NAMA_OBAT,
    KOLOM_ZAT_AKTIF
]

kolom_hilang = [
    kolom
    for kolom in kolom_wajib
    if kolom not in df_clean.columns
]

if kolom_hilang:
    raise ValueError(
        f"Kolom berikut belum tersedia di df_clean: {kolom_hilang}"
    )

print(f"Jumlah data pada df_clean: {len(df_clean):,} baris")

def bersihkan_teks(series):
    """
    Membersihkan teks:
    - lowercase / casefold
    - menghapus simbol
    - merapikan spasi
    - string kosong menjadi NA
    """

    return (
        series
        .astype("string")
        .str.casefold()
        .str.replace(r"[^a-z0-9\s]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )

def bersihkan_keyword(kata):
    """
    Membersihkan keyword agar konsisten dengan teks data.
    """

    kata = str(kata).casefold()
    kata = re.sub(r"[^a-z0-9\s]", " ", kata)
    kata = re.sub(r"\s+", " ", kata)
    return kata.strip()

POLA = {}

for kategori, daftar_keyword in OBAT_KATEGORI.items():

    keyword_bersih = sorted(
        {
            bersihkan_keyword(k)
            for k in daftar_keyword
            if bersihkan_keyword(k)
        },
        key=len,
        reverse=True
    )

    POLA[kategori] = (
        r"(?<![a-z])(?:"
        + "|".join(
            re.escape(k)
            for k in keyword_bersih
        )
        + r")(?![a-z])"
    )

def cari_kategori(teks):
    """
    Mengembalikan:
        kategori jika hanya satu kategori cocok
        None jika tidak ada kategori
        KONFLIK jika lebih dari satu kategori cocok
    """

    if pd.isna(teks):
        return None

    teks = str(teks).strip()

    if teks == "":
        return None

    hasil = []

    for kategori, pola in POLA.items():

        if re.search(
            pola,
            teks,
            flags=re.IGNORECASE
        ):
            hasil.append(kategori)

    if len(hasil) == 0:
        return None

    if len(hasil) == 1:
        return hasil[0]

    return "KONFLIK"

teks_nama_obat = bersihkan_teks(
    df_clean[KOLOM_NAMA_OBAT]
)

teks_zat_aktif = bersihkan_teks(
    df_clean[KOLOM_ZAT_AKTIF]
)

nama_unik = teks_nama_obat.dropna().unique()

kamus_nama = {
    nama: cari_kategori(nama)
    for nama in nama_unik
}

kat_nama = teks_nama_obat.map(
    lambda x: kamus_nama.get(x, None)
    if pd.notna(x)
    else None
)

zat_unik = teks_zat_aktif.dropna().unique()

kamus_zat = {
    zat: cari_kategori(zat)
    for zat in zat_unik
}

kat_zat = teks_zat_aktif.map(
    lambda x: kamus_zat.get(x, None)
    if pd.notna(x)
    else None
)

kategori_final = pd.Series(
    "Lainnya",
    index=df_clean.index,
    dtype="string"
)

sumber_final = pd.Series(
    "TIDAK TERKLASIFIKASI",
    index=df_clean.index,
    dtype="string"
)

status_final = pd.Series(
    "NORMAL",
    index=df_clean.index,
    dtype="string"
)

kategori_valid = set(OBAT_KATEGORI.keys())

nama_valid = kat_nama.isin(kategori_valid)

kategori_final.loc[nama_valid] = (
    kat_nama.loc[nama_valid]
)

sumber_final.loc[nama_valid] = (
    "NAMA OBAT JADI"
)

nama_tidak_terklasifikasi = (
    ~kat_nama.isin(kategori_valid)
)

zat_valid = kat_zat.isin(kategori_valid)

gunakan_zat = (
    nama_tidak_terklasifikasi
    & zat_valid
)

kategori_final.loc[gunakan_zat] = (
    kat_zat.loc[gunakan_zat]
)

sumber_final.loc[gunakan_zat] = (
    "NAMA ZAT AKTIF"
)

nama_kosong = teks_nama_obat.isna()
zat_kosong = teks_zat_aktif.isna()

keduanya_kosong = (
    nama_kosong
    & zat_kosong
)

kategori_final.loc[keduanya_kosong] = (
    "Tidak diketahui"
)

sumber_final.loc[keduanya_kosong] = (
    "NAMA DAN ZAT AKTIF KOSONG"
)

status_final.loc[keduanya_kosong] = (
    "DATA KOSONG"
)

konflik_nama = kat_nama.eq("KONFLIK")

status_final.loc[konflik_nama] = (
    "PERLU DITINJAU"
)

kategori_final.loc[konflik_nama] = (
    "Lainnya"
)

sumber_final.loc[konflik_nama] = (
    "KONFLIK NAMA OBAT JADI"
)

konflik_zat = kat_zat.eq("KONFLIK")

konflik_zat_perlu_tinjau = (
    konflik_zat
    & ~nama_valid
)

status_final.loc[konflik_zat_perlu_tinjau] = (
    "PERLU DITINJAU"
)

kategori_final.loc[konflik_zat_perlu_tinjau] = (
    "Lainnya"
)

sumber_final.loc[konflik_zat_perlu_tinjau] = (
    "KONFLIK NAMA ZAT AKTIF"
)

keduanya_valid = (
    kat_nama.isin(kategori_valid)
    & kat_zat.isin(kategori_valid)
)

konflik_dua_sumber = (
    keduanya_valid
    & (kat_nama != kat_zat)
)

status_final.loc[konflik_dua_sumber] = (
    "PERLU DITINJAU"
)

sumber_final.loc[konflik_dua_sumber] = (
    "NAMA OBAT JADI + ZAT AKTIF BERBEDA"
)

kategori_final.loc[konflik_dua_sumber] = (
    kat_nama.loc[konflik_dua_sumber]
)

df_clean["Kategori Obat"] = kategori_final
df_clean["Sumber Kategori Obat"] = sumber_final
df_clean["Status Klasifikasi Obat"] = status_final

print("\n")
print("=" * 80)
print("RINGKASAN KLASIFIKASI JENIS OBAT")
print("=" * 80)

ringkasan = (
    df_clean["Kategori Obat"]
    .value_counts(dropna=False)
    .rename_axis("Kategori Obat")
    .reset_index(name="Jumlah Baris")
)

ringkasan["Persentase (%)"] = (
    ringkasan["Jumlah Baris"]
    / len(df_clean)
    * 100
).round(3)

display(ringkasan)

print("\n")
print("=" * 80)
print("SUMBER KLASIFIKASI")
print("=" * 80)

display(
    df_clean["Sumber Kategori Obat"]
    .value_counts(dropna=False)
    .rename_axis("Sumber")
    .reset_index(name="Jumlah Baris")
)

print("\n")
print("=" * 80)
print("STATUS KLASIFIKASI")
print("=" * 80)

display(
    df_clean["Status Klasifikasi Obat"]
    .value_counts(dropna=False)
    .rename_axis("Status")
    .reset_index(name="Jumlah Baris")
)

print("\n")
print("=" * 80)
print("20 NAMA OBAT TERBANYAK PER KATEGORI")
print("=" * 80)

daftar_kategori = (
    list(OBAT_KATEGORI.keys())
    + ["Lainnya", "Tidak diketahui"]
)

for kategori in daftar_kategori:

    data_kategori = df_clean[
        df_clean["Kategori Obat"] == kategori
    ]

    print(f"\n--- {kategori} ---")
    print(f"Jumlah baris: {len(data_kategori):,}")

    if len(data_kategori) > 0:

        display(
            data_kategori[KOLOM_NAMA_OBAT]
            .value_counts()
            .head(20)
            .rename_axis(KOLOM_NAMA_OBAT)
            .reset_index(name="Jumlah Baris")
        )

print("\n")
print("=" * 80)
print("DATA YANG PERLU DITINJAU")
print("=" * 80)

data_tinjau = df_clean[
    df_clean["Status Klasifikasi Obat"] == "PERLU DITINJAU"
]

print(
    f"Jumlah data yang perlu ditinjau: "
    f"{len(data_tinjau):,} baris"
)

if len(data_tinjau) > 0:

    display(
        data_tinjau[
            [
                KOLOM_NAMA_OBAT,
                KOLOM_ZAT_AKTIF,
                "Kategori Obat",
                "Sumber Kategori Obat",
                "Status Klasifikasi Obat"
            ]
        ].head(50)
    )

print("\n")
print("=" * 80)
print("CONTOH HASIL AKHIR")
print("=" * 80)

display(
    df_clean[
        [
            KOLOM_NAMA_OBAT,
            KOLOM_ZAT_AKTIF,
            "Kategori Obat",
            "Sumber Kategori Obat",
            "Status Klasifikasi Obat"
        ]
    ].head(20)
)

Membaca kembali `data_baru_clean.csv` yang berisi hasil pengayaan.

In [ ]:
import pandas as pd
df_clean= pd.read_csv("data_baru_clean.csv")
print(df_clean)

Konversi vial dan ampul serta temuan anomali PBF.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

KONVERSI_VIAL = True
KONVERSI_AMPUL = True

AMPUL_OBAT = {"Ketamine", "Meropenem"}

BATAS_JUMLAH_JANGGAL = None

BATAS_TANDAI_TEMUAN = 100000

BERKAS_TEMUAN = "temuan_pbf_jumlah_janggal.xlsx"

RETUR_POLA = (
    r"gudang pusat|pabrik|produsen|principal|retur|pemusnahan"
)

if "df_clean" not in globals():
    raise NameError(
        "df_clean belum ada. Jalankan sel pemuatan data terlebih dahulu."
    )

kolom_wajib = [
    "Satuan",
    "Jumlah",
    "Jumlah Setelah Konversi",
    "Kategori Satuan",
]

kolom_hilang = [
    kolom for kolom in kolom_wajib
    if kolom not in df_clean.columns
]

if kolom_hilang:
    raise ValueError(
        f"Kolom berikut tidak ada di df_clean: {kolom_hilang}"
    )

kolom_cadangan_jumlah = "Jumlah Setelah Konversi (sebelum vial)"
kolom_cadangan_satuan = "Kategori Satuan (sebelum vial)"

if kolom_cadangan_jumlah not in df_clean.columns:
    df_clean[kolom_cadangan_jumlah] = df_clean["Jumlah Setelah Konversi"]

if kolom_cadangan_satuan not in df_clean.columns:
    df_clean[kolom_cadangan_satuan] = df_clean["Kategori Satuan"]

df_clean["Jumlah Setelah Konversi"] = df_clean[kolom_cadangan_jumlah].copy()
df_clean["Kategori Satuan"] = (
    df_clean[kolom_cadangan_satuan].astype("object").copy()
)

sat = (
    df_clean["Satuan"]
    .astype("string")
    .fillna("")
    .str.upper()
)

ada_vial = sat.str.contains("VIAL", regex=False, na=False).to_numpy(
    dtype=bool
)

if "Kategori Obat" in df_clean.columns:
    _obat = df_clean["Kategori Obat"].astype("string")
else:
    _obat = pd.Series("", index=df_clean.index, dtype="string")

ada_ampul = (
    KONVERSI_AMPUL
    & sat.str.contains(r"AMP[OU]L", regex=True, na=False).to_numpy(dtype=bool)
    & _obat.isin(list(AMPUL_OBAT)).to_numpy(dtype=bool)
    & ~ada_vial
)

ada_unit = ada_vial | ada_ampul

_pv = pd.to_numeric(
    sat.str.extract(r"(\d+)\s*VIAL", expand=False),
    errors="coerce"
).to_numpy(dtype=float, na_value=np.nan)

_pa = pd.to_numeric(
    sat.str.extract(r"(\d+)\s*AMP[OU]L", expand=False),
    errors="coerce"
).to_numpy(dtype=float, na_value=np.nan)

per_dus = np.ones(len(df_clean), dtype=float)

per_dus = np.where(
    ada_vial & ~np.isnan(_pv),
    _pv,
    per_dus
)

per_dus = np.where(
    ada_ampul & ~np.isnan(_pa),
    _pa,
    per_dus
)

df_clean["Vial per kemasan"] = np.where(
    ada_unit,
    per_dus,
    np.nan
)

df_clean["Jenis unit injeksi"] = np.where(
    ada_vial,
    "VIAL",
    np.where(ada_ampul, "AMPUL", "")
)

if KONVERSI_VIAL:
    jml = pd.to_numeric(
        df_clean["Jumlah"], errors="coerce"
    ).to_numpy(dtype=float, na_value=np.nan)

    asli = pd.to_numeric(
        df_clean[kolom_cadangan_jumlah], errors="coerce"
    ).to_numpy(dtype=float, na_value=np.nan)

    baru = np.where(
        ada_unit,
        jml * per_dus,
        asli
    )

    df_clean["Jumlah Setelah Konversi"] = baru

    df_clean.loc[ada_vial, "Kategori Satuan"] = "VIAL"
    df_clean.loc[ada_ampul, "Kategori Satuan"] = "AMPUL"

_v_awal = pd.to_numeric(
    df_clean["Jumlah Setelah Konversi"],
    errors="coerce"
)

df_clean["Temuan jumlah janggal"] = (
    _v_awal > BATAS_TANDAI_TEMUAN
).fillna(False)

df_clean["Nilai jumlah janggal"] = _v_awal.where(
    df_clean["Temuan jumlah janggal"]
)

df_clean["Jumlah janggal (dikosongkan)"] = False

if BATAS_JUMLAH_JANGGAL is not None:
    v = pd.to_numeric(
        df_clean["Jumlah Setelah Konversi"],
        errors="coerce"
    )

    janggal = (v > BATAS_JUMLAH_JANGGAL).fillna(False)

    df_clean.loc[janggal, "Jumlah Setelah Konversi"] = np.nan
    df_clean.loc[janggal, "Jumlah janggal (dikosongkan)"] = True

print(
    f"Konversi vial/ampul: "
    f"{'AKTIF' if KONVERSI_VIAL else 'dilewati'}"
    f" | baris vial: {int(ada_vial.sum()):,}"
    f" | baris ampul ({', '.join(sorted(AMPUL_OBAT))}): "
    f"{int(ada_ampul.sum()):,}"
    f" | dari {len(df_clean):,} baris"
)

if ada_unit.any():
    kolom_obat = (
        "Kategori Obat"
        if "Kategori Obat" in df_clean.columns
        else None
    )

    grup = (
        [kolom_obat, "Jenis unit injeksi"]
        if kolom_obat
        else ["Jenis unit injeksi"]
    )

    ring = (
        df_clean.loc[ada_unit]
        .groupby(grup, dropna=False)
        .agg(
            baris=("Vial per kemasan", "size"),
            vial_per_kemasan_median=("Vial per kemasan", "median"),
            jumlah_dus_median=("Jumlah", "median"),
            jumlah_vial_median=("Jumlah Setelah Konversi", "median"),
            jumlah_vial_maks=("Jumlah Setelah Konversi", "max"),
        )
    )

    display(ring)

    print(
        "Sebaran vial/ampul per kemasan:",
        df_clean.loc[ada_unit, "Vial per kemasan"]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    tanpa_angka = (
        (ada_vial & np.isnan(_pv))
        | (ada_ampul & np.isnan(_pa))
    )

    print(
        "Satuan vial/ampul tanpa angka di depannya "
        "(dianggap 1 per kemasan):",
        int(tanpa_angka.sum()),
        "baris"
    )

    _ampul_lain = (
        sat.str.contains(r"AMP[OU]L", regex=True, na=False).to_numpy(dtype=bool)
        & ~ada_unit
    )

    print(
        "Ampul pada obat lain atau yang tidak diproses:",
        int(_ampul_lain.sum()),
        "baris"
    )

print(
    "\nKategori Satuan sekarang:",
    df_clean["Kategori Satuan"].value_counts(dropna=False).to_dict()
)

besar = pd.to_numeric(
    df_clean["Jumlah Setelah Konversi"],
    errors="coerce"
)

kolom_tampil = [
    c for c in [
        "Nama PBF",
        "Nama Obat Jadi",
        "Kategori Obat",
        "Satuan",
        "Kategori Satuan",
        "Jumlah",
        "Jumlah Setelah Konversi",
        "Jenis Sarana",
    ]
    if c in df_clean.columns
]

print(
    "\n15 jumlah terbesar saat ini "
    "(periksa nilai aslinya di e-WAS):"
)

display(
    df_clean.loc[
        besar.nlargest(15).index,
        kolom_tampil
    ]
)

if BATAS_JUMLAH_JANGGAL is not None:
    print(
        f"Jumlah dikosongkan karena > {BATAS_JUMLAH_JANGGAL:,}: "
        f"{int(df_clean['Jumlah janggal (dikosongkan)'].sum()):,} baris"
    )

_tem = df_clean[df_clean["Temuan jumlah janggal"]]

jumlah_pbf = (
    _tem["Nama PBF"].nunique()
    if "Nama PBF" in _tem.columns
    else "?"
)

print(
    f"\nTEMUAN jumlah > {BATAS_TANDAI_TEMUAN:,}: "
    f"{len(_tem):,} transaksi dari {jumlah_pbf} PBF"
)

_kg = [
    c for c in [
        "Nama PBF",
        "Kategori Obat",
        "Nama Obat Jadi",
        "Satuan",
    ]
    if c in df_clean.columns
]

if len(_tem) > 0 and _kg:
    ringkas_temuan_pbf = (
        _tem.groupby(_kg, dropna=False)
        .agg(
            Transaksi=("Nilai jumlah janggal", "size"),
            Total=("Nilai jumlah janggal", "sum"),
            Terbesar=("Nilai jumlah janggal", "max"),
        )
        .sort_values("Total", ascending=False)
        .reset_index()
    )

    display(ringkas_temuan_pbf)

    _kd = [
        c for c in [
            "Nama PBF",
            "Nama Obat Jadi",
            "Kategori Obat",
            "Satuan",
            "Jumlah",
            "Nilai jumlah janggal",
            "Jenis Sarana",
            "Tujuan Penyaluran",
            "Nama Kota/Kab Tujuan",
            "Tanggal Penyaluran",
            "Tanggal Kedaluwarsa",
            "No. Faktur",
        ]
        if c in df_clean.columns
    ]

    _detail = _tem[_kd].copy()
    _detail.insert(0, "Indeks df_clean", _tem.index)

    with pd.ExcelWriter(BERKAS_TEMUAN, engine="openpyxl") as _p:
        ringkas_temuan_pbf.to_excel(
            _p,
            sheet_name="Ringkasan_per_PBF",
            index=False
        )

        _detail.sort_values(
            "Nilai jumlah janggal",
            ascending=False
        ).to_excel(
            _p,
            sheet_name="Detail_transaksi",
            index=False
        )

    print(
        f"Disimpan: {BERKAS_TEMUAN} "
        "(ringkasan per PBF dan detail transaksi; "
        "nilai asli perlu dicek di e-WAS)"
    )

    try:
        from google.colab import files as _f
        _f.download(BERKAS_TEMUAN)
    except Exception:
        pass

else:
    ringkas_temuan_pbf = pd.DataFrame()
    print("Tidak ada jumlah di atas batas temuan.")

_kol_tuj = [
    c for c in ["Tujuan Penyaluran", "Nama Sarana Tujuan"]
    if c in df_clean.columns
]

_retur = pd.Series(False, index=df_clean.index)

for _c in _kol_tuj:
    _retur |= (
        df_clean[_c]
        .astype("string")
        .str.contains(
            RETUR_POLA,
            case=False,
            na=False,
            regex=True
        )
    )

df_clean["Indikasi retur"] = _retur

print(
    f"\nIndikasi retur/pemusnahan "
    f"(kolom {_kol_tuj or 'tidak ada'}): "
    f"{int(_retur.sum()):,} baris"
)

if _retur.any() and _kol_tuj:
    print(
        "Nilai tujuan yang cocok "
        "(periksa apakah benar retur; sesuaikan RETUR_POLA bila perlu):"
    )

    print(
        df_clean.loc[_retur, _kol_tuj[0]]
        .astype("string")
        .value_counts()
        .head(10)
        .to_string()
    )

### **EDA**
Konfigurasi dan pemuatan data untuk eksplorasi.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

SUMBER_DATA = None
SUMBER_RAW = None
KOLOM_JUMLAH = "Jumlah Setelah Konversi"
KOLOM_JUMLAH_ASLI = "Jumlah"
KOLOM_SATUAN = "Kategori Satuan"
KOLOM_SARANA = "Jenis Sarana"
KOLOM_OBAT = "Kategori Obat"
KOLOM_TGL_SALUR = "Tanggal Penyaluran"
KOLOM_TGL_ED = "Tanggal Kedaluwarsa"
KOLOM_FAKTUR = "No. Faktur"
KANDIDAT_PBF = ["Nama PBF", "PBF", "Nama Pelapor", "Pelapor"]
KANDIDAT_TUJUAN = ["Tujuan Penyaluran", "Nama Sarana Tujuan", "Nama Tujuan"]
KOLOM_LOKASI = ["Alamat Tujuan", "Nama Kota/Kab Tujuan", "Nama Provinsi Tujuan"]
PLACEHOLDER_FAKTUR = {"general", "nihil", "tidak ada", "n/a", "na", "null", "none", "kosong", "-", "@"}
TOP_N = 15
MIN_TRANSAKSI_PBF = 30
TAHUN_MIN, TAHUN_MAX = 2015, 2035
EKSPOR = True
BERKAS_EKSPOR = "data_understanding.xlsx"

def _baca(path):
    nama = str(path).lower()
    if nama.endswith((".pkl", ".pickle")):
        return pd.read_pickle(path)
    if nama.endswith(".parquet"):
        return pd.read_parquet(path)
    if nama.endswith(".csv"):
        return pd.read_csv(path, low_memory=False)
    raise ValueError("Format harus .pkl, .parquet, atau .csv")

if "df_clean" not in globals():
    if SUMBER_DATA is None:
        raise NameError("df_clean belum ada di sesi. Isi SUMBER_DATA dengan path berkas.")
    df_clean = _baca(SUMBER_DATA)
if SUMBER_RAW is not None:
    df_raw = _baca(SUMBER_RAW)
if "df_raw" not in globals():
    df_raw = None

D = df_clean
PBF = next((k for k in KANDIDAT_PBF if k in D.columns), None)
TUJUAN = next((k for k in KANDIDAT_TUJUAN if k in D.columns), None)
def ada(*kolom):
    return all(k in D.columns for k in kolom)

print(f"df_clean: {len(D):,} baris x {D.shape[1]} kolom")
print(f"Kolom PBF: {PBF} | Kolom tujuan: {TUJUAN}")
HASIL = {}
TEMUAN = []

Profil kolom: tipe, jumlah kosong, dan nilai unik.

In [ ]:
profil = pd.DataFrame({
    "Tipe": D.dtypes.astype(str),
    "Terisi": D.notna().sum(),
    "Kosong": D.isna().sum(),
    "% Kosong": (D.isna().mean() * 100).round(2),
    "Nilai unik": D.nunique(dropna=True),
    "Contoh": [D[c].dropna().iloc[0] if D[c].notna().any() else None for c in D.columns],
}).reset_index(names="Kolom")
display(profil)
HASIL["Profil_Kolom"] = profil

tinggi_kosong = profil[profil["% Kosong"] >= 5]["Kolom"].tolist()
if tinggi_kosong:
    TEMUAN.append(f"Kolom dengan >=5% kosong: {', '.join(tinggi_kosong)}")

n_dup = int(D.duplicated().sum())
print(f"\nBaris duplikat penuh: {n_dup:,} ({n_dup / len(D) * 100:.3f}%)")
HASIL["Ringkasan_Duplikat"] = pd.DataFrame([["Duplikat penuh", n_dup]], columns=["Ukuran", "Jumlah"])
if n_dup:
    TEMUAN.append(f"{n_dup:,} baris duplikat penuh")
if not D.index.is_unique:
    print("PERINGATAN: indeks df_clean tidak unik.")

if df_raw is not None:
    print(f"\nData mentah: {len(df_raw):,} baris; df_clean: {len(D):,}; dikeluarkan: {len(df_raw) - len(D):,}")
    HASIL["Log_Eksklusi"] = pd.DataFrame([["Data mentah", len(df_raw)], ["df_clean", len(D)],
                                          ["Dikeluarkan", len(df_raw) - len(D)]], columns=["Tahap", "Baris"])
    TEMUAN.append(f"Pembersihan mengeluarkan {len(df_raw) - len(D):,} dari {len(df_raw):,} baris")

Tabel frekuensi kolom utama.

In [ ]:
def tabel_frekuensi(kolom, n=TOP_N):
    t = D[kolom].astype("string").fillna("(kosong)").value_counts()
    hasil = pd.DataFrame({"Jumlah": t, "%": (t / len(D) * 100).round(3)})
    hasil["% kumulatif"] = hasil["%"].cumsum().round(3)
    return hasil.head(n).rename_axis(kolom).reset_index()

for kolom in [KOLOM_SARANA, KOLOM_SATUAN, KOLOM_OBAT, "Sumber Kategori Obat"]:
    if ada(kolom):
        print(f"\n{kolom}  ({D[kolom].nunique(dropna=True)} nilai berbeda)")
        t = tabel_frekuensi(kolom, 30)
        display(t)
        HASIL[f"Frek_{kolom}"[:31]] = t

for etiket, kolom in [("PBF", PBF), ("Tujuan", TUJUAN)]:
    if kolom:
        print(f"\n{etiket}: {kolom}  ({D[kolom].nunique(dropna=True):,} nilai berbeda); {TOP_N} teratas")
        t = tabel_frekuensi(kolom)
        display(t)
        HASIL[f"Top_{etiket}"] = t

if ada(KOLOM_SATUAN):
    tinjau = D[KOLOM_SATUAN].astype("string").str.casefold().str.contains("tinjau", na=False)
    if tinjau.any():
        TEMUAN.append(f"{int(tinjau.sum()):,} baris berstatus 'tinjau' pada {KOLOM_SATUAN} (tidak dinilai IQR)")
if ada(KOLOM_OBAT):
    n_lain = int(D[KOLOM_OBAT].astype("string").isin(["Lainnya"]).sum())
    TEMUAN.append(f"Obat pantauan/TB = {len(D) - n_lain:,} baris; kategori 'Lainnya' = {n_lain:,} ({n_lain / len(D) * 100:.1f}%)")

Statistik jumlah dan diagram kotak.

In [ ]:
if not ada(KOLOM_JUMLAH):
    print(f"Kolom {KOLOM_JUMLAH} tidak ada; bagian ini dilewati.")

else:
    v = pd.to_numeric(D[KOLOM_JUMLAH], errors="coerce")

    print(
        f"{KOLOM_JUMLAH}: terisi {int(v.notna().sum()):,}; "
        f"tidak numerik/kosong {int(v.isna().sum()):,}; "
        f"nol {int((v == 0).sum()):,}; "
        f"negatif {int((v < 0).sum()):,}"
    )

    desk = v.describe(
        percentiles=[.01, .05, .25, .5, .75, .95, .99]
    ).to_frame("Nilai").T

    desk["skewness"] = v.skew()
    display(desk.round(3))

    HASIL["Deskripsi_Jumlah"] = desk.reset_index(drop=True)

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))

    positif = v[v > 0].dropna()

    if not positif.empty:
        ax[0].hist(
            np.log10(positif),
            bins=60,
            color="#6a9fb5"
        )
        ax[0].set_xlabel("log10(Jumlah Setelah Konversi)")
        ax[0].set_ylabel("Jumlah transaksi")
        ax[0].set_title("Distribusi jumlah (skala log)")
    else:
        ax[0].text(
            0.5, 0.5, "Tidak ada nilai positif",
            ha="center", va="center",
            transform=ax[0].transAxes
        )
        ax[0].set_title("Distribusi jumlah")

    if ada(KOLOM_SATUAN):
        urut = (
            D[KOLOM_SATUAN]
            .astype("string")
            .value_counts()
            .head(8)
            .index.tolist()
        )

        data = []
        label_valid = []

        for u in urut:
            nilai = pd.to_numeric(
                D.loc[
                    D[KOLOM_SATUAN].astype("string") == u,
                    KOLOM_JUMLAH
                ],
                errors="coerce"
            )

            nilai = nilai[nilai > 0].dropna()

            if not nilai.empty:
                data.append(np.log10(nilai))
                label_valid.append(str(u))

        if data:
            ax[1].boxplot(
                data,
                labels=label_valid,
                showfliers=False
            )
            ax[1].set_ylabel("log10(Jumlah)")
            ax[1].set_title(
                "Per Kategori Satuan (tanpa pencilan)"
            )
            ax[1].tick_params(axis="x", rotation=30)
        else:
            ax[1].text(
                0.5, 0.5, "Tidak ada nilai positif",
                ha="center", va="center",
                transform=ax[1].transAxes
            )
            ax[1].set_title("Per Kategori Satuan")
    else:
        ax[1].text(
            0.5, 0.5, "Kolom satuan tidak tersedia",
            ha="center", va="center",
            transform=ax[1].transAxes
        )
        ax[1].set_title("Per Kategori Satuan")

    plt.tight_layout()
    plt.show()

    for etiket, kolom in [
        ("Kategori Satuan", KOLOM_SATUAN),
        ("Kategori Obat", KOLOM_OBAT)
    ]:
        if ada(kolom):
            g = pd.DataFrame({
                "g": D[kolom].astype("string"),
                "v": v
            }).dropna(subset=["g", "v"])

            q = (
                g.groupby("g")["v"]
                .describe(percentiles=[.25, .5, .75, .95])
                .rename_axis(etiket)
                .reset_index()
            )

            q["Batas IQR (Q3+1,5 IQR)"] = (
                q["75%"] + 1.5 * (q["75%"] - q["25%"])
            )

            q["% di atas batas"] = [
                float(
                    (
                        g.loc[g["g"] == k, "v"] > b
                    ).mean() * 100
                )
                for k, b in zip(
                    q[etiket],
                    q["Batas IQR (Q3+1,5 IQR)"]
                )
            ]

            print(
                f"\nJumlah menurut {etiket} "
                "(kelompok kecil memiliki batas IQR yang tidak stabil)"
            )
            display(q.round(3))

            HASIL[f"Jumlah_per_{etiket}"[:31]] = q

    if ada(KOLOM_JUMLAH_ASLI):
        asli = pd.to_numeric(
            D[KOLOM_JUMLAH_ASLI],
            errors="coerce"
        )

        rasio = (
            v / asli.replace(0, np.nan)
        ).replace([np.inf, -np.inf], np.nan)

        rasio_valid = rasio.dropna()

        if not rasio_valid.empty:
            print(
                f"\nRasio {KOLOM_JUMLAH} / {KOLOM_JUMLAH_ASLI}: "
                f"median {rasio_valid.median():.3f}; "
                f"baris dengan rasio != 1: "
                f"{int((rasio_valid.round(6) != 1).sum()):,}"
            )
        else:
            print(
                "\nRasio tidak dapat dihitung karena "
                "tidak ada nilai valid."
            )

Pemeriksaan tanggal penyaluran dan kedaluwarsa.

In [ ]:
if not ada(KOLOM_TGL_SALUR, KOLOM_TGL_ED):
    print("Kolom tanggal tidak lengkap; bagian ini dilewati.")
else:
    tp = pd.to_datetime(D[KOLOM_TGL_SALUR], errors="coerce").dt.normalize()
    te = pd.to_datetime(D[KOLOM_TGL_ED], errors="coerce").dt.normalize()
    print(f"Terbaca: penyaluran {int(tp.notna().sum()):,}/{int(D[KOLOM_TGL_SALUR].notna().sum()):,}; "
          f"kedaluwarsa {int(te.notna().sum()):,}/{int(D[KOLOM_TGL_ED].notna().sum()):,}")
    print("Contoh mentah:", D[KOLOM_TGL_SALUR].head(3).tolist(), D[KOLOM_TGL_ED].dropna().head(3).tolist())

    def luar_rentang(s):
        return s.notna() & ((s.dt.year < TAHUN_MIN) | (s.dt.year > TAHUN_MAX))
    n_luar_p, n_luar_e = int(luar_rentang(tp).sum()), int(luar_rentang(te).sum())
    print(f"Tanggal di luar {TAHUN_MIN}-{TAHUN_MAX}: penyaluran {n_luar_p:,}; kedaluwarsa {n_luar_e:,}")
    if n_luar_p or n_luar_e:
        TEMUAN.append(f"Tanggal tidak wajar (di luar {TAHUN_MIN}-{TAHUN_MAX}): penyaluran {n_luar_p:,}, kedaluwarsa {n_luar_e:,}")

    wajar = tp[tp.notna() & ~luar_rentang(tp)]
    print(f"Rentang tanggal penyaluran (wajar): {wajar.min().date()} s.d. {wajar.max().date()}")
    per_bulan = wajar.dt.to_period("M").value_counts().sort_index()
    per_tahun = wajar.dt.year.value_counts().sort_index()
    display(pd.DataFrame({"Jumlah": per_tahun, "%": (per_tahun / per_tahun.sum() * 100).round(2)}).rename_axis("Tahun"))
    HASIL["Per_Bulan"] = per_bulan.rename_axis("Bulan").reset_index(name="Jumlah").astype({"Bulan": str})
    if len(per_tahun) > 1:
        TEMUAN.append(f"Tahun terpadat: {int(per_tahun.idxmax())} ({per_tahun.max() / per_tahun.sum() * 100:.1f}% data)")

    fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
    ax[0].bar(range(len(per_bulan)), per_bulan.to_numpy(), color="#6a9fb5")
    langkah = max(1, len(per_bulan) // 12)
    ax[0].set_xticks(range(0, len(per_bulan), langkah))
    ax[0].set_xticklabels([str(p) for p in per_bulan.index[::langkah]], rotation=45, ha="right", fontsize=8)
    ax[0].set_title("Transaksi per bulan penyaluran")

    ok = tp.notna() & te.notna() & ~luar_rentang(tp) & ~luar_rentang(te)
    sisa_hari = (te - tp).dt.days.where(ok)
    batas6 = tp + pd.DateOffset(months=6)
    status = pd.Series("Tanggal kosong / tidak wajar", index=D.index, dtype="object")
    status[ok & (te <= tp)] = "Kedaluwarsa <= penyaluran"
    status[ok & (te > tp) & (te <= batas6)] = "0-6 bulan"
    status[ok & (te > batas6) & (te <= tp + pd.DateOffset(months=12))] = ">6-12 bulan"
    status[ok & (te > tp + pd.DateOffset(months=12)) & (te <= tp + pd.DateOffset(months=24))] = ">12-24 bulan"
    status[ok & (te > tp + pd.DateOffset(months=24))] = ">24 bulan"
    urutan = ["Kedaluwarsa <= penyaluran", "0-6 bulan", ">6-12 bulan", ">12-24 bulan", ">24 bulan", "Tanggal kosong / tidak wajar"]
    t_sisa = status.value_counts().reindex(urutan, fill_value=0)
    ax[1].barh(range(len(t_sisa)), t_sisa.to_numpy(), color="#c98b5a")
    ax[1].set_yticks(range(len(t_sisa)))
    ax[1].set_yticklabels(t_sisa.index, fontsize=8)
    ax[1].invert_yaxis()
    ax[1].set_title("Sisa masa simpan saat penyaluran")
    plt.tight_layout()
    plt.show()
    tabel_sisa = pd.DataFrame({"Jumlah": t_sisa, "%": (t_sisa / len(D) * 100).round(3)}).rename_axis("Sisa masa simpan").reset_index()
    display(tabel_sisa)
    HASIL["Sisa_Masa_Simpan"] = tabel_sisa
    TEMUAN.append(f"Kedaluwarsa <= penyaluran: {int(t_sisa['Kedaluwarsa <= penyaluran']):,}; "
                  f"0-6 bulan: {int(t_sisa['0-6 bulan']):,}; tanggal kosong/tidak wajar: {int(t_sisa['Tanggal kosong / tidak wajar']):,}")
    if ok.any():
        print("\nSisa masa simpan (hari) pada baris wajar:")
        display(sisa_hari.describe(percentiles=[.01, .05, .25, .5, .75, .95]).round(1).to_frame("hari").T)

Flag kualitas data per transaksi.

In [ ]:
flag = pd.DataFrame(index=D.index)
if ada(KOLOM_FAKTUR):
    f = D[KOLOM_FAKTUR].astype("string").str.strip()
    flag["Faktur bermasalah"] = (f.isna() | ~f.str.contains(r"[A-Za-z0-9]", regex=True, na=False)
                                 | f.str.casefold().isin(PLACEHOLDER_FAKTUR)).to_numpy(dtype=bool)
lokasi_ada = [k for k in KOLOM_LOKASI if k in D.columns]
if lokasi_ada:
    kosong_lokasi = pd.Series(False, index=D.index)
    for k in lokasi_ada:
        kosong_lokasi |= D[k].isna() | D[k].astype("string").str.strip().eq("").fillna(True)
    flag["Lokasi tidak lengkap"] = kosong_lokasi.to_numpy(dtype=bool)
if ada(KOLOM_TGL_ED):
    flag["Kedaluwarsa kosong/tidak terbaca"] = pd.to_datetime(D[KOLOM_TGL_ED], errors="coerce").isna().to_numpy(dtype=bool)

if flag.empty:
    print("Tidak ada kolom faktur/lokasi/kedaluwarsa untuk diperiksa.")
else:
    ringkas = pd.DataFrame({"Baris bermasalah": flag.sum(), "%": (flag.mean() * 100).round(3)}).rename_axis("Masalah").reset_index()
    display(ringkas)
    HASIL["Kualitas_Pelaporan"] = ringkas
    for _, r in ringkas.iterrows():
        TEMUAN.append(f"{r['Masalah']}: {int(r['Baris bermasalah']):,} baris ({r['%']:.2f}%)")

    if PBF:
        g = flag.groupby(D[PBF].astype("string"))
        rekap = g.sum().astype(int)
        rekap.insert(0, "Transaksi", g.size())
        for k in flag.columns:
            rekap[f"% {k}"] = (rekap[k] / rekap["Transaksi"] * 100).round(2)
        rekap = rekap[rekap["Transaksi"] >= MIN_TRANSAKSI_PBF].sort_values("Transaksi", ascending=False).rename_axis("PBF").reset_index()
        print(f"\nRekap per PBF (>= {MIN_TRANSAKSI_PBF} transaksi), {TOP_N} teratas menurut volume")
        display(rekap.head(TOP_N))
        HASIL["Rekap_Per_PBF"] = rekap

Tabulasi silang kategori obat × jenis sarana.

In [ ]:
if ada(KOLOM_OBAT, KOLOM_SARANA):
    ct = pd.crosstab(D[KOLOM_OBAT].astype("string").fillna("(kosong)"), D[KOLOM_SARANA].astype("string").fillna("(kosong)"))
    print("Jumlah transaksi: Kategori Obat x Jenis Sarana")
    display(ct)
    pct = (ct.div(ct.sum(axis=1), axis=0) * 100).round(1)
    print("Persen per baris (distribusi sarana tujuan tiap kategori obat)")
    display(pct)
    HASIL["Obat_x_Sarana"] = ct.reset_index()
    HASIL["Obat_x_Sarana_pct"] = pct.reset_index()

    fig, ax = plt.subplots(figsize=(0.9 * ct.shape[1] + 3, 0.5 * ct.shape[0] + 2))
    gambar = ax.imshow(pct.to_numpy(dtype=float), cmap="Blues", aspect="auto")
    ax.set_xticks(range(ct.shape[1]))
    ax.set_xticklabels(ct.columns, rotation=40, ha="right", fontsize=8)
    ax.set_yticks(range(ct.shape[0]))
    ax.set_yticklabels(ct.index, fontsize=8)
    for i in range(pct.shape[0]):
        for j in range(pct.shape[1]):
            ax.text(j, i, f"{pct.iat[i, j]:.0f}", ha="center", va="center", fontsize=7)
    ax.set_title("% transaksi per Kategori Obat menurut Jenis Sarana")
    plt.colorbar(gambar, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()

    nama_sarana = D[KOLOM_SARANA].astype("string").str.strip().value_counts()
    print("Nilai Jenis Sarana yang jarang (<0,5% data); periksa apakah perlu dipetakan:")
    jarang = nama_sarana[nama_sarana / len(D) < 0.005]
    display(jarang.rename_axis("Jenis Sarana").reset_index(name="Jumlah") if len(jarang) else pd.DataFrame({"info": ["tidak ada"]}))
else:
    print("Kolom Kategori Obat / Jenis Sarana tidak ada; bagian ini dilewati.")

Ringkasan temuan EDA.

In [ ]:
ringkasan = pd.DataFrame({"No": range(1, len(TEMUAN) + 1), "Temuan": TEMUAN})
pd.set_option("display.max_colwidth", 200)
display(ringkasan)
HASIL = {"Ringkasan_Temuan": ringkasan, **HASIL}

if EKSPOR:
    try:
        with pd.ExcelWriter(BERKAS_EKSPOR, engine="openpyxl") as penulis:
            for nama, tabel in HASIL.items():
                tabel.to_excel(penulis, sheet_name=str(nama)[:31], index=False)
        print(f"Disimpan: {BERKAS_EKSPOR}")
    except Exception as galat:
        print(f"Ekspor Excel gagal ({galat}); menyimpan sebagai CSV.")
        for nama, tabel in HASIL.items():
            tabel.to_csv(f"du_{nama}.csv", index=False)

**PENYESUAIAN LOKASI (INDIKATOR E5)**

Indikator E5: E5a (lokasi kosong atau tidak terbaca) dan E5b (kota-provinsi tidak cocok). E4 dan E5 hanya indikator pendukung dan tidak masuk perhitungan posterior.

In [ ]:
import re
import numpy as np
import pandas as pd
from IPython.display import display

KOLOM_ALAMAT = "Alamat Tujuan"
KOLOM_KOTA = "Nama Kota/Kab Tujuan"
KOLOM_PROV = "Nama Provinsi Tujuan"
KANDIDAT_PBF = ["Nama PBF", "PBF", "Nama Pelapor", "Pelapor"]
MIN_DUKUNGAN_KOTA = 20
MIN_PANGSA_PROVINSI = 0.80
TAMBAH_KE_CATATAN = True
BERKAS_E5 = "e5_lokasi.xlsx"
DAFTAR_PROVINSI = {
    "aceh", "sumatera utara", "sumatera barat", "riau", "kepulauan riau", "jambi", "sumatera selatan",
    "kepulauan bangka belitung", "bangka belitung", "bengkulu", "lampung", "dki jakarta", "jawa barat",
    "jawa tengah", "di yogyakarta", "jawa timur", "banten", "bali", "nusa tenggara barat", "nusa tenggara timur",
    "kalimantan barat", "kalimantan tengah", "kalimantan selatan", "kalimantan timur", "kalimantan utara",
    "sulawesi utara", "gorontalo", "sulawesi tengah", "sulawesi barat", "sulawesi selatan", "sulawesi tenggara",
    "maluku", "maluku utara", "papua", "papua barat", "papua selatan", "papua tengah", "papua pegunungan",
    "papua barat daya",
}
RUMPUN_PROVINSI = {p: "rumpun papua" for p in
                   ["papua", "papua barat", "papua selatan", "papua tengah", "papua pegunungan", "papua barat daya"]}
PROVINSI_ACUAN_MANUAL = {
    "sumbawa": "nusa tenggara barat",
}
POLA_BUKAN_KOTA = r"(?:^|\s)(?:pt|cv|ud|pbf|apotek|klinik|toko)(?:\s|$)"
SINONIM_PROVINSI = {
    "dki jakarta": "dki jakarta", "jakarta": "dki jakarta", "d k i jakarta": "dki jakarta",
    "di yogyakarta": "di yogyakarta", "daerah istimewa yogyakarta": "di yogyakarta", "yogyakarta": "di yogyakarta",
    "jogjakarta": "di yogyakarta", "d i yogyakarta": "di yogyakarta",
    "nad": "aceh", "nanggroe aceh darussalam": "aceh", "kepulauan riau": "kepulauan riau", "kepri": "kepulauan riau",
}

if "df_clean" not in globals():
    raise NameError("df_clean belum ada di sesi.")
ada_k = [k for k in [KOLOM_ALAMAT, KOLOM_KOTA, KOLOM_PROV] if k in df_clean.columns]
if not ada_k:
    raise ValueError("Tidak ada kolom lokasi di df_clean; periksa nama kolom di bagian konfigurasi.")
PBF = next((k for k in KANDIDAT_PBF if k in df_clean.columns), None)
D = df_clean

def bersih(seri):
    s = seri.astype("string").str.casefold().str.strip()
    s = s.str.replace(r"[.,;:()/\\-]+", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
    return s.replace("", pd.NA)

def norm_kota(seri):
    s = bersih(seri)
    s = s.str.replace(r"^(kota administrasi|kota adm|kota|kabupaten|kab)\s+", "", regex=True)
    s = s.str.replace(r"\s+(kota|kab|kabupaten)$", "", regex=True)
    return s.replace("", pd.NA)

def norm_prov(seri):
    s = bersih(seri).str.replace(r"^(provinsi|prov)\s+", "", regex=True)
    s = s.map(lambda x: SINONIM_PROVINSI.get(x, x) if isinstance(x, str) else x)
    return s.replace({"bangka belitung": "kepulauan bangka belitung"})

def ke_rumpun(seri):
    return seri.map(lambda x: RUMPUN_PROVINSI.get(x, x) if isinstance(x, str) else x)

kosong = pd.DataFrame({k: bersih(D[k]).isna() for k in ada_k})
e5a_kosong = kosong.any(axis=1).to_numpy(dtype=bool)

tidak_terbaca = np.zeros(len(D), dtype=bool)
if KOLOM_PROV in D.columns:
    _p = norm_prov(D[KOLOM_PROV])
    tidak_terbaca |= (_p.notna() & ~_p.isin(DAFTAR_PROVINSI)).to_numpy(dtype=bool)
if KOLOM_KOTA in D.columns:
    tidak_terbaca |= bersih(D[KOLOM_KOTA]).str.contains(POLA_BUKAN_KOTA, regex=True, na=False).to_numpy(dtype=bool)
e5a = e5a_kosong | tidak_terbaca

e5b = np.zeros(len(D), dtype=bool)
tabel_ambigu = pd.DataFrame()
tabel_peta = pd.DataFrame()
if KOLOM_KOTA in D.columns and KOLOM_PROV in D.columns:
    kota = norm_kota(D[KOLOM_KOTA])
    prov = norm_prov(D[KOLOM_PROV])
    prov = ke_rumpun(prov)
    ada = kota.notna() & prov.notna() & ~pd.Series(tidak_terbaca, index=D.index)
    hitung = (pd.DataFrame({"kota": kota[ada], "prov": prov[ada]})
              .groupby(["kota", "prov"]).size().rename("n").reset_index())
    total_kota = hitung.groupby("kota")["n"].transform("sum")
    hitung["pangsa"] = hitung["n"] / total_kota
    hitung["total"] = total_kota
    utama = hitung.sort_values("n", ascending=False).drop_duplicates("kota")
    peta = utama[(utama["total"] >= MIN_DUKUNGAN_KOTA) & (utama["pangsa"] >= MIN_PANGSA_PROVINSI)]
    prov_normal = dict(zip(peta["kota"], peta["prov"]))
    for _k, _v in PROVINSI_ACUAN_MANUAL.items():
        prov_normal[_k] = RUMPUN_PROVINSI.get(_v, _v)
    tabel_peta = peta.rename(columns={"kota": "Kota (ternormalisasi)", "prov": "Provinsi dominan",
                                      "n": "Baris provinsi dominan", "pangsa": "Pangsa", "total": "Total baris kota"})
    ambigu = utama[(utama["total"] >= MIN_DUKUNGAN_KOTA) & (utama["pangsa"] < MIN_PANGSA_PROVINSI)
                   & ~utama["kota"].isin(list(PROVINSI_ACUAN_MANUAL))]["kota"]
    tabel_ambigu = (hitung[hitung["kota"].isin(ambigu)].sort_values(["kota", "n"], ascending=[True, False])
                    .rename(columns={"kota": "Kota (ternormalisasi)", "prov": "Provinsi", "n": "Baris", "pangsa": "Pangsa", "total": "Total baris kota"}))
    normal = kota.map(prov_normal)
    e5b = (ada & normal.notna() & (prov != normal)).to_numpy(dtype=bool)
else:
    print("Kolom kota atau provinsi tidak ada; E5b dilewati.")

e5_final = e5a | e5b
D["E5 kosong"] = e5a.astype(int)
D["E5 kota-provinsi tidak cocok"] = e5b.astype(int)
D["E5 lokasi bermasalah (final)"] = e5_final.astype(int)

if TAMBAH_KE_CATATAN and "Catatan Pendukung" in D.columns:
    tambahan = np.full(len(D), "", dtype=object)
    tambahan = np.where(e5a, tambahan + "Lokasi kosong/tidak terbaca; ", tambahan)
    tambahan = np.where(e5b, tambahan + "Kota-provinsi tidak cocok; ", tambahan)
    lama = D["Catatan Pendukung"].astype("string").fillna("-")
    gabung = np.where(tambahan == "", lama, np.where(lama == "-", pd.Series(tambahan).str.rstrip("; "),
                                                        lama + "; " + pd.Series(tambahan).str.rstrip("; ")))
    D["Catatan Pendukung"] = pd.Series(gabung, index=D.index)

n = len(D)
ringkas = pd.DataFrame([
    ["E5a: lokasi kosong atau tidak terbaca (gabungan)", int(e5a.sum()), e5a.mean() * 100],
    ["   dari itu: kosong", int(e5a_kosong.sum()), e5a_kosong.mean() * 100],
    ["   dari itu: tidak terbaca (provinsi bukan nama provinsi / kota berisi nama perusahaan)", int(tidak_terbaca.sum()), tidak_terbaca.mean() * 100],
    ["E5b: kota-provinsi tidak cocok", int(e5b.sum()), e5b.mean() * 100],
    ["E5 final (E5a atau E5b)", int(e5_final.sum()), e5_final.mean() * 100],
], columns=["Komponen", "Baris", "%"])
print(f"Kolom dipakai: {ada_k}")
display(ringkas.round({"%": 3}))
if len(tabel_peta):
    print(f"\nKota dengan provinsi dominan jelas: {len(tabel_peta):,}; kota ambigu (perlu cek manual): {tabel_ambigu['Kota (ternormalisasi)'].nunique() if len(tabel_ambigu) else 0}")
if len(tabel_ambigu):
    display(tabel_ambigu.head(20).round({"Pangsa": 3}))

kolom_contoh = [c for c in [KOLOM_ALAMAT, KOLOM_KOTA, KOLOM_PROV] if c in D.columns]
contoh_b = D.loc[e5b, kolom_contoh].head(20).copy()
if KOLOM_ALAMAT in D.columns and len(contoh_b):
    _al = bersih(contoh_b[KOLOM_ALAMAT]).fillna("")
    _k = norm_kota(contoh_b[KOLOM_KOTA]).fillna("")
    _p = norm_prov(contoh_b[KOLOM_PROV]).fillna("")
    _petunjuk = []
    for a, k, p in zip(_al, _k, _p):
        if k and k in a: _petunjuk.append("kota ada di alamat: provinsi kemungkinan salah")
        elif p and p in a: _petunjuk.append("provinsi ada di alamat: kota kemungkinan salah")
        else: _petunjuk.append("alamat tidak memuat kota/provinsi: cek manual")
    contoh_b["Petunjuk alamat"] = _petunjuk
if len(contoh_b):
    print("\nContoh E5b (kota-provinsi tidak cocok); periksa apakah kesalahan input atau nama kembar:")
    display(contoh_b)
rekap = pd.DataFrame()
if PBF:
    g = pd.DataFrame({"PBF": D[PBF].astype("string"), "E5a": e5a, "E5b": e5b, "E5 final": e5_final})
    rekap = g.groupby("PBF").agg(Transaksi=("E5a", "size"), E5a=("E5a", "sum"), E5b=("E5b", "sum"),
                                 E5_final=("E5 final", "sum")).reset_index()
    rekap["% E5 final"] = (rekap["E5_final"] / rekap["Transaksi"] * 100).round(2)
    rekap = rekap[rekap["Transaksi"] >= 30].sort_values("% E5 final", ascending=False)
    print("\nPBF dengan persentase masalah lokasi tertinggi (>= 30 transaksi), 15 teratas")
    display(rekap.head(15))

try:
    with pd.ExcelWriter(BERKAS_E5, engine="openpyxl") as penulis:
        ringkas.to_excel(penulis, sheet_name="Ringkasan", index=False)
        tabel_peta.to_excel(penulis, sheet_name="Peta_Kota_Provinsi", index=False)
        tabel_ambigu.to_excel(penulis, sheet_name="Kota_Ambigu", index=False)
        contoh_b.to_excel(penulis, sheet_name="Contoh_E5b")
        rekap.to_excel(penulis, sheet_name="Rekap_PBF", index=False)
    print(f"\nDisimpan: {BERKAS_E5}")
except Exception as galat:
    print(f"Ekspor Excel gagal: {galat}")
print("Kolom baru di df_clean: E5 kosong, E5 kota-provinsi tidak cocok, E5 lokasi bermasalah (final)")

### **Model Bayesian**
Konfigurasi parameter model: prior, LR, aturan E3, dan skema.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

SUMBER_DATA = None

KOLOM_JUMLAH = "Jumlah Setelah Konversi"
KELOMPOK_E1 = ["Kelompok E1"]
KELOMPOK_E3 = ["Kategori Obat", "Kategori Satuan"]
MIN_TRANSAKSI_IQR = 30
FAKTOR_IQR = 1.5
E3_FALLBACK_KE_E1 = True
E1_NA_BILA_E3 = True

ATURAN_E3 = {
    "Ketamine":  {"mode": "luar",  "sarana": {"Rumah Sakit"}},
    "Meropenem": {"mode": "luar",  "sarana": {"Rumah Sakit"}},
    "Orlistat":  {"mode": "dalam", "sarana": {"Klinik", "Apotek"}},
    "Cefixime":  {"mode": "dalam", "sarana": {"Klinik", "Apotek"}},
    "TBC":       {"mode": "dalam", "sarana": {"Apotek", "Klinik"}},
}
TOGGLE_CEFIXIME_RULE = "WAWANCARA"
TOGGLE_EXPIRY_RULE = "FORM_T009"
TOGGLE_E2K_RULE = "FORM_T005_T011"
E1_PBF_RENDAH = True
E1_RS_BIASA_AMAN = True
E2_RS_BIASA_AMAN = False
E2_SPLIT_SARANA = True
SARANA_E2_KUAT = {"apotek", "klinik"}
OBAT_RS_DIKAJI = {"Ketamine", "Meropenem"}
E1_VIAL_PER_OBAT = True
for _nama, _nilai, _boleh in [("TOGGLE_CEFIXIME_RULE", TOGGLE_CEFIXIME_RULE, ("WAWANCARA", "FORM_T012")),
                              ("TOGGLE_EXPIRY_RULE", TOGGLE_EXPIRY_RULE, ("WAWANCARA", "FORM_T009")),
                              ("TOGGLE_E2K_RULE", TOGGLE_E2K_RULE, ("WAWANCARA", "FORM_T005_T011"))]:
    if _nilai not in _boleh:
        raise ValueError(f"{_nama} harus salah satu dari {_boleh}")
E2K_ISOLASI = TOGGLE_E2K_RULE == "WAWANCARA"

E3_KUAT = {"Ketamine", "Meropenem", "Orlistat"} | ({"Cefixime"} if TOGGLE_CEFIXIME_RULE == "FORM_T012" else set())
HIERARKI_OBAT = ["Ketamine", "Meropenem", "Orlistat", "Cefixime", "TBC"]

E2K_TERMASUK_KOSONG = False
TAHUN_WAJAR = (2015, 2040)
SARANA_DIKENAL = {"Apotek", "PBF", "Rumah Sakit", "Puskesmas",
                  "Klinik", "Balai Kesehatan", "Dinas Kesehatan"}

PRIOR_UTAMA = 0.15
LR_UTAMA = {"E1": 5.0, "E2": 25.0 if TOGGLE_EXPIRY_RULE == "WAWANCARA" else 10.0, "E2p": 25.0, "E2k": 6.0, "E3": 20.0, "E3b": 6.0}
PRIOR_TABEL = [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]

AMBANG_SEDANG = 0.30
AMBANG_TINGGI = 0.70
SKEMA_UTAMA = "A"
MIN_INDIKATOR_BANDING = 1
TOLERANSI = 1e-12

OAT = {
    "Prior": [0.05, 0.30, 0.50],
    "LR E1": [2.5, 10.0],
    "LR E2": [5.0, 10.0] if TOGGLE_EXPIRY_RULE == "WAWANCARA" else [5.0, 25.0],
    "LR E3": [10.0, 40.0],
    "LR E3b": [3.0, 10.0],
    "LR E2p": [10.0, 40.0],
    "LR E2k": [3.0, 10.0],
    "Ambang Sedang": [0.20, 0.40],
    "Ambang Tinggi": [0.60, 0.80],
}
GABUNGAN = [("G1 Konservatif", 0.05, 0.5), ("G2 Sensitif", 0.40, 2.0)]

PRIOR_SWEEP = [round(0.05 + 0.01 * i, 2) for i in range(46)]
PRIOR_TAMPIL = [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]
LR_SWEEP = [1.5, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, 30, 50]
PRIOR_GRID = [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]
PENGALI_LR = [0.25, 0.5, 1.0, 2.0, 4.0]
AMBANG_SEDANG_SWEEP = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
AMBANG_TINGGI_SWEEP = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]
N_MC = 5000
MC_SEED = 2026
MC_PRIOR = (0.05, 0.50)
MC_LR = (1.5, 30.0)
MC_AMBANG_SEDANG = (0.20, 0.40)
MC_AMBANG_TINGGI = (0.60, 0.80)
UJI_FAKTOR_IQR = [1.0, 1.5, 2.0, 3.0]
UJI_BULAN_E2 = [3, 6, 9, 12]
UJI_MIN_TRANSAKSI = [10, 30, 100]

PLACEHOLDER_FAKTUR = {"general", "nihil", "tidak ada", "n/a", "na", "null", "none", "kosong", "-"}
KOLOM_LOKASI = ["Alamat Tujuan", "Nama Kota/Kab Tujuan", "Nama Provinsi Tujuan"]

EKSPOR = True
BERKAS_EKSPOR = "hasil_bayes_oat.xlsx"
EKSPOR_PER_BARIS = False
BERKAS_PER_BARIS = "hasil_bayes_per_transaksi.csv"

LABEL = ["RENDAH", "SEDANG", "TINGGI", "BELUM DAPAT DINILAI"]

if "df_clean" not in globals():
    if SUMBER_DATA is None:
        raise NameError("df_clean belum ada di sesi. Jalankan notebook pembersihan dahulu, "
                        "atau isi SUMBER_DATA dengan path berkas.")
    nama_berkas = str(SUMBER_DATA).lower()
    if nama_berkas.endswith((".pkl", ".pickle")):
        df_clean = pd.read_pickle(SUMBER_DATA)
    elif nama_berkas.endswith(".parquet"):
        df_clean = pd.read_parquet(SUMBER_DATA)
    elif nama_berkas.endswith(".csv"):
        df_clean = pd.read_csv(SUMBER_DATA, low_memory=False)
    else:
        raise ValueError("Format SUMBER_DATA harus .pkl, .parquet, atau .csv")

WAJIB = [KOLOM_JUMLAH, "Kategori Satuan", "Jenis Sarana", "Kategori Obat",
         "Tanggal Penyaluran", "Tanggal Kedaluwarsa"]
kurang = [k for k in WAJIB if k not in df_clean.columns]
if kurang:
    raise ValueError(f"Kolom belum tersedia di df_clean: {kurang}")
_ks = df_clean["Kategori Satuan"].astype("string")
_inj = _ks.isin(["VIAL", "AMPUL"]) & E1_VIAL_PER_OBAT
df_clean["Kelompok E1"] = _ks.where(~_inj, _ks + " | " + df_clean["Kategori Obat"].astype("string").fillna("(obat kosong)"))
if not df_clean.index.is_unique:
    raise ValueError("Indeks df_clean tidak unik. Jalankan: df_clean = df_clean.reset_index(drop=True)")

print(f"df_clean: {len(df_clean):,} baris, {df_clean.shape[1]} kolom")
display(df_clean["Kategori Obat"].value_counts(dropna=False)
        .rename_axis("Kategori Obat").reset_index(name="Jumlah Baris"))

Fungsi bantu dan indikator E1, E2, E2k, E3.

In [ ]:
def bool_np(serie):
    """Seri boolean (boleh mengandung NA) -> array numpy bool; NA dianggap False."""
    return serie.fillna(False).to_numpy(dtype=bool)

def tandai_iqr(data, kelompok, kolom):
    """1 jika nilai > Q3 + FAKTOR_IQR*IQR dalam kelompoknya, 0 jika tidak.
    NA bila kelompok < MIN_TRANSAKSI_IQR, nilai kosong, atau kelompok mengandung 'tinjau'.
    Mengembalikan (status Int64, tabel batas per kelompok)."""
    hasil = pd.Series(pd.NA, index=data.index, dtype="Int64")
    kolom_stat = ["Jumlah_Transaksi", "Q1", "Q3", "Batas_Atas", "Dinilai"]
    kosong = pd.DataFrame(columns=list(kelompok) + kolom_stat)
    if data.empty:
        return hasil, kosong

    f = data[kelompok].astype("string").copy()
    f["_v"] = pd.to_numeric(data[kolom], errors="coerce")
    valid = f[kelompok].notna().all(axis=1) & f["_v"].notna()
    tinjau = pd.concat(
        [f[c].str.casefold().str.contains("tinjau", na=False) for c in kelompok], axis=1
    ).any(axis=1)
    layak = (valid & ~tinjau).to_numpy(dtype=bool)
    if not layak.any():
        return hasil, kosong

    g = f.loc[layak].groupby(kelompok)["_v"]
    stat = pd.DataFrame({"Jumlah_Transaksi": g.size(),
                         "Q1": g.quantile(0.25), "Q3": g.quantile(0.75)})
    stat["Batas_Atas"] = stat["Q3"] + FAKTOR_IQR * (stat["Q3"] - stat["Q1"])
    stat = stat.reset_index()

    idx = f.index[layak]
    gabung = f.loc[layak, kelompok].reset_index(drop=True).merge(stat, on=kelompok, how="left")
    n_grup = gabung["Jumlah_Transaksi"].to_numpy(dtype=float)
    batas = gabung["Batas_Atas"].to_numpy(dtype=float)
    nilai = f.loc[layak, "_v"].to_numpy(dtype=float)
    dinilai = n_grup >= MIN_TRANSAKSI_IQR
    hasil.loc[idx[dinilai]] = (nilai > batas).astype(int)[dinilai]
    stat["Dinilai"] = stat["Jumlah_Transaksi"] >= MIN_TRANSAKSI_IQR
    return hasil, stat

tak_numerik = int(pd.to_numeric(df_clean[KOLOM_JUMLAH], errors="coerce").isna().sum())
print(f"{KOLOM_JUMLAH}: {tak_numerik:,} nilai tidak numerik/kosong")

e1_rev, batas_e1 = tandai_iqr(df_clean, KELOMPOK_E1, KOLOM_JUMLAH)
df_clean["E1_rev"] = e1_rev
print("\nE1 — batas IQR per kelompok")
display(batas_e1)

penyaluran = pd.to_datetime(df_clean["Tanggal Penyaluran"], errors="coerce").dt.normalize()
kedaluwarsa = pd.to_datetime(df_clean["Tanggal Kedaluwarsa"], errors="coerce").dt.normalize()
print(f"\nTanggal Penyaluran terbaca: {int(penyaluran.notna().sum()):,} dari "
      f"{int(df_clean['Tanggal Penyaluran'].notna().sum()):,} | "
      f"Tanggal Kedaluwarsa terbaca: {int(kedaluwarsa.notna().sum()):,} dari "
      f"{int(df_clean['Tanggal Kedaluwarsa'].notna().sum()):,}")
print("Contoh nilai mentah:", df_clean["Tanggal Penyaluran"].head(3).tolist(),
      df_clean["Tanggal Kedaluwarsa"].dropna().head(3).tolist())

tanggal_lengkap = penyaluran.notna() & kedaluwarsa.notna()
batas_enam_bulan = penyaluran + pd.DateOffset(months=6)
perlu_klarifikasi = tanggal_lengkap & (kedaluwarsa <= penyaluran)
mendekati = tanggal_lengkap & (kedaluwarsa > penyaluran) & (kedaluwarsa <= batas_enam_bulan)
lebih_enam_bulan = tanggal_lengkap & (kedaluwarsa > batas_enam_bulan)

e2_rev = pd.Series(pd.NA, index=df_clean.index, dtype="Int64")
e2_rev.loc[mendekati] = 1
e2_rev.loc[lebih_enam_bulan] = 0
df_clean["E2_rev"] = e2_rev

_ed_mentah = df_clean["Tanggal Kedaluwarsa"].notna()
_sal_mentah = df_clean["Tanggal Penyaluran"].notna()
_tdk_terbaca = (_ed_mentah & kedaluwarsa.isna()) | (_sal_mentah & penyaluran.isna())
_tahun_aneh = ((kedaluwarsa.dt.year < TAHUN_WAJAR[0]) | (kedaluwarsa.dt.year > TAHUN_WAJAR[1])
               | (penyaluran.dt.year < TAHUN_WAJAR[0]) | (penyaluran.dt.year > TAHUN_WAJAR[1])).fillna(False)
_masalah_tgl = (perlu_klarifikasi | _tdk_terbaca | _tahun_aneh)
e2k_rev = pd.Series(pd.NA, index=df_clean.index, dtype="Int64")
e2k_rev.loc[tanggal_lengkap & ~_masalah_tgl] = 0
e2k_rev.loc[_masalah_tgl] = 1
if E2K_TERMASUK_KOSONG:
    e2k_rev.loc[kedaluwarsa.isna() & ~_masalah_tgl] = 1
df_clean["E2k_rev"] = e2k_rev
print(f"E2k (tanggal perlu diklarifikasi): {int(bool_np(e2k_rev.eq(1)).sum()):,} baris = 1 "
      f"(kedaluwarsa <= penyaluran: {int(perlu_klarifikasi.sum()):,}; tidak terbaca: {int(_tdk_terbaca.sum()):,}; "
      f"tahun di luar {TAHUN_WAJAR}: {int(_tahun_aneh.sum()):,})")

status_e2 = pd.Series("PERLU DITINJAU — TANGGAL KEDALUWARSA KOSONG/TIDAK TERBACA",
                      index=df_clean.index, dtype="string")
status_e2.loc[perlu_klarifikasi] = "PERLU KLARIFIKASI — KEDALUWARSA <= TANGGAL PENYALURAN"
status_e2.loc[mendekati] = "MENDEKATI KEDALUWARSA <=6 BULAN"
status_e2.loc[lebih_enam_bulan] = "SISA MASA SIMPAN >6 BULAN"
df_clean["Status E2_rev"] = status_e2
print("\nE2 — status")
display(status_e2.value_counts(dropna=False).rename_axis("Status E2").reset_index(name="Jumlah Baris"))

for _obat, _aturan in ATURAN_E3.items():
    if _aturan["mode"] not in ("luar", "dalam"):
        raise ValueError(f"ATURAN_E3[{_obat}]: mode harus 'luar' atau 'dalam'")

kategori_obat = df_clean["Kategori Obat"].astype("string")
label_ada = set(kategori_obat.dropna().unique())
hilang = sorted(set(ATURAN_E3) - label_ada)
if hilang:
    print("PERINGATAN: label", hilang, "tidak ada di Kategori Obat.")

obat_pantauan = bool_np(kategori_obat.isin(sorted(ATURAN_E3)))
sarana_cf = df_clean["Jenis Sarana"].astype("string").str.strip().str.casefold()
dikenal = bool_np(sarana_cf.isin({s.casefold() for s in SARANA_DIKENAL}))

cek_sarana = (df_clean["Jenis Sarana"].astype("string").str.strip().value_counts(dropna=False)
              .rename_axis("Jenis Sarana").reset_index(name="Jumlah Baris"))
cek_sarana["Dikenali kode"] = cek_sarana["Jenis Sarana"].astype("string").str.casefold().isin(
    {s.casefold() for s in SARANA_DIKENAL})
print("\nNilai Jenis Sarana ('False' dianggap NA pada E3; sesuaikan SARANA_DIKENAL bila perlu)")
display(cek_sarana)

sub = df_clean.loc[obat_pantauan]
besar_sub, batas_e3 = tandai_iqr(sub, KELOMPOK_E3, KOLOM_JUMLAH)
jumlah_besar = pd.Series(pd.NA, index=df_clean.index, dtype="Int64")
if len(sub) > 0:
    jumlah_besar.loc[sub.index] = besar_sub
if E3_FALLBACK_KE_E1:
    pakai_e1 = jumlah_besar.isna().to_numpy(dtype=bool) & obat_pantauan & e1_rev.notna().to_numpy(dtype=bool)
    jumlah_besar.loc[pakai_e1] = e1_rev.loc[pakai_e1]
df_clean["Jumlah Besar E3"] = jumlah_besar

def terapkan_aturan_e3(jumlah_besar):
    """Aturan pakar: obat pantauan + jumlah besar + sarana yang diperhatikan -> E3 = 1."""
    ya = bool_np(jumlah_besar.eq(1))
    tidak = bool_np(jumlah_besar.eq(0))
    hasil = pd.Series(pd.NA, index=df_clean.index, dtype="Int64")
    for obat, aturan in ATURAN_E3.items():
        pilih = bool_np(kategori_obat.eq(obat)) & dikenal
        sasaran = bool_np(sarana_cf.isin({s.casefold() for s in aturan["sarana"]}))
        perhatian = ~sasaran if aturan["mode"] == "luar" else sasaran
        hasil.loc[pilih & ~perhatian] = 0
        hasil.loc[pilih & perhatian & tidak] = 0
        hasil.loc[pilih & perhatian & ya] = 1
    return hasil

besar_ya = bool_np(jumlah_besar.eq(1))
besar_tidak = bool_np(jumlah_besar.eq(0))
e3_rev = terapkan_aturan_e3(jumlah_besar)
df_clean["E3_rev"] = e3_rev

def pisah_e3(e3_semua):
    """Pisahkan hasil aturan E3 menjadi E3 (obat di E3_KUAT) dan E3b (obat pantauan lainnya); baris di luar kelompok = NA."""
    kuat = bool_np(kategori_obat.isin(sorted(E3_KUAT)))
    ringan = bool_np(kategori_obat.isin(sorted(set(ATURAN_E3) - set(E3_KUAT))))
    e3k = e3_semua.copy(); e3k.loc[~kuat] = pd.NA
    e3r = e3_semua.copy(); e3r.loc[~ringan] = pd.NA
    return e3k, e3r

baris_e3 = []
for obat, aturan in ATURAN_E3.items():
    m_obat = bool_np(kategori_obat.eq(obat))
    baris_e3.append({
        "Kategori Obat": obat,
        "Diperhatikan bila jumlah besar ke": ("selain " if aturan["mode"] == "luar" else "")
                                              + ", ".join(sorted(aturan["sarana"])),
        "Baris": int(m_obat.sum()),
        "E3 dinilai": int(e3_rev[m_obat].notna().sum()),
        "E3 = 1": int(bool_np(e3_rev[m_obat].eq(1)).sum()),
    })
print("\nE3 — aturan dan hasil per obat pantauan")
display(pd.DataFrame(baris_e3))
print("\nE3 — batas 'jumlah besar' per Kategori Obat x Kategori Satuan")
display(batas_e3)

_e3_kuat, _e3_ringan = pisah_e3(e3_rev)
df_clean["E3 semua pantauan"] = e3_rev
df_clean["E3_rev"] = _e3_kuat
df_clean["E3b_rev"] = _e3_ringan
print(f"E3 kuat ({sorted(E3_KUAT)}): {int(bool_np(_e3_kuat.eq(1)).sum()):,} baris = 1; "
      f"E3b ({sorted(set(ATURAN_E3) - set(E3_KUAT))}): {int(bool_np(_e3_ringan.eq(1)).sum()):,} baris = 1")

Indikator pendukung E4 dan E5.

In [ ]:
if "E4" in df_clean.columns:
    flag_faktur = bool_np(df_clean["E4"].isna())
    print("Penanda faktur: memakai kolom E4 yang sudah ada.")
elif "No. Faktur" in df_clean.columns:
    _faktur = df_clean["No. Faktur"].astype("string").str.strip()
    flag_faktur = bool_np(
        _faktur.isna()
        | ~_faktur.str.contains(r"[A-Za-z0-9]", regex=True, na=False)
        | _faktur.str.casefold().isin(PLACEHOLDER_FAKTUR)
    )
    print("Penanda faktur: dihitung dari No. Faktur.")
else:
    flag_faktur = np.zeros(len(df_clean), dtype=bool)
    print("PERINGATAN: kolom E4 dan No. Faktur tidak ada; penanda faktur dikosongkan.")

if "E5 lokasi bermasalah (final)" in df_clean.columns:
    flag_lokasi = df_clean["E5 lokasi bermasalah (final)"].fillna(0).astype(int).eq(1).to_numpy(dtype=bool)
    print("Penanda lokasi: memakai hasil e5_lokasi (E5a kosong atau E5b kota-provinsi tidak cocok).")
elif "E5" in df_clean.columns:
    flag_lokasi = bool_np(df_clean["E5"].isna() | df_clean["E5"].eq(1))
    print("Penanda lokasi: memakai kolom E5 yang sudah ada.")
else:
    _ada = [k for k in KOLOM_LOKASI if k in df_clean.columns]
    flag_lokasi = np.zeros(len(df_clean), dtype=bool)
    for k in _ada:
        flag_lokasi |= bool_np(df_clean[k].isna() | df_clean[k].astype("string").str.strip().eq(""))
    print(f"Penanda lokasi: dihitung dari kelengkapan {_ada}.")

flag_tgl = kedaluwarsa.isna().to_numpy(dtype=bool)
flag_ed = perlu_klarifikasi.to_numpy(dtype=bool)

flag_ket_rs = bool_np(kategori_obat.eq("Ketamine")) & bool_np(sarana_cf.eq("rumah sakit")) & besar_ya
flag_pantau_rs = bool_np(kategori_obat.isin(["Cefixime", "Orlistat"])) & bool_np(sarana_cf.eq("rumah sakit")) & besar_ya

ALASAN_E3 = {"Ketamine": "Ketamin jumlah besar ke selain RS (risiko penyalahgunaan)",
             "Meropenem": "Meropenem jumlah besar ke selain RS (antibiotik reserve)",
             "Cefixime": "Cefixime jumlah besar ke klinik/apotek (antibiotik watch)",
             "Orlistat": "Orlistat jumlah besar ke klinik/apotek (potensi penyalahgunaan pelangsingan)",
             "TBC": "Obat TB jumlah besar ke apotek/klinik (obat program)"}
df_clean["Alasan E3"] = np.where(bool_np(e3_rev.eq(1)),
                                 kategori_obat.map(ALASAN_E3).fillna("Obat pantauan jumlah besar").astype(object), "-")

teks = np.full(len(df_clean), "", dtype=object)
for nama, flag in [("Ketamin jumlah besar ke RS (perlu pengkajian RS)", flag_ket_rs),
                   ("Cefixime/Orlistat jumlah besar ke RS (perlu diperhatikan)", flag_pantau_rs),
                   ("Faktur bermasalah", flag_faktur),
                   ("Lokasi bermasalah (kosong / kota-provinsi tidak cocok)", flag_lokasi),
                   ("Kedaluwarsa <= penyaluran (klarifikasi)", flag_ed),
                   ("Tanggal kedaluwarsa kosong/tidak valid", flag_tgl),
                   ("TEMUAN DATA ANOMALI: jumlah tidak wajar, dikeluarkan dari penilaian risiko (verifikasi di e-WAS)",
                    df_clean["Temuan jumlah janggal"].fillna(False).to_numpy(dtype=bool) if "Temuan jumlah janggal" in df_clean.columns else np.zeros(len(df_clean), dtype=bool))]:
    teks = np.where(flag, teks + nama + "; ", teks)
catatan = pd.Series(teks, index=df_clean.index).str.rstrip("; ")
df_clean["Catatan Pendukung"] = catatan.where(catatan != "", "-")
print(f"Transaksi dengan catatan pendukung: {int((df_clean['Catatan Pendukung'] != '-').sum()):,}")

Matriks bukti X (1 = ada, 0 = tidak, kosong dilewati) dan frekuensi teramati.

In [ ]:
def ke_biner(serie, nama):
    angka = pd.to_numeric(serie, errors="coerce")
    salah = serie.notna() & ~angka.isin([0, 1])
    if salah.any():
        raise ValueError(f"{nama}: {int(salah.sum()):,} nilai bukan 0/1/NA")
    return angka.astype("Int64")

X = pd.DataFrame({
    "E1": ke_biner(df_clean["E1_rev"], "E1_rev"),
    "E2": ke_biner(df_clean["E2_rev"], "E2_rev"),
    "E2k": ke_biner(df_clean["E2k_rev"], "E2k_rev"),
    "E3": ke_biner(df_clean["E3_rev"], "E3_rev"),
    "E3b": ke_biner(df_clean["E3b_rev"], "E3b_rev"),
}, index=df_clean.index)

ANOMALI = (bool_np(df_clean["Temuan jumlah janggal"]) if "Temuan jumlah janggal" in df_clean.columns
           else np.zeros(len(df_clean), dtype=bool))

def terapkan_anomali(Xn):
    hasil = Xn.copy()
    if ANOMALI.any():
        for _k in hasil.columns:
            hasil.loc[ANOMALI, _k] = pd.NA
    return hasil

X = terapkan_anomali(X)
df_clean["Temuan data anomali"] = ANOMALI
print(f"Temuan data anomali: {int(ANOMALI.sum()):,} transaksi dikeluarkan dari penilaian risiko (semua indikator = NA).")

RS_BIASA = bool_np(sarana_cf.eq("rumah sakit")) & ~bool_np(kategori_obat.isin(sorted(OBAT_RS_DIKAJI)))

def terapkan_e1_pbf(e1):
    """Wawancara pakar: jumlah besar ke PBF = wajar; obat biasa besar ke RS = aman (v3). E1 = 1 pada kasus itu menjadi 0."""
    hasil = e1.copy()
    if E1_PBF_RENDAH:
        _pbf = bool_np(sarana_cf.eq("pbf"))
        hasil.loc[_pbf & bool_np(hasil.eq(1))] = 0
    if E1_RS_BIASA_AMAN:
        hasil.loc[RS_BIASA & bool_np(hasil.eq(1))] = 0
    return hasil

def terapkan_e2_rs(e2):
    """Wawancara lanjutan (v3): kedaluwarsa <= 6 bulan ke RS dengan jumlah biasa (obat biasa) = Rendah. E2 yang terisi diset 0."""
    hasil = e2.copy()
    if E2_RS_BIASA_AMAN:
        hasil.loc[RS_BIASA & hasil.notna().to_numpy(dtype=bool)] = 0
    return hasil

AP_KL = bool_np(sarana_cf.isin(sorted(SARANA_E2_KUAT)))

def pisah_e2(e2):
    """Pisahkan E2 menurut sarana: E2 (tujuan selain apotek/klinik) dan E2p (apotek/klinik); di luar kelompoknya = NA."""
    kosong = pd.Series(pd.NA, index=e2.index, dtype="Int64")
    if not E2_SPLIT_SARANA:
        return e2, kosong
    biasa, kuat = e2.copy(), e2.copy()
    biasa.loc[AP_KL] = pd.NA
    kuat.loc[~AP_KL] = pd.NA
    return biasa, kuat

_e1_sebelum = int(bool_np(X["E1"].eq(1)).sum())
X["E1"] = terapkan_e1_pbf(X["E1"])
print(f"Aturan E1 (PBF dan obat biasa ke RS): {_e1_sebelum - int(bool_np(X['E1'].eq(1)).sum()):,} baris E1 = 1 diubah menjadi 0.")
_e2_sebelum = int(bool_np(X["E2"].eq(1)).sum())
X["E2"] = terapkan_e2_rs(X["E2"])
print(f"Aturan E2 (obat biasa ke RS): {_e2_sebelum - int(bool_np(X['E2'].eq(1)).sum()):,} baris E2 = 1 diubah menjadi 0.")
X["E2"], X["E2p"] = pisah_e2(X["E2"])
print(f"E2 dipecah menurut sarana: E2 = 1 pada {int(bool_np(X['E2'].eq(1)).sum()):,} baris (selain apotek/klinik), "
      f"E2p = 1 pada {int(bool_np(X['E2p'].eq(1)).sum()):,} baris (apotek/klinik).")
if E2K_ISOLASI:
    X["E2k"] = pd.Series(pd.NA, index=X.index, dtype="Int64")
    print("E2k diisolasi (netral): tanggal kosong/janggal tidak memengaruhi posterior; tetap muncul di Catatan Pendukung.")

if E1_NA_BILA_E3:
    _e3_aktif = bool_np(X["E3"].eq(1)) | bool_np(X["E3b"].eq(1))
    _ganda = int((_e3_aktif & bool_np(X["E1"].eq(1))).sum())
    X.loc[_e3_aktif, "E1"] = pd.NA
    print(f"E1 diabaikan (NA) pada {int(_e3_aktif.sum()):,} baris yang E3 = 1; "
          f"{_ganda:,} di antaranya juga E1 = 1 (hitung ganda yang dihindari).")
df_clean["E1_model"] = X["E1"]

baris_frek, FREK = [], {}
for k in X.columns:
    n_dinilai = int(X[k].notna().sum())
    n_satu = int(bool_np(X[k].eq(1)).sum())
    m = n_satu / n_dinilai if n_dinilai else np.nan
    FREK[k] = m
    baris_frek.append({"Indikator": k, "Baris dinilai": n_dinilai, "Bernilai 1": n_satu,
                       "Frekuensi m": m, "Baris NA": len(X) - n_dinilai})
tabel_frek = pd.DataFrame(baris_frek)

LR_USULAN = dict(LR_UTAMA)
_baris_lr = []
for _k in X.columns:
    _m = FREK[_k]
    _maks = np.inf if (not _m or _m <= PRIOR_UTAMA) else (1 - PRIOR_UTAMA) / (_m - PRIOR_UTAMA)
    _dipakai = LR_UTAMA[_k]
    if _dipakai >= _maks:
        _dipakai = round(0.95 * _maks, 2)
        LR_UTAMA[_k] = _dipakai
    _baris_lr.append({"Indikator": _k, "m teramati": _m, "LR usulan": LR_USULAN[_k],
                      "LR maksimum layak (prior utama)": _maks, "LR dipakai": _dipakai,
                      "Dipotong": _dipakai != LR_USULAN[_k]})
tabel_lr_layak = pd.DataFrame(_baris_lr)
print("Kelayakan LR terhadap frekuensi teramati (prior utama)")
display(tabel_lr_layak.round(4))
if tabel_lr_layak["Dipotong"].any():
    print("PERINGATAN: LR usulan tidak layak untuk indikator yang ditandai 'Dipotong' karena frekuensinya terlalu "
          "tinggi dibanding prior (p1 akan melebihi 1). LR diturunkan otomatis. Artinya penilaian pakar "
          "(indikator itu saja sudah Tinggi) tidak konsisten dengan frekuensi di data pada prior ini; "
          "bahas dengan pakar atau naikkan prior.")

INDIKATOR = [k for k in X.columns if FREK[k] and FREK[k] > 0]
dikeluarkan = [k for k in X.columns if k not in INDIKATOR and not (k == "E2k" and E2K_ISOLASI)]
print("Frekuensi teramati per indikator")
display(tabel_frek.round({"Frekuensi m": 5}))
if dikeluarkan:
    print(f"PERINGATAN: {dikeluarkan} tidak pernah bernilai 1 / tidak ada data; "
          f"dikeluarkan dari model.")
if not INDIKATOR:
    raise ValueError("Tidak ada indikator yang dapat dipakai.")
print("Indikator dalam model:", INDIKATOR)

Kalibrasi parameter: p0 = m / (π·LR + 1 − π) dan p1 = LR · p0.

In [ ]:
def kalibrasi(m, prior, lr):
    """Turunkan (p1, p0) dari frekuensi teramati m, prior, dan rasio likelihood lr."""
    if lr <= 1:
        raise ValueError(f"LR harus > 1 (diberikan {lr})")
    if not 0 < prior < 1:
        raise ValueError(f"Prior harus di antara 0 dan 1 (diberikan {prior})")
    p0 = m / (prior * lr + 1 - prior)
    p1 = lr * p0
    if not (0 < p0 < 1 and 0 < p1 < 1):
        raise ValueError(f"Parameter tidak valid (p1={p1:.4f}, p0={p0:.4f}); "
                         f"turunkan LR atau naikkan prior")
    return p1, p0

def bangun_parameter(prior, lr):
    return {k: kalibrasi(FREK[k], prior, lr[k]) for k in INDIKATOR}

baris = []
for prior in PRIOR_TABEL:
    for k in INDIKATOR:
        try:
            p1, p0 = kalibrasi(FREK[k], prior, LR_UTAMA[k])
            baris.append({"Prior": prior, "Indikator": k, "m teramati": FREK[k],
                          "LR": LR_UTAMA[k], "p1 = P(E=1|R)": p1, "p0 = P(E=1|~R)": p0,
                          "p1 maksimum (m/prior)": FREK[k] / prior,
                          "Cek m = prior*p1+(1-prior)*p0": prior * p1 + (1 - prior) * p0})
        except ValueError as galat:
            baris.append({"Prior": prior, "Indikator": k, "m teramati": FREK[k],
                          "LR": LR_UTAMA[k], "p1 = P(E=1|R)": np.nan, "p0 = P(E=1|~R)": np.nan,
                          "p1 maksimum (m/prior)": FREK[k] / prior,
                          "Cek m = prior*p1+(1-prior)*p0": np.nan})
tabel_kalibrasi = pd.DataFrame(baris)
print("Parameter terkalibrasi per prior (LR tetap = LR_UTAMA)")
display(tabel_kalibrasi.round(5))

print("\nParameter referensi (prior utama)")
PARAM_REF = bangun_parameter(PRIOR_UTAMA, LR_UTAMA)
tabel_ref = pd.DataFrame(
    [{"Indikator": k, "Prior": PRIOR_UTAMA, "LR": LR_UTAMA[k], "p1": PARAM_REF[k][0],
      "p0": PARAM_REF[k][1], "P(E=0|R)": 1 - PARAM_REF[k][0], "P(E=0|~R)": 1 - PARAM_REF[k][1]}
     for k in INDIKATOR]
)
display(tabel_ref.round(5))

Pola indikator unik dan harapan pakar per pola.

In [ ]:
KODE = X[INDIKATOR].fillna(-1).to_numpy(dtype=int)
POLA_UNIK, INVERS, JUMLAH_POLA = np.unique(KODE, axis=0, return_inverse=True, return_counts=True)
INVERS = np.asarray(INVERS).reshape(-1)
TERSEDIA_POLA = (POLA_UNIK >= 0).sum(axis=1)
N_AKTIF_POLA = (POLA_UNIK == 1).sum(axis=1)

def posterior_pola(prior, parameter):
    s1 = np.full(len(POLA_UNIK), float(prior))
    s0 = np.full(len(POLA_UNIK), 1.0 - float(prior))
    for j, k in enumerate(INDIKATOR):
        p1, p0 = parameter[k]
        v = POLA_UNIK[:, j]
        s1 = np.where(v == 1, s1 * p1, np.where(v == 0, s1 * (1 - p1), s1))
        s0 = np.where(v == 1, s0 * p0, np.where(v == 0, s0 * (1 - p0), s0))
    return np.where(TERSEDIA_POLA > 0, s1 / (s1 + s0), np.nan)

def kategori_pola(post, sedang, tinggi):
    ada = np.isfinite(post)
    ke_sedang = ada & ((post >= sedang) | np.isclose(post, sedang, rtol=0, atol=TOLERANSI))
    ke_tinggi = ada & ((post >= tinggi) | np.isclose(post, tinggi, rtol=0, atol=TOLERANSI))
    hasil = np.full(len(post), "BELUM DAPAT DINILAI", dtype=object)
    hasil[ada] = "RENDAH"
    hasil[ke_sedang] = "SEDANG"
    hasil[ke_tinggi] = "TINGGI"
    return hasil

def kategori_baseline_pola():
    """Threshold sederhana: 0 indikator aktif = Rendah, 1 = Sedang, >=2 = Tinggi."""
    hasil = np.select([N_AKTIF_POLA == 0, N_AKTIF_POLA == 1], ["RENDAH", "SEDANG"], "TINGGI")
    hasil = hasil.astype(object)
    hasil[TERSEDIA_POLA == 0] = "BELUM DAPAT DINILAI"
    return hasil

def hitung_kategori(kat):
    return {lab: int(JUMLAH_POLA[kat == lab].sum()) for lab in LABEL}

def teks_pola(baris):
    return " / ".join("NA" if v < 0 else str(int(v)) for v in baris)

POST_REF = posterior_pola(PRIOR_UTAMA, PARAM_REF)
KAT_REF = kategori_pola(POST_REF, AMBANG_SEDANG, AMBANG_TINGGI)
KAT_BASE = kategori_baseline_pola()

df_clean["Posterior Bayes"] = POST_REF[INVERS]
df_clean["Kategori Bayes"] = pd.Series(KAT_REF[INVERS], index=df_clean.index, dtype="string")
df_clean["Kategori Baseline"] = pd.Series(KAT_BASE[INVERS], index=df_clean.index, dtype="string")
df_clean["Jumlah Indikator Tersedia"] = TERSEDIA_POLA[INVERS]

tabel_pola = pd.DataFrame({
    "Pola (" + " / ".join(INDIKATOR) + ")": [teks_pola(b) for b in POLA_UNIK],
    "Jumlah Baris": JUMLAH_POLA,
    "Posterior": POST_REF,
    "Kategori Bayes": KAT_REF,
    "Kategori Baseline": KAT_BASE,
}).sort_values("Jumlah Baris", ascending=False).reset_index(drop=True)
tabel_pola["Sama"] = tabel_pola["Kategori Bayes"] == tabel_pola["Kategori Baseline"]

print(f"Referensi: prior={PRIOR_UTAMA}, LR={LR_UTAMA}, ambang={AMBANG_SEDANG}/{AMBANG_TINGGI}")
print("\nPosterior dan kategori per pola indikator (1 = aktif, 0 = tidak, NA = tidak tersedia)")
display(tabel_pola.round({"Posterior": 4}))

dist = pd.DataFrame({"Bayesian": hitung_kategori(KAT_REF), "Baseline": hitung_kategori(KAT_BASE)})
dist["% Bayesian"] = (dist["Bayesian"] / len(df_clean) * 100).round(3)
dist["% Baseline"] = (dist["Baseline"] / len(df_clean) * 100).round(3)
print("\nDistribusi kategori")
display(dist)

Skenario referensi dan perhitungan posterior.

In [ ]:
HARAPAN_PAKAR = {("E1",): "SEDANG", ("E3",): "TINGGI", ("E3b",): "SEDANG",
                 ("E2",): "TINGGI" if TOGGLE_EXPIRY_RULE == "WAWANCARA" else "SEDANG",
                 ("E1", "E2"): "TINGGI", ("E2", "E3"): "TINGGI",
                 ("E2p",): "TINGGI", ("E1", "E2p"): "TINGGI", ("E2p", "E3"): "TINGGI"}
if not E2K_ISOLASI:
    HARAPAN_PAKAR[("E2k",)] = "SEDANG"
_baris = []
for _i, _pola in enumerate(POLA_UNIK):
    _aktif = tuple(k for _j, k in enumerate(INDIKATOR) if _pola[_j] == 1)
    if _aktif in HARAPAN_PAKAR:
        _baris.append({"Indikator aktif": " + ".join(_aktif), "Pola (" + " / ".join(INDIKATOR) + ")": teks_pola(_pola),
                       "Jumlah Baris": int(JUMLAH_POLA[_i]), "Posterior": POST_REF[_i],
                       "Kategori model": KAT_REF[_i], "Kategori pakar": HARAPAN_PAKAR[_aktif],
                       "Sesuai": KAT_REF[_i] == HARAPAN_PAKAR[_aktif]})
tabel_cek_pakar = pd.DataFrame(_baris).sort_values(["Indikator aktif", "Jumlah Baris"], ascending=[True, False])
display(tabel_cek_pakar.round({"Posterior": 4}))
if len(tabel_cek_pakar):
    _tot = int(tabel_cek_pakar["Jumlah Baris"].sum())
    _ok = int(tabel_cek_pakar.loc[tabel_cek_pakar["Sesuai"], "Jumlah Baris"].sum())
    print(f"Baris pada pola yang dinilai pakar: {_tot:,}; sesuai penilaian pakar: {_ok:,} ({_ok / _tot * 100:.1f}%)")
_target = (hitung_kategori(KAT_REF)["SEDANG"] + hitung_kategori(KAT_REF)["TINGGI"]) / len(df_clean) * 100
print(f"Sedang + Tinggi = {_target:.1f}% dari seluruh transaksi (pakar: sekitar 15% layak diprioritaskan)")

Pemilihan skema A (ambang posterior 0,30 dan 0,70, skema utama) atau skema B.

In [ ]:
SKEMA_VALID = ("A", "B")
if SKEMA_UTAMA not in SKEMA_VALID:
    raise ValueError(f"SKEMA_UTAMA harus salah satu dari {SKEMA_VALID}")

def odds(p):
    return p / (1.0 - p)

def ambang_efektif(prior, sedang_ref, tinggi_ref, skema):
    """sedang_ref dan tinggi_ref = ambang posterior pada PRIOR_UTAMA.
    Skema B: dipakai apa adanya. Skema A: dikonversi ke faktor odds terhadap prior,
    lalu diterapkan pada prior skenario."""
    if skema not in SKEMA_VALID:
        raise ValueError(f"Skema harus salah satu dari {SKEMA_VALID}")
    if skema == "B":
        return sedang_ref, tinggi_ref
    hasil = []
    for ref in (sedang_ref, tinggi_ref):
        o = (odds(ref) / odds(PRIOR_UTAMA)) * odds(prior)
        hasil.append(o / (1.0 + o))
    return hasil[0], hasil[1]

def label_posterior(p, sedang, tinggi):
    if p >= tinggi or np.isclose(p, tinggi, rtol=0, atol=TOLERANSI):
        return "TINGGI"
    if p >= sedang or np.isclose(p, sedang, rtol=0, atol=TOLERANSI):
        return "SEDANG"
    return "RENDAH"

_kandidat = np.where((N_AKTIF_POLA == 0) & (TERSEDIA_POLA > 0))[0]
if len(_kandidat) == 0:
    raise ValueError("Tidak ada pola tanpa bukti risiko di data.")
IDX_TANPA_BUKTI = int(_kandidat[np.argmax(JUMLAH_POLA[_kandidat])])
TEKS_TANPA_BUKTI = teks_pola(POLA_UNIK[IDX_TANPA_BUKTI])
print(f"Pola tanpa bukti paling umum: {TEKS_TANPA_BUKTI} "
      f"({int(JUMLAH_POLA[IDX_TANPA_BUKTI]):,} baris, indikator: {' / '.join(INDIKATOR)})")

def posterior_tanpa_bukti(prior):
    parameter = bangun_parameter(prior, LR_UTAMA)
    return float(posterior_pola(prior, parameter)[IDX_TANPA_BUKTI])

print(f"Faktor odds ambang referensi: Sedang = {odds(AMBANG_SEDANG) / odds(PRIOR_UTAMA):.3f}x, "
      f"Tinggi = {odds(AMBANG_TINGGI) / odds(PRIOR_UTAMA):.3f}x terhadap prior {PRIOR_UTAMA}")

baris = []
for prior in PRIOR_TABEL:
    sa, ta = ambang_efektif(prior, AMBANG_SEDANG, AMBANG_TINGGI, "A")
    sb, tb = ambang_efektif(prior, AMBANG_SEDANG, AMBANG_TINGGI, "B")
    try:
        p_nol = posterior_tanpa_bukti(prior)
        kat_a = label_posterior(p_nol, sa, ta)
        kat_b = label_posterior(p_nol, sb, tb)
    except ValueError:
        p_nol, kat_a, kat_b = np.nan, "-", "-"
    baris.append({"Prior": prior, "Ambang Sedang A": sa, "Ambang Tinggi A": ta,
                  "Ambang Sedang B": sb, "Ambang Tinggi B": tb,
                  "Posterior transaksi tanpa bukti (pola paling umum)": p_nol,
                  "Kategori (A)": kat_a, "Kategori (B)": kat_b})
tabel_ambang = pd.DataFrame(baris)
print("\nAmbang efektif per prior, dan kategori transaksi tanpa bukti risiko (pola paling umum)")
display(tabel_ambang.round(4))

Perbandingan pola yang memiliki cukup indikator tersedia.

In [ ]:
mask_banding = TERSEDIA_POLA >= MIN_INDIKATOR_BANDING
n_banding = int(JUMLAH_POLA[mask_banding].sum())
sama_pola = KAT_REF == KAT_BASE
n_sama = int(JUMLAH_POLA[mask_banding & sama_pola].sum())
persen_sama_ref = n_sama / n_banding * 100 if n_banding else np.nan

silang = (pd.DataFrame({"Baseline": KAT_BASE[mask_banding], "Bayesian": KAT_REF[mask_banding],
                        "n": JUMLAH_POLA[mask_banding]})
          .pivot_table(index="Baseline", columns="Bayesian", values="n", aggfunc="sum", fill_value=0)
          .reindex(index=LABEL, columns=LABEL, fill_value=0))
silang = silang.loc[(silang.sum(axis=1) > 0), (silang.sum(axis=0) > 0)]

print(f"Dibandingkan: {n_banding:,} transaksi (>= {MIN_INDIKATOR_BANDING} indikator tersedia)")
print(f"Kategori sama: {persen_sama_ref:.4f}% | berbeda: {n_banding - n_sama:,} transaksi")
display(silang)

beda = tabel_pola[(~tabel_pola["Sama"]) & (tabel_pola["Pola (" + " / ".join(INDIKATOR) + ")"].map(
    lambda t: any(v != "NA" for v in t.split(" / "))))]
if len(beda):
    print("\nPola yang kategorinya berbeda antara Bayesian dan baseline")
    display(beda.round({"Posterior": 4}))
else:
    print("\nTidak ada pola yang berbeda: pada konfigurasi referensi, Bayesian = baseline.")

Grafik sebaran kategori risiko.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
warna = {"RENDAH": "#8fbf8f", "SEDANG": "#e6c35c", "TINGGI": "#d9534f", "BELUM DAPAT DINILAI": "#bbbbbb"}
for sumbu, kolom, judul in [(ax[0], "Bayesian", "Bayesian (referensi)"), (ax[1], "Baseline", "Threshold (baseline)")]:
    nilai = [dist.loc[lab, kolom] for lab in LABEL]
    sumbu.bar(range(len(LABEL)), nilai, color=[warna[l] for l in LABEL])
    sumbu.set_xticks(range(len(LABEL)))
    sumbu.set_xticklabels(["Rendah", "Sedang", "Tinggi", "Belum"], fontsize=9)
    sumbu.set_title(judul, fontsize=11)
    sumbu.set_ylabel("Jumlah transaksi")
    for i, v in enumerate(nilai):
        sumbu.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

Tabel pola indikator dan kategori yang dihasilkan.

In [ ]:
kolom_pola = "Pola (" + " / ".join(INDIKATOR) + ")"
valid_pola = TERSEDIA_POLA > 0
level_bayes = len(np.unique(np.round(POST_REF[valid_pola], 10)))
level_baseline = len(np.unique(N_AKTIF_POLA[valid_pola]))
print(f"Tingkat skor yang dibedakan: Bayesian = {level_bayes} tingkat, "
      f"threshold (jumlah indikator aktif) = {level_baseline} tingkat")

dalam_kategori = tabel_pola[tabel_pola["Kategori Bayes"].isin(["SEDANG", "TINGGI"])].copy()
dalam_kategori["Indikator aktif"] = dalam_kategori[kolom_pola].map(
    lambda t: t.split(" / ").count("1"))
dalam_kategori = dalam_kategori.sort_values("Posterior", ascending=False).reset_index(drop=True)
print("\nPola pada kategori Sedang dan Tinggi, diurutkan menurut skor Bayesian")
display(dalam_kategori[[kolom_pola, "Indikator aktif", "Jumlah Baris", "Posterior",
                        "Kategori Bayes", "Kategori Baseline"]].round({"Posterior": 4}))

### **Analisis OAT**
Pembentukan skenario satu-parameter-per-kali.

In [ ]:
def buat_skenario():
    daftar = [dict(kode="S0", nama="Referensi", parameter="-", nilai="-", jenis="Referensi",
                   prior=PRIOR_UTAMA, lr=dict(LR_UTAMA), sedang=AMBANG_SEDANG, tinggi=AMBANG_TINGGI)]
    nomor = 0
    for parameter, daftar_nilai in OAT.items():
        for nilai in daftar_nilai:
            sk = dict(prior=PRIOR_UTAMA, lr=dict(LR_UTAMA), sedang=AMBANG_SEDANG, tinggi=AMBANG_TINGGI)
            if parameter == "Prior":
                sk["prior"] = nilai
            elif parameter.startswith("LR "):
                indikator = parameter.split()[1]
                if indikator not in INDIKATOR:
                    continue
                sk["lr"][indikator] = nilai
            elif parameter == "Ambang Sedang":
                sk["sedang"] = nilai
            elif parameter == "Ambang Tinggi":
                sk["tinggi"] = nilai
            else:
                raise ValueError(f"Parameter OAT tidak dikenal: {parameter}")
            nomor += 1
            daftar.append(dict(kode=f"S{nomor}", nama=f"{parameter} = {nilai}", parameter=parameter,
                               nilai=nilai, jenis="OAT", **sk))
    for nama, prior, pengali in GABUNGAN:
        nomor += 1
        lr = {k: LR_UTAMA[k] * pengali for k in INDIKATOR}
        daftar.append(dict(kode=f"S{nomor}", nama=f"{nama} (prior {prior}; semua LR x{pengali})",
                           parameter="Gabungan", nilai="-", jenis="Gabungan (bukan OAT)",
                           prior=prior, lr=lr, sedang=AMBANG_SEDANG, tinggi=AMBANG_TINGGI))
    return daftar

SKEMA_URUT = [SKEMA_UTAMA] + [s for s in SKEMA_VALID if s != SKEMA_UTAMA]
hasil_oat, perubahan_pola, parameter_skenario = [], [], []
KAT_SKENARIO = {}

for skema in SKEMA_URUT:
    for sk in buat_skenario():
        if not (0 < sk["sedang"] < sk["tinggi"] < 1):
            hasil_oat.append(dict(Skema=skema, Kode=sk["kode"], Skenario=sk["nama"], Jenis=sk["jenis"],
                                  Keterangan="DILEWATI: ambang tidak valid"))
            continue
        try:
            parameter = bangun_parameter(sk["prior"], sk["lr"])
        except ValueError as galat:
            hasil_oat.append(dict(Skema=skema, Kode=sk["kode"], Skenario=sk["nama"], Jenis=sk["jenis"],
                                  Keterangan=f"DILEWATI: {galat}"))
            continue

        sedang_eff, tinggi_eff = ambang_efektif(sk["prior"], sk["sedang"], sk["tinggi"], skema)
        post = posterior_pola(sk["prior"], parameter)
        kat = kategori_pola(post, sedang_eff, tinggi_eff)
        KAT_SKENARIO[(sk["kode"], skema)] = kat
        hitung = hitung_kategori(kat)
        berubah = kat != KAT_REF
        n_berubah = int(JUMLAH_POLA[berubah].sum())
        sama_base = int(JUMLAH_POLA[mask_banding & (kat == KAT_BASE)].sum())

        hasil_oat.append({
            "Skema": skema, "Kode": sk["kode"], "Skenario": sk["nama"], "Jenis": sk["jenis"],
            "Parameter diubah": sk["parameter"],
            "Ambang Sedang efektif": sedang_eff, "Ambang Tinggi efektif": tinggi_eff,
            "Rendah": hitung["RENDAH"], "Sedang": hitung["SEDANG"], "Tinggi": hitung["TINGGI"],
            "Belum dinilai": hitung["BELUM DAPAT DINILAI"],
            "% Rendah": hitung["RENDAH"] / len(df_clean) * 100,
            "% Sedang": hitung["SEDANG"] / len(df_clean) * 100,
            "% Tinggi": hitung["TINGGI"] / len(df_clean) * 100,
            "Baris berubah vs S0": n_berubah,
            "% berubah vs S0": n_berubah / len(df_clean) * 100,
            "% sama dg baseline": sama_base / n_banding * 100 if n_banding else np.nan,
            "Keterangan": "",
        })
        if skema == "A":
            for k, (p1, p0) in parameter.items():
                parameter_skenario.append({"Kode": sk["kode"], "Indikator": k, "Prior": sk["prior"],
                                           "LR": sk["lr"][k], "p1": p1, "p0": p0})
        for i in np.where(berubah)[0]:
            perubahan_pola.append({"Skema": skema, "Kode": sk["kode"], "Skenario": sk["nama"],
                                   "Pola": teks_pola(POLA_UNIK[i]), "Jumlah Baris": int(JUMLAH_POLA[i]),
                                   "Posterior S0": POST_REF[i], "Posterior skenario": post[i],
                                   "Kategori S0": KAT_REF[i], "Kategori skenario": kat[i]})

tabel_oat_semua = pd.DataFrame(hasil_oat)
tabel_oat = {s: tabel_oat_semua[tabel_oat_semua["Skema"] == s].reset_index(drop=True) for s in SKEMA_URUT}
tabel_perubahan = pd.DataFrame(perubahan_pola)
tabel_param_skenario = pd.DataFrame(parameter_skenario)

kolom_tampil = ["Kode", "Skenario", "Jenis", "Ambang Sedang efektif", "Ambang Tinggi efektif",
                "Rendah", "Sedang", "Tinggi", "Baris berubah vs S0", "% berubah vs S0",
                "% sama dg baseline", "Keterangan"]
t_utama = tabel_oat[SKEMA_UTAMA]
print(f"Hasil OAT skema utama ({SKEMA_UTAMA})")
display(t_utama[[c for c in kolom_tampil if c in t_utama.columns]].round(
    {"Ambang Sedang efektif": 4, "Ambang Tinggi efektif": 4,
     "% berubah vs S0": 3, "% sama dg baseline": 3}))

Perbandingan skema A dan B.

In [ ]:
kolom_bandingkan = ["Ambang Sedang efektif", "Ambang Tinggi efektif", "Rendah", "Sedang", "Tinggi",
                    "Baris berubah vs S0", "% sama dg baseline"]
kunci = ["Kode", "Skenario", "Jenis"]
a = tabel_oat["A"].reindex(columns=kunci + kolom_bandingkan)
b = tabel_oat["B"].reindex(columns=kunci + kolom_bandingkan)
perbandingan_skema = a.merge(b, on=kunci, suffixes=(" (A)", " (B)"))

beda_ab = {}
for kode_sk in perbandingan_skema["Kode"]:
    ka, kb = KAT_SKENARIO.get((kode_sk, "A")), KAT_SKENARIO.get((kode_sk, "B"))
    beda_ab[kode_sk] = int(JUMLAH_POLA[ka != kb].sum()) if ka is not None and kb is not None else np.nan
perbandingan_skema["Baris beda A vs B"] = perbandingan_skema["Kode"].map(beda_ab)

print("Perbandingan skema A (ambang relatif) dan B (ambang tetap) per skenario")
display(perbandingan_skema.round(4))

n_sama = int((perbandingan_skema["Baris beda A vs B"] == 0).sum())
print(f"\nSkenario yang hasilnya identik di kedua skema: {n_sama} dari "
      f"{int(perbandingan_skema['Baris beda A vs B'].notna().sum())}")
print("Perbedaan skema hanya dapat muncul pada skenario yang mengubah prior (dan skenario G).")

Sensitivitas per parameter.

In [ ]:
def peringkat_skema(skema):
    t = tabel_oat[skema]
    t = t[(t["Jenis"] == "OAT") & t["% berubah vs S0"].notna()]
    if t.empty:
        return pd.DataFrame()
    return (t.groupby("Parameter diubah")
            .agg(Skenario_valid=("Kode", "count"),
                 Maks_baris_berubah=("Baris berubah vs S0", "max"),
                 Maks_persen_berubah=("% berubah vs S0", "max")))

pa, pb = peringkat_skema("A"), peringkat_skema("B")
if pa.empty and pb.empty:
    peringkat = pd.DataFrame()
    print("Tidak ada skenario OAT yang valid.")
else:
    peringkat = pa.join(pb, how="outer", lsuffix=" (A)", rsuffix=" (B)")
    peringkat = peringkat.sort_values(f"Maks_baris_berubah ({SKEMA_UTAMA})", ascending=False)
    print("Peringkat sensitivitas parameter (baris yang berubah kategori, tertinggi per parameter)")
    display(peringkat.round(3))

print("\nPola yang berpindah kategori pada tiap skenario")
if len(tabel_perubahan):
    display(tabel_perubahan.sort_values(["Skema", "Kode", "Jumlah Baris"], ascending=[True, True, False])
            .round({"Posterior S0": 4, "Posterior skenario": 4}))
else:
    print("Tidak ada pola yang berpindah kategori pada skenario manapun.")

Fungsi pembantu uji menyeluruh.

In [ ]:
MODE_REKAL, MODE_TETAP = "rekalibrasi", "tetap"
MODE_URUT = (MODE_REKAL, MODE_TETAP)

def parameter_mode(prior, lr, mode):
    if mode == MODE_REKAL:
        return bangun_parameter(prior, lr)
    hasil = {}
    for k in INDIKATOR:
        p0 = PARAM_REF[k][1]
        p1 = lr[k] * p0
        if not (0 < p1 < 1):
            raise ValueError(f"p1 tidak valid pada {k} (p1={p1:.4f})")
        hasil[k] = (p1, p0)
    return hasil

def jalankan(prior, lr, sedang, tinggi, skema, mode):
    """Satu konfigurasi -> (ringkasan, kategori per pola)."""
    param = parameter_mode(prior, lr, mode)
    se, ti = ambang_efektif(prior, sedang, tinggi, skema)
    post = posterior_pola(prior, param)
    kat = kategori_pola(post, se, ti)
    h = hitung_kategori(kat)
    n = len(df_clean)
    berubah = int(JUMLAH_POLA[kat != KAT_REF].sum())
    sama_base = int(JUMLAH_POLA[mask_banding & (kat == KAT_BASE)].sum())
    return {"Ambang Sedang efektif": se, "Ambang Tinggi efektif": ti,
            "Rendah": h["RENDAH"], "Sedang": h["SEDANG"], "Tinggi": h["TINGGI"],
            "% Sedang": h["SEDANG"] / n * 100, "% Tinggi": h["TINGGI"] / n * 100,
            "Baris berubah vs S0": berubah, "% berubah vs S0": berubah / n * 100,
            "% sama dg baseline": sama_base / n_banding * 100 if n_banding else np.nan}, kat

def baris_hasil(meta, fungsi):
    try:
        ringkas, _ = fungsi()
        return {**meta, **ringkas, "Keterangan": ""}
    except ValueError as galat:
        return {**meta, "Keterangan": f"DILEWATI: {galat}"}

_uji, _kat = jalankan(PRIOR_UTAMA, LR_UTAMA, AMBANG_SEDANG, AMBANG_TINGGI, SKEMA_UTAMA, MODE_REKAL)
assert (_kat == KAT_REF).all(), "Fungsi jalankan() tidak mereproduksi skenario referensi"
print("Fungsi uji siap; konfigurasi referensi tereproduksi.")

OAT: mode kalibrasi ulang dan mode tetap.

In [ ]:
baris = []
for mode in MODE_URUT:
    for skema in SKEMA_VALID:
        for prior in PRIOR_SWEEP:
            baris.append(baris_hasil({"Mode": mode, "Skema": skema, "Prior": prior},
                                     lambda: jalankan(prior, LR_UTAMA, AMBANG_SEDANG, AMBANG_TINGGI, skema, mode)))
tabel_sapu_prior = pd.DataFrame(baris)

tampil = tabel_sapu_prior[tabel_sapu_prior["Prior"].isin(PRIOR_TAMPIL)]
print("Ringkasan sapuan prior (Sedang / Tinggi, jumlah transaksi)")
ring = tampil.assign(Ringkas=lambda d: d["Sedang"].map("{:,.0f}".format) + " / " + d["Tinggi"].map("{:,.0f}".format)
                     ).pivot_table(index=["Mode", "Skema"], columns="Prior", values="Ringkas", aggfunc="first")
display(ring)
print("\nBaris yang berubah kategori dibanding referensi (%)")
display(tampil.pivot_table(index=["Mode", "Skema"], columns="Prior", values="% berubah vs S0").round(3))

print("\nPrior pertama saat >1% dan >5% transaksi berubah kategori")
titik = []
for (mode, skema), g in tabel_sapu_prior.groupby(["Mode", "Skema"]):
    g = g.dropna(subset=["% berubah vs S0"]).sort_values("Prior")
    g = g[~np.isclose(g["Prior"], PRIOR_UTAMA)]
    titik.append({"Mode": mode, "Skema": skema,
                  "Prior pertama >1%": g.loc[g["% berubah vs S0"] > 1, "Prior"].min(),
                  "Prior pertama >5%": g.loc[g["% berubah vs S0"] > 5, "Prior"].min(),
                  "Maks % berubah": g["% berubah vs S0"].max(),
                  "Prior dilewati (tidak valid)": int(tabel_sapu_prior[(tabel_sapu_prior["Mode"] == mode) &
                                                                         (tabel_sapu_prior["Skema"] == skema)]["Rendah"].isna().sum())})
tabel_titik_prior = pd.DataFrame(titik)
display(tabel_titik_prior.round(3))

fig, ax = plt.subplots(len(MODE_URUT), len(SKEMA_VALID), figsize=(11, 3.6 * len(MODE_URUT)), sharex=True)
ax = np.atleast_2d(ax)
for i, mode in enumerate(MODE_URUT):
    for j, skema in enumerate(SKEMA_VALID):
        g = tabel_sapu_prior[(tabel_sapu_prior["Mode"] == mode) & (tabel_sapu_prior["Skema"] == skema)].dropna(subset=["Rendah"])
        ax[i, j].plot(g["Prior"], g["% Sedang"], color=warna["SEDANG"], label="% Sedang")
        ax[i, j].plot(g["Prior"], g["% Tinggi"], color=warna["TINGGI"], label="% Tinggi")
        ax[i, j].plot(g["Prior"], g["% berubah vs S0"], color="#555555", ls="--", label="% berubah vs referensi")
        ax[i, j].axvline(PRIOR_UTAMA, color="#999999", lw=0.8)
        ax[i, j].set_title(f"Skema {skema}, mode {mode}", fontsize=10)
        ax[i, j].set_ylabel("% transaksi")
        if i == len(MODE_URUT) - 1:
            ax[i, j].set_xlabel("Prior")
        ax[i, j].legend(fontsize=7)
plt.tight_layout()
plt.show()

OAT: variasi LR per indikator.

In [ ]:
baris = []
for k in INDIKATOR:
    for mode in MODE_URUT:
        for nilai in LR_SWEEP:
            lr = dict(LR_UTAMA)
            lr[k] = float(nilai)
            baris.append(baris_hasil({"Indikator": k, "Mode": mode, "LR": nilai},
                                     lambda: jalankan(PRIOR_UTAMA, lr, AMBANG_SEDANG, AMBANG_TINGGI, SKEMA_UTAMA, mode)))
tabel_sapu_lr = pd.DataFrame(baris)

print("Baris yang berubah kategori dibanding referensi (%), per indikator dan LR")
for mode in MODE_URUT:
    print(f"\nMode: {mode}")
    display(tabel_sapu_lr[tabel_sapu_lr["Mode"] == mode].pivot_table(
        index="Indikator", columns="LR", values="% berubah vs S0").round(3))

fig, ax = plt.subplots(1, len(INDIKATOR), figsize=(4.6 * len(INDIKATOR), 3.6), sharey=False)
ax = np.atleast_1d(ax)
for sumbu, k in zip(ax, INDIKATOR):
    for mode, gaya in zip(MODE_URUT, ["-", ":"]):
        g = tabel_sapu_lr[(tabel_sapu_lr["Indikator"] == k) & (tabel_sapu_lr["Mode"] == mode)].dropna(subset=["Rendah"])
        sumbu.plot(g["LR"], g["% Sedang"], gaya, color=warna["SEDANG"], label=f"% Sedang ({mode})")
        sumbu.plot(g["LR"], g["% Tinggi"], gaya, color=warna["TINGGI"], label=f"% Tinggi ({mode})")
    sumbu.axvline(LR_UTAMA[k], color="#999999", lw=0.8)
    sumbu.set_xscale("log")
    sumbu.set_xlabel(f"LR {k}")
    sumbu.set_ylabel("% transaksi")
    sumbu.set_title(f"Sapuan LR {k}", fontsize=10)
    sumbu.legend(fontsize=6)
plt.tight_layout()
plt.show()

OAT: variasi skema.

In [ ]:
baris = []
for skema in SKEMA_VALID:
    for prior in PRIOR_GRID:
        for pengali in PENGALI_LR:
            lr = {k: LR_UTAMA[k] * pengali for k in INDIKATOR}
            baris.append(baris_hasil({"Skema": skema, "Prior": prior, "Pengali LR": pengali},
                                     lambda: jalankan(prior, lr, AMBANG_SEDANG, AMBANG_TINGGI, skema, MODE_REKAL)))
tabel_grid = pd.DataFrame(baris)

metrik = [("% Tinggi", "% transaksi Tinggi"), ("% berubah vs S0", "% berubah vs referensi")]
fig, ax = plt.subplots(len(metrik), len(SKEMA_VALID), figsize=(10.5, 4.2 * len(metrik)))
ax = np.atleast_2d(ax)
for i, (kolom, judul) in enumerate(metrik):
    for j, skema in enumerate(SKEMA_VALID):
        pv = (tabel_grid[tabel_grid["Skema"] == skema]
              .pivot_table(index="Prior", columns="Pengali LR", values=kolom))
        gambar = ax[i, j].imshow(pv.to_numpy(dtype=float), cmap="YlOrRd", aspect="auto")
        ax[i, j].set_xticks(range(len(pv.columns)))
        ax[i, j].set_xticklabels([f"x{c:g}" for c in pv.columns])
        ax[i, j].set_yticks(range(len(pv.index)))
        ax[i, j].set_yticklabels(pv.index)
        ax[i, j].set_title(f"{judul} - skema {skema}", fontsize=10)
        ax[i, j].set_xlabel("Pengali semua LR")
        ax[i, j].set_ylabel("Prior")
        for a_ in range(pv.shape[0]):
            for b_ in range(pv.shape[1]):
                nilai = pv.iat[a_, b_]
                ax[i, j].text(b_, a_, "-" if not np.isfinite(nilai) else f"{nilai:.1f}",
                              ha="center", va="center", fontsize=8)
        plt.colorbar(gambar, ax=ax[i, j], shrink=0.8)
plt.tight_layout()
plt.show()
n_tidak_valid = int(tabel_grid["Rendah"].isna().sum())
print(f"Kombinasi tidak valid (p1 >= 1 pada kalibrasi): {n_tidak_valid} dari {len(tabel_grid)} (ditandai '-')")

OAT: variasi ambang kategori Sedang.

In [ ]:
baris = []
for nilai in AMBANG_SEDANG_SWEEP:
    baris.append(baris_hasil({"Parameter": "Ambang Sedang", "Nilai": nilai},
                             lambda: jalankan(PRIOR_UTAMA, LR_UTAMA, nilai, AMBANG_TINGGI, SKEMA_UTAMA, MODE_REKAL)
                             if nilai < AMBANG_TINGGI else (_ for _ in ()).throw(ValueError("Sedang >= Tinggi"))))
for nilai in AMBANG_TINGGI_SWEEP:
    baris.append(baris_hasil({"Parameter": "Ambang Tinggi", "Nilai": nilai},
                             lambda: jalankan(PRIOR_UTAMA, LR_UTAMA, AMBANG_SEDANG, nilai, SKEMA_UTAMA, MODE_REKAL)
                             if nilai > AMBANG_SEDANG else (_ for _ in ()).throw(ValueError("Tinggi <= Sedang"))))
tabel_sapu_ambang = pd.DataFrame(baris)
display(tabel_sapu_ambang[["Parameter", "Nilai", "Rendah", "Sedang", "Tinggi", "% berubah vs S0",
                           "% sama dg baseline", "Keterangan"]].round(3))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for sumbu, parameter in zip(ax, ["Ambang Sedang", "Ambang Tinggi"]):
    g = tabel_sapu_ambang[tabel_sapu_ambang["Parameter"] == parameter].dropna(subset=["Rendah"])
    sumbu.plot(g["Nilai"], g["% Sedang"], color=warna["SEDANG"], label="% Sedang")
    sumbu.plot(g["Nilai"], g["% Tinggi"], color=warna["TINGGI"], label="% Tinggi")
    sumbu.set_xlabel(parameter)
    sumbu.set_ylabel("% transaksi")
    sumbu.legend(fontsize=8)
plt.tight_layout()
plt.show()

### **Monte Carlo**
Simulasi dengan parameter yang diacak untuk menilai ketidakpastian.

In [ ]:
IDX_KAT = {lab: i for i, lab in enumerate(LABEL)}
PARAM_MC = ["Prior"] + [f"LR {k}" for k in INDIKATOR] + ["Ambang Sedang", "Ambang Tinggi"]

def monte_carlo(skema, mode=MODE_REKAL):
    rng = np.random.default_rng(MC_SEED)
    cacah = np.zeros((len(POLA_UNIK), len(LABEL)), dtype=int)
    undian, ditolak, percobaan = [], 0, 0
    while len(undian) < N_MC and percobaan < N_MC * 6:
        percobaan += 1
        prior = rng.uniform(*MC_PRIOR)
        lr = {k: float(np.exp(rng.uniform(np.log(MC_LR[0]), np.log(MC_LR[1])))) for k in INDIKATOR}
        sed, tin = rng.uniform(*MC_AMBANG_SEDANG), rng.uniform(*MC_AMBANG_TINGGI)
        if not sed < tin:
            continue
        try:
            ringkas, kat = jalankan(prior, lr, sed, tin, skema, mode)
        except ValueError:
            ditolak += 1
            continue
        for lab, j in IDX_KAT.items():
            cacah[kat == lab, j] += 1
        undian.append({"Prior": prior, **{f"LR {k}": lr[k] for k in INDIKATOR},
                       "Ambang Sedang": sed, "Ambang Tinggi": tin,
                       "% Sedang": ringkas["% Sedang"], "% Tinggi": ringkas["% Tinggi"],
                       "% berubah vs S0": ringkas["% berubah vs S0"],
                       "% sama dg baseline": ringkas["% sama dg baseline"]})
    return pd.DataFrame(undian), cacah, ditolak

def peringkat_korelasi(d):
    baris = []
    for p in PARAM_MC:
        baris.append({"Parameter": p, **{t: d[p].rank().corr(d[t].rank())
                                         for t in ["% Sedang", "% Tinggi", "% berubah vs S0", "% sama dg baseline"]}})
    return pd.DataFrame(baris)

MC_UNDIAN, MC_POLA, MC_PARAM, MC_RINGKAS = {}, [], [], []
for skema in SKEMA_VALID:
    d, cacah, ditolak = monte_carlo(skema)
    MC_UNDIAN[skema] = d
    n_ok = len(d)
    if n_ok == 0:
        print(f"Skema {skema}: tidak ada undian valid.")
        continue
    stabil = np.array([cacah[i, IDX_KAT[KAT_REF[i]]] / n_ok for i in range(len(POLA_UNIK))])
    for i in range(len(POLA_UNIK)):
        MC_POLA.append({"Skema": skema, "Pola (" + " / ".join(INDIKATOR) + ")": teks_pola(POLA_UNIK[i]),
                        "Jumlah Baris": int(JUMLAH_POLA[i]), "Kategori referensi": KAT_REF[i],
                        **{f"% undian {lab}": cacah[i, j] / n_ok * 100 for lab, j in IDX_KAT.items()},
                        "Stabilitas (% undian = referensi)": stabil[i] * 100})
    dinilai = TERSEDIA_POLA > 0
    total = int(JUMLAH_POLA[dinilai].sum())
    korelasi = peringkat_korelasi(d).assign(Skema=skema)
    MC_PARAM.append(korelasi)
    baris_ringkas = {"Skema": skema, "Undian valid": n_ok, "Undian ditolak (kalibrasi tidak valid)": ditolak,
                     "Median % berubah vs referensi": d["% berubah vs S0"].median(),
                     "Persentil 5-95 % berubah": f"{d['% berubah vs S0'].quantile(.05):.2f} - {d['% berubah vs S0'].quantile(.95):.2f}",
                     "Persentil 5-95 % Sedang": f"{d['% Sedang'].quantile(.05):.2f} - {d['% Sedang'].quantile(.95):.2f}",
                     "Persentil 5-95 % Tinggi": f"{d['% Tinggi'].quantile(.05):.2f} - {d['% Tinggi'].quantile(.95):.2f}"}
    for batas in (0.95, 0.80, 0.50):
        baris_ringkas[f"% transaksi yang kategorinya sama dg referensi di >= {int(batas * 100)}% undian"] = (
            JUMLAH_POLA[dinilai & (stabil >= batas)].sum() / total * 100)
    MC_RINGKAS.append(baris_ringkas)

tabel_mc_ringkas = pd.DataFrame(MC_RINGKAS)
tabel_mc_pola = pd.DataFrame(MC_POLA)
tabel_mc_param = pd.concat(MC_PARAM, ignore_index=True) if MC_PARAM else pd.DataFrame()

print(f"Monte Carlo: {N_MC} undian valid per skema; prior {MC_PRIOR}, LR {MC_LR} (log), "
      f"ambang Sedang {MC_AMBANG_SEDANG}, Tinggi {MC_AMBANG_TINGGI}")
display(tabel_mc_ringkas.set_index("Skema").T)

print("\nStabilitas per pola indikator (skema utama), diurutkan menurut jumlah baris")
kolom_pola_mc = "Pola (" + " / ".join(INDIKATOR) + ")"
display(tabel_mc_pola[tabel_mc_pola["Skema"] == SKEMA_UTAMA]
        .sort_values("Jumlah Baris", ascending=False).drop(columns="Skema").round(2))

print("\nKorelasi peringkat (Spearman) parameter dengan hasil agregat")
for skema in SKEMA_VALID:
    if len(tabel_mc_param):
        print(f"Skema {skema}")
        display(tabel_mc_param[tabel_mc_param["Skema"] == skema].drop(columns="Skema").set_index("Parameter").round(3))

if len(tabel_mc_param):
    fig, ax = plt.subplots(1, len(SKEMA_VALID), figsize=(5.6 * len(SKEMA_VALID), 3.6), sharex=True)
    ax = np.atleast_1d(ax)
    for sumbu, skema in zip(ax, SKEMA_VALID):
        g = tabel_mc_param[tabel_mc_param["Skema"] == skema].set_index("Parameter")
        urut = g["% berubah vs S0"].abs().sort_values().index
        sumbu.barh(range(len(urut)), g.loc[urut, "% Tinggi"], color=warna["TINGGI"], alpha=0.8, label="% Tinggi")
        sumbu.barh(np.arange(len(urut)) + 0.0, g.loc[urut, "% Sedang"], color=warna["SEDANG"], alpha=0.5, label="% Sedang")
        sumbu.set_yticks(range(len(urut)))
        sumbu.set_yticklabels(urut, fontsize=8)
        sumbu.axvline(0, color="#555555", lw=0.6)
        sumbu.set_xlabel("Korelasi peringkat (Spearman)")
        sumbu.set_title(f"Skema {skema}", fontsize=10)
        sumbu.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

Penghitungan ulang indikator untuk definisi alternatif.

In [ ]:
def bangun_X(faktor=None, bulan=6, min_n=None, eksklusif=None):
    """Hitung ulang E1, E2, E3 untuk definisi tertentu (mengembalikan DataFrame Int64 seperti X)."""
    global FAKTOR_IQR, MIN_TRANSAKSI_IQR
    faktor = FAKTOR_IQR if faktor is None else faktor
    min_n = MIN_TRANSAKSI_IQR if min_n is None else min_n
    eksklusif = E1_NA_BILA_E3 if eksklusif is None else eksklusif
    simpan = (FAKTOR_IQR, MIN_TRANSAKSI_IQR)
    FAKTOR_IQR, MIN_TRANSAKSI_IQR = faktor, min_n
    try:
        e1, _ = tandai_iqr(df_clean, KELOMPOK_E1, KOLOM_JUMLAH)
        sub_ = df_clean.loc[obat_pantauan]
        besar, _ = tandai_iqr(sub_, KELOMPOK_E3, KOLOM_JUMLAH)
    finally:
        FAKTOR_IQR, MIN_TRANSAKSI_IQR = simpan
    batas = penyaluran + pd.DateOffset(months=bulan)
    e2 = pd.Series(pd.NA, index=df_clean.index, dtype="Int64")
    e2.loc[tanggal_lengkap & (kedaluwarsa > penyaluran) & (kedaluwarsa <= batas)] = 1
    e2.loc[tanggal_lengkap & (kedaluwarsa > batas)] = 0
    jb = pd.Series(pd.NA, index=df_clean.index, dtype="Int64")
    if len(sub_) > 0:
        jb.loc[sub_.index] = besar
    if E3_FALLBACK_KE_E1:
        pakai = jb.isna().to_numpy(dtype=bool) & obat_pantauan & e1.notna().to_numpy(dtype=bool)
        jb.loc[pakai] = e1.loc[pakai]
    e3 = terapkan_aturan_e3(jb)
    e3k, e3r = pisah_e3(e3)
    e2k_b = pd.Series(pd.NA, index=df_clean.index, dtype="Int64") if E2K_ISOLASI else df_clean["E2k_rev"].astype("Int64")
    e2_biasa, e2_kuat = pisah_e2(terapkan_e2_rs(e2))
    hasil = pd.DataFrame({"E1": terapkan_e1_pbf(e1), "E2": e2_biasa, "E2p": e2_kuat, "E2k": e2k_b, "E3": e3k, "E3b": e3r}, index=df_clean.index)
    if eksklusif:
        hasil.loc[bool_np(hasil["E3"].eq(1)) | bool_np(hasil["E3b"].eq(1)), "E1"] = pd.NA
    return terapkan_anomali(hasil)

def kategori_dari_X(Xn, prior, lr, sedang, tinggi, skema):
    """Posterior dan kategori per baris untuk matriks bukti Xn (parameter dikalibrasi ulang dari frekuensinya)."""
    ind = [k for k in INDIKATOR if Xn[k].notna().any() and bool_np(Xn[k].eq(1)).any()]
    m = {k: float(bool_np(Xn[k].eq(1)).sum() / Xn[k].notna().sum()) for k in ind}
    param = {k: kalibrasi(m[k], prior, lr[k]) for k in ind}
    kode = Xn[ind].fillna(-1).to_numpy(dtype=int)
    pola, inv, cnt = np.unique(kode, axis=0, return_inverse=True, return_counts=True)
    inv = np.asarray(inv).reshape(-1)
    s1 = np.full(len(pola), float(prior))
    s0 = np.full(len(pola), 1.0 - float(prior))
    for j, k in enumerate(ind):
        p1, p0 = param[k]
        v = pola[:, j]
        s1 = np.where(v == 1, s1 * p1, np.where(v == 0, s1 * (1 - p1), s1))
        s0 = np.where(v == 1, s0 * p0, np.where(v == 0, s0 * (1 - p0), s0))
    tersedia = (pola >= 0).sum(axis=1)
    post = np.where(tersedia > 0, s1 / (s1 + s0), np.nan)
    se, ti = ambang_efektif(prior, sedang, tinggi, skema)
    kat = kategori_pola(post, se, ti)
    return kat[inv], m

_X0 = bangun_X(FAKTOR_IQR, 6, MIN_TRANSAKSI_IQR, E1_NA_BILA_E3)
for k in ["E1", "E2", "E2p", "E2k", "E3", "E3b"]:
    assert (_X0[k].fillna(-1).astype(int).to_numpy() == X[k].fillna(-1).astype(int).to_numpy()).all(), \
        f"bangun_X tidak mereproduksi {k}"
KAT_BARIS_REF = df_clean["Kategori Bayes"].astype(str).to_numpy()

varian = [("Referensi", dict())]
varian += [(f"Pengali IQR = {v}", dict(faktor=v)) for v in UJI_FAKTOR_IQR if v != FAKTOR_IQR]
varian += [(f"Batas E2 = {v} bulan", dict(bulan=v)) for v in UJI_BULAN_E2 if v != 6]
varian += [(f"Min. transaksi per kelompok = {v}", dict(min_n=v)) for v in UJI_MIN_TRANSAKSI if v != MIN_TRANSAKSI_IQR]
varian += [(f"E1 diabaikan bila E3=1: {not E1_NA_BILA_E3}", dict(eksklusif=not E1_NA_BILA_E3))]

baris = []
for nama, kw in varian:
    try:
        Xn = bangun_X(**kw)
        kat_baris, m = kategori_dari_X(Xn, PRIOR_UTAMA, LR_UTAMA, AMBANG_SEDANG, AMBANG_TINGGI, SKEMA_UTAMA)
        h = {lab: int((kat_baris == lab).sum()) for lab in LABEL}
        berubah = int((kat_baris != KAT_BARIS_REF).sum())
        baris.append({"Definisi": nama, **{f"m {k}": m.get(k, np.nan) for k in INDIKATOR},
                      "Rendah": h["RENDAH"], "Sedang": h["SEDANG"], "Tinggi": h["TINGGI"],
                      "Belum dinilai": h["BELUM DAPAT DINILAI"],
                      "Baris berubah vs referensi": berubah,
                      "% berubah vs referensi": berubah / len(df_clean) * 100, "Keterangan": ""})
    except ValueError as galat:
        baris.append({"Definisi": nama, "Keterangan": f"DILEWATI: {galat}"})
tabel_definisi = pd.DataFrame(baris)
print("Dampak perubahan definisi indikator (satu per satu) pada kategori Bayesian")
display(tabel_definisi.round({"m E1": 5, "m E2": 5, "m E3": 5, "% berubah vs referensi": 3}))

Ringkasan hasil Monte Carlo dan ekspor.

In [ ]:
ringkas = []
ringkas.append(["Jumlah transaksi", f"{len(df_clean):,}"])
ringkas.append(["Indikator dalam model", ", ".join(INDIKATOR)])
ringkas.append(["Prior referensi", f"{PRIOR_UTAMA}"])
ringkas.append(["LR referensi", ", ".join(f"{k}={LR_UTAMA[k]}" for k in INDIKATOR)])
ringkas.append(["Ambang referensi Sedang / Tinggi", f"{AMBANG_SEDANG} / {AMBANG_TINGGI}"])
ringkas.append(["Skema ambang utama", SKEMA_UTAMA])
hk = hitung_kategori(KAT_REF)
ringkas.append(["Kategori Bayes (Rendah/Sedang/Tinggi)", f"{hk['RENDAH']:,} / {hk['SEDANG']:,} / {hk['TINGGI']:,}"])
ringkas.append(["Kesepakatan Bayesian vs threshold (referensi)", f"{persen_sama_ref:.4f}%"])
ringkas.append(["Tingkat skor: Bayesian vs threshold", f"{level_bayes} vs {level_baseline}"])
ringkas.append(["E1 diabaikan bila E3 = 1", str(E1_NA_BILA_E3)])
for _, r_ in tabel_mc_ringkas.iterrows():
    _kunci = [c for c in tabel_mc_ringkas.columns if c.startswith("% transaksi yang kategorinya sama dg referensi di >= 95%")]
    if _kunci:
        ringkas.append([f"[MC skema {r_['Skema']}] % transaksi stabil di >=95% undian", f"{r_[_kunci[0]]:.2f}%"])
    ringkas.append([f"[MC skema {r_['Skema']}] median % berubah vs referensi", f"{r_['Median % berubah vs referensi']:.3f}%"])

for skema in SKEMA_URUT:
    t = tabel_oat[skema]
    oat_saja = t[(t["Jenis"] == "OAT") & t["% berubah vs S0"].notna()]
    if len(oat_saja):
        teratas = oat_saja.sort_values("Baris berubah vs S0", ascending=False).iloc[0]
        ringkas.append([f"[Skema {skema}] OAT paling berpengaruh",
                        f"{teratas['Skenario']} ({int(teratas['Baris berubah vs S0']):,} baris, "
                        f"{teratas['% berubah vs S0']:.3f}%)"])
        n_nol = int((oat_saja["Baris berubah vs S0"] == 0).sum())
        ringkas.append([f"[Skema {skema}] OAT tanpa perubahan kategori", f"{n_nol} dari {len(oat_saja)}"])
        for _, baris_p in oat_saja[oat_saja["Parameter diubah"] == "Prior"].iterrows():
            ringkas.append([f"[Skema {skema}] {baris_p['Skenario']}",
                            f"{int(baris_p['Baris berubah vs S0']):,} baris berubah "
                            f"({baris_p['% berubah vs S0']:.3f}%)"])
tabel_ringkas = pd.DataFrame(ringkas, columns=["Ukuran", "Nilai"])
display(tabel_ringkas)

if EKSPOR:
    lembar = {
        "Ringkasan": tabel_ringkas,
        "Frekuensi_Indikator": tabel_frek,
        "Kalibrasi_Prior": tabel_kalibrasi,
        "Parameter_Referensi": tabel_ref,
        "Ambang_Efektif": tabel_ambang,
        "Pola_Referensi": tabel_pola,
        "Pola_Dalam_Kategori": dalam_kategori,
        "Silang_Bayes_Baseline": silang.reset_index(),
        "OAT_Skema_A": tabel_oat["A"],
        "OAT_Skema_B": tabel_oat["B"],
        "Perbandingan_Skema": perbandingan_skema,
        "Peringkat_Sensitivitas": peringkat.reset_index() if len(peringkat) else peringkat,
        "OAT_Perubahan_Pola": tabel_perubahan,
        "OAT_Parameter": tabel_param_skenario,
        "Sapu_Prior": tabel_sapu_prior,
        "Sapu_Prior_Titik": tabel_titik_prior,
        "Sapu_LR": tabel_sapu_lr,
        "Grid_Prior_x_LR": tabel_grid,
        "Sapu_Ambang": tabel_sapu_ambang,
        "MC_Ringkasan": tabel_mc_ringkas,
        "MC_Stabilitas_Pola": tabel_mc_pola,
        "MC_Korelasi_Parameter": tabel_mc_param,
        "Uji_Definisi_Indikator": tabel_definisi,
        "Batas_IQR_E1": batas_e1,
        "Batas_IQR_E3": batas_e3,
    }
    try:
        with pd.ExcelWriter(BERKAS_EKSPOR, engine="openpyxl") as penulis:
            for nama, tabel in lembar.items():
                tabel.to_excel(penulis, sheet_name=nama[:31], index=False)
        print(f"Disimpan: {BERKAS_EKSPOR}")
    except Exception as galat:
        print(f"Ekspor Excel gagal ({galat}); menyimpan sebagai CSV.")
        for nama, tabel in lembar.items():
            tabel.to_csv(f"hasil_{nama}.csv", index=False)

if EKSPOR_PER_BARIS:
    kolom_baris = [c for c in ["Nama PBF", "Tujuan Penyaluran", "Jenis Sarana", "Kategori Obat",
                               KOLOM_JUMLAH, "E1_rev", "E2_rev", "E2k_rev", "E3_rev", "E3b_rev", "Posterior Bayes",
                               "Kategori Bayes", "Kategori Baseline", "Catatan Pendukung"]
                   if c in df_clean.columns]
    df_clean[kolom_baris].to_csv(BERKAS_PER_BARIS, index=False)
    print(f"Disimpan: {BERKAS_PER_BARIS}")

print("\nKolom baru di df_clean: E1_rev, E2_rev, Status E2_rev, Jumlah Besar E3, E3_rev, E3b_rev, "
      "Catatan Pendukung, Posterior Bayes, Kategori Bayes, Kategori Baseline, Jumlah Indikator Tersedia")

PEMETAAN TRANSAKSI DENGAN HASIL VALIDASI

### Pemetaan transaksi pakar
Pemetaan 12 transaksi pakar (T001-T012) dan uji kendala LR.

In [ ]:
import itertools
import numpy as np
import pandas as pd
from IPython.display import display

PAKAR = {
    "T001": "BELUM DAPAT DITENTUKAN", "T002": "TINGGI", "T003": "RENDAH",
    "T004": "RENDAH", "T005": "SEDANG", "T006": "RENDAH", "T007": "RENDAH",
    "T008": "TINGGI", "T009": "SEDANG", "T010": "RENDAH", "T011": "SEDANG",
    "T012": "TINGGI",
}

INDEKS_T = {}

CIRI_T = {
    "T001": {"No. Faktur": "8341104471", "Kategori Obat": "Cefixime", "Tanggal Penyaluran": "2025-08-05"},
    "T002": {"No. Faktur": "General", "Kategori Obat": "Cefixime", "Jenis Sarana": "Apotek",
             "Tanggal Penyaluran": "2025-07-31", "Tanggal Kedaluwarsa": "2025-12-31"},
    "T003": {"No. Faktur": "56240054961", "Kategori Obat": "TBC", "Tanggal Penyaluran": "2024-10-11"},
    "T004": {"No. Faktur": "130524003961", "Kategori Obat": "TBC", "Tanggal Penyaluran": "2024-01-23"},
    "T005": {"No. Faktur": "1322009", "Kategori Obat": "Cefixime", "Tanggal Penyaluran": "2025-02-01"},
    "T006": {"No. Faktur": "125078669", "Kategori Obat": "Cefixime", "Tanggal Penyaluran": "2025-07-15"},
    "T007": {"No. Faktur": "50103368", "Kategori Obat": "Cefixime", "Tanggal Penyaluran": "2025-01-23"},
    "T008": {"No. Faktur": "312326000139", "Kategori Obat": "Ketamine", "Tanggal Penyaluran": "2026-03-31"},
    "T009": {"No. Faktur": "49250270654", "Kategori Obat": "Ketamine", "Tanggal Penyaluran": "2025-12-30"},
    "T010": {"No. Faktur": "400117250005231", "Kategori Obat": "Cefixime", "Tanggal Penyaluran": "2025-03-11"},
    "T011": {"No. Faktur": "8341111990", "Kategori Obat": "Cefixime", "Tanggal Penyaluran": "2025-08-23"},
    "T012": {"No. Faktur": "NJ/2506/1506", "Kategori Obat": "Cefixime", "Tanggal Penyaluran": "2025-06-23"},
}

CIRI_T_CADANGAN = {
    "T004": [{"No. Faktur Asli": "130524003961"},
             {"Tanggal Penyaluran": "2024-01-23", "Jumlah": 25, "Jenis Sarana": "Rumah Sakit", "Kategori Obat": "TBC"},
             {"Tanggal Penyaluran": "2024-01-23", "Jumlah": 25, "Jenis Sarana": "Rumah Sakit"}],
    "T012": [{"No. Faktur Asli": "NJ/2506/1506"},
             {"Tanggal Penyaluran": "2025-06-23", "Kategori Obat": "Cefixime", "Jumlah": 15, "Jenis Sarana": "Apotek",
              "Tanggal Kedaluwarsa": "2027-02-28"},
             {"Tanggal Penyaluran": "2025-06-23", "Kategori Obat": "Cefixime", "Jumlah": 15, "Jenis Sarana": "Apotek"}],
}

LR_GRID_PAKAR = [1.5, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, 30]
BERKAS_PAKAR = "pemetaan_pakar.xlsx"

URUT = {"RENDAH": 0, "SEDANG": 1, "TINGGI": 2}
prasyarat = ["df_clean", "X", "FREK", "INDIKATOR", "kalibrasi", "PRIOR_UTAMA",
             "AMBANG_SEDANG", "AMBANG_TINGGI", "LR_UTAMA"]
hilang = [n for n in prasyarat if n not in globals()]
if hilang:
    raise NameError(f"Jalankan notebook bayesian_uji_menyeluruh.ipynb sampai selesai dahulu. Belum ada: {hilang}")
for kolom in ["Posterior Bayes", "Kategori Bayes", "Kategori Baseline"]:
    if kolom not in df_clean.columns:
        raise ValueError(f"Kolom '{kolom}' belum ada di df_clean; jalankan Bagian 3 notebook.")
if not set(PAKAR) >= set(INDEKS_T) | set(CIRI_T):
    raise ValueError("Kode transaksi di INDEKS_T / CIRI_T harus ada di PAKAR (T001-T012).")

def cari_kandidat(ciri):
    mask = np.ones(len(df_clean), dtype=bool)
    dipakai = []
    for kolom, nilai in ciri.items():
        if nilai is None or (isinstance(nilai, str) and not nilai.strip()):
            continue
        if kolom not in df_clean.columns:
            print(f"   peringatan: kolom '{kolom}' tidak ada di df_clean; diabaikan")
            continue
        seri = df_clean[kolom]
        if "tanggal" in kolom.casefold():
            ada = pd.to_datetime(seri, errors="coerce").dt.normalize()
            cocok = ada.eq(pd.to_datetime(nilai, errors="coerce").normalize()).to_numpy(dtype=bool)
        elif isinstance(nilai, (int, float, np.number)):
            angka = pd.to_numeric(seri, errors="coerce").to_numpy(dtype=float)
            cocok = np.isfinite(angka) & (np.abs(angka - float(nilai)) <= 0.005 * abs(float(nilai)) + 1e-9)
        else:
            cocok = seri.astype("string").str.casefold().str.contains(
                str(nilai).casefold(), regex=False, na=False).to_numpy(dtype=bool)
        mask &= cocok
        dipakai.append(kolom)
    return df_clean.index[mask], dipakai

indeks_final, catatan_cari = {}, {}
for kode in PAKAR:
    if kode in INDEKS_T:
        if INDEKS_T[kode] not in df_clean.index:
            raise KeyError(f"{kode}: indeks {INDEKS_T[kode]!r} tidak ada di df_clean")
        indeks_final[kode] = INDEKS_T[kode]
        catatan_cari[kode] = "indeks diisi langsung"
    elif kode in CIRI_T:
        kandidat, dipakai = cari_kandidat(CIRI_T[kode])
        if len(kandidat) == 0 and any("tanggal" in k.casefold() for k in CIRI_T[kode]):
            kandidat, dipakai = cari_kandidat({k: v for k, v in CIRI_T[kode].items() if "tanggal" not in k.casefold()})
            dipakai = dipakai + ["(tanpa tanggal)"]
        for _alt in CIRI_T_CADANGAN.get(kode, []):
            if len(kandidat) != 0:
                break
            kandidat, dipakai = cari_kandidat(_alt)
            dipakai = dipakai + ["(ciri cadangan)"]
        if len(kandidat) == 1:
            indeks_final[kode] = kandidat[0]
            catatan_cari[kode] = f"1 kandidat (kolom: {', '.join(dipakai)})"
        elif len(kandidat) == 0:
            _fk = str(CIRI_T[kode].get("No. Faktur", "")).casefold()
            _ada = 0
            if _fk:
                for _kf in ["No. Faktur", "No. Faktur Asli"]:
                    if _kf in df_clean.columns:
                        _ada += int(df_clean[_kf].astype("string").str.casefold().str.contains(_fk, regex=False, na=False).sum())
            catatan_cari[kode] = ("TIDAK ADA kandidat. Nomor faktur " + ("TIDAK ditemukan di df_clean sama sekali (kemungkinan baris terbuang saat pembersihan; "
                                  "cek di data mentah)" if _ada == 0 else f"ada di {_ada} baris, tetapi ciri lain tidak cocok; longgarkan ciri"))
        else:
            catatan_cari[kode] = f"{len(kandidat)} kandidat; pilih satu lalu isi INDEKS_T"
            print(f"\n{kode}: {len(kandidat)} kandidat (maks 10 ditampilkan)")
            kolom_lihat = [c for c in ["Nama PBF", "Kategori Obat", "Jenis Sarana", "Jumlah Setelah Konversi",
                                       "Kategori Satuan", "Tanggal Penyaluran", "Tanggal Kedaluwarsa"]
                           if c in df_clean.columns]
            display(df_clean.loc[kandidat[:10], kolom_lihat])
    else:
        catatan_cari[kode] = "belum dipetakan"

print("\nStatus pemetaan")
display(pd.DataFrame({"Kode": list(PAKAR), "Penilaian pakar": list(PAKAR.values()),
                      "Indeks df_clean": [indeks_final.get(k, None) for k in PAKAR],
                      "Keterangan": [catatan_cari[k] for k in PAKAR]}))

if not indeks_final:
    print("\nBelum ada transaksi yang terpetakan. Isi INDEKS_T atau CIRI_T di bagian konfigurasi, lalu jalankan ulang sel ini.")
else:
    baris = []
    for kode, idx in indeks_final.items():
        r = df_clean.loc[idx]
        e1 = X.loc[idx, "E1"] if "E1" in X.columns else pd.NA
        e2 = (X.loc[idx, "E2"] if "E2" in X.columns else pd.NA)
        if (e2 is pd.NA or pd.isna(e2)) and "E2p" in X.columns:
            e2 = X.loc[idx, "E2p"]
        e3 = X.loc[idx, "E3"] if "E3" in X.columns else pd.NA
        e3b = X.loc[idx, "E3b"] if "E3b" in X.columns else pd.NA
        e2k = X.loc[idx, "E2k"] if "E2k" in X.columns else pd.NA
        pakar = PAKAR[kode]
        kb, kbase = str(r["Kategori Bayes"]), str(r["Kategori Baseline"])
        nilai = pakar in URUT
        baris.append({
            "Kode": kode, "Indeks": idx, "Penilaian pakar": pakar,
            "E1 (model)": e1, "E2": e2, "E3": e3, "E3b": e3b, "E2k": e2k,
            "Posterior": r["Posterior Bayes"], "Kategori Bayes": kb, "Kategori Baseline": kbase,
            "Cocok Bayes": (kb == pakar) if nilai else None,
            "Cocok Baseline": (kbase == pakar) if nilai else None,
            "Selisih tingkat Bayes (model - pakar)": (URUT.get(kb, np.nan) - URUT[pakar]) if nilai and kb in URUT else np.nan,
            "Alasan E3": r["Alasan E3"] if "Alasan E3" in df_clean.columns else "",
            "Catatan Pendukung": r["Catatan Pendukung"] if "Catatan Pendukung" in df_clean.columns else "",
        })
    tabel_pakar = pd.DataFrame(baris)
    print("\nPenilaian pakar vs model per transaksi")
    display(tabel_pakar.round({"Posterior": 4}))

    ev = tabel_pakar[tabel_pakar["Penilaian pakar"].isin(URUT)]
    n_ev = len(ev)
    if n_ev:
        sel = ev["Selisih tingkat Bayes (model - pakar)"].abs()
        sel_base = (ev["Kategori Baseline"].map(URUT) - ev["Penilaian pakar"].map(URUT)).abs()
        ringkas = pd.DataFrame([
            ["Transaksi yang dapat dibandingkan", n_ev],
            ["Bayesian: kategori sama persis", f"{int(ev['Cocok Bayes'].sum())} dari {n_ev}"],
            ["Bayesian: selisih <= 1 tingkat", f"{int((sel <= 1).sum())} dari {n_ev}"],
            ["Baseline: kategori sama persis", f"{int(ev['Cocok Baseline'].sum())} dari {n_ev}"],
            ["Baseline: selisih <= 1 tingkat", f"{int((sel_base <= 1).sum())} dari {n_ev}"],
        ], columns=["Ukuran", "Nilai"])
        print("\nRingkasan kesesuaian (uji awal, bukan akurasi: sampel kecil dan satu pakar)")
        display(ringkas)
        print("Silang pakar (baris) x Bayesian (kolom)")
        display(pd.crosstab(ev["Penilaian pakar"], ev["Kategori Bayes"]))
        salah = ev[~ev["Cocok Bayes"].astype(bool)]
        if len(salah):
            print("\nTransaksi yang kategorinya berbeda (periksa satu per satu):")
            display(salah[["Kode", "Penilaian pakar", "Kategori Bayes", "E1 (model)", "E2", "E3", "E3b", "Posterior",
                           "Alasan E3", "Catatan Pendukung"]].round({"Posterior": 4}))

        if n_ev < 4:
            print("\nUji kendala LR dilewati: kurang dari 4 transaksi yang dapat dibandingkan.")
            tabel_lr, ringkas_lr = pd.DataFrame(), pd.DataFrame()
        else:
            idx_ev = [indeks_final[k] for k in ev["Kode"]]
            target = ev["Penilaian pakar"].map(URUT).to_numpy()
            nilai_x = {k: pd.to_numeric(X.loc[idx_ev, k], errors="coerce").astype("float64").to_numpy()
                       for k in INDIKATOR}

            def kategori_baris(lr):
                s1 = np.full(n_ev, float(PRIOR_UTAMA))
                s0 = np.full(n_ev, 1.0 - float(PRIOR_UTAMA))
                for k in INDIKATOR:
                    p1, p0 = kalibrasi(FREK[k], PRIOR_UTAMA, lr[k])
                    v = nilai_x[k]
                    s1 = np.where(v == 1, s1 * p1, np.where(v == 0, s1 * (1 - p1), s1))
                    s0 = np.where(v == 1, s0 * p0, np.where(v == 0, s0 * (1 - p0), s0))
                post = s1 / (s1 + s0)
                return np.where(post >= AMBANG_TINGGI, 2, np.where(post >= AMBANG_SEDANG, 1, 0))

            hasil = []
            for kombinasi in itertools.product(LR_GRID_PAKAR, repeat=len(INDIKATOR)):
                lr = dict(zip(INDIKATOR, kombinasi))
                try:
                    kat = kategori_baris(lr)
                except ValueError:
                    continue
                hasil.append({**{f"LR {k}": lr[k] for k in INDIKATOR},
                              "Cocok persis": int((kat == target).sum()),
                              "Selisih <=1": int((np.abs(kat - target) <= 1).sum())})
            tabel_lr = pd.DataFrame(hasil)
            if tabel_lr.empty:
                print("\nTidak ada kombinasi LR yang valid.")
                ringkas_lr = pd.DataFrame()
            else:
                maks = int(tabel_lr["Cocok persis"].max())
                terbaik = tabel_lr[tabel_lr["Cocok persis"] == maks]
                ref = kategori_baris(LR_UTAMA)
                cocok_ref = int((ref == target).sum())
                print(f"\nUji kendala LR pada {n_ev} transaksi (prior {PRIOR_UTAMA}, ambang {AMBANG_SEDANG}/{AMBANG_TINGGI})")
                print(f"LR referensi {LR_UTAMA}: cocok persis {cocok_ref} dari {n_ev}")
                print(f"Kecocokan tertinggi di grid: {maks} dari {n_ev}, dicapai oleh {len(terbaik)} dari {len(tabel_lr)} kombinasi")
                ringkas_lr = pd.DataFrame({
                    "Indikator": INDIKATOR,
                    "LR minimum (kombinasi terbaik)": [terbaik[f"LR {k}"].min() for k in INDIKATOR],
                    "LR maksimum (kombinasi terbaik)": [terbaik[f"LR {k}"].max() for k in INDIKATOR],
                    "LR referensi": [LR_UTAMA[k] for k in INDIKATOR],
                })
                display(ringkas_lr)
                print("Kombinasi terbaik (maks 15 ditampilkan):")
                display(terbaik.head(15))
                print("PERINGATAN: hanya", n_ev, "kasus untuk", len(INDIKATOR),
                      "parameter. Baca sebagai RENTANG yang konsisten dengan penilaian pakar, bukan nilai terbaik.")

    try:
        with pd.ExcelWriter(BERKAS_PAKAR, engine="openpyxl") as penulis:
            tabel_pakar.to_excel(penulis, sheet_name="Pakar_vs_Model", index=False)
            if n_ev:
                ringkas.to_excel(penulis, sheet_name="Ringkasan", index=False)
            if "tabel_lr" in globals() and len(tabel_lr):
                ringkas_lr.to_excel(penulis, sheet_name="Rentang_LR", index=False)
                tabel_lr.to_excel(penulis, sheet_name="Grid_LR", index=False)
        print(f"\nDisimpan: {BERKAS_PAKAR}")
    except Exception as galat:
        print(f"Ekspor Excel gagal: {galat}")

Diskrepansi elisitasi: semua kombinasi saklar aturan Cefixime dan kedaluwarsa.

In [ ]:
import itertools
import numpy as np
import pandas as pd
from IPython.display import display

BERKAS_DISKREPANSI = "diskrepansi_elisitasi.xlsx"
SINGKAT = {"WAWANCARA": "W", "FORM_T012": "T012", "FORM_T009": "T009", "FORM_T005_T011": "T005/T011"}

def kategori_skenario(cef, exp, e2k):
    """Kategori tiap baris df_clean untuk satu kombinasi saklar (prior, LR lain, dan ambang sama seperti referensi)."""
    global INDIKATOR
    kuat = {"Ketamine", "Meropenem", "Orlistat"} | ({"Cefixime"} if cef == "FORM_T012" else set())
    ringan = set(ATURAN_E3) - kuat
    e3s = df_clean["E3 semua pantauan"].astype("Int64")
    e3k = e3s.copy()
    e3k.loc[~bool_np(kategori_obat.isin(sorted(kuat)))] = pd.NA
    e3r = e3s.copy()
    e3r.loc[~bool_np(kategori_obat.isin(sorted(ringan)))] = pd.NA
    e2k_s = (pd.Series(pd.NA, index=df_clean.index, dtype="Int64") if e2k == "WAWANCARA"
             else df_clean["E2k_rev"].astype("Int64"))
    _e2b, _e2p = pisah_e2(terapkan_e2_rs(df_clean["E2_rev"].astype("Int64")))
    Xn = pd.DataFrame({"E1": terapkan_e1_pbf(df_clean["E1_rev"].astype("Int64")),
                       "E2": _e2b, "E2p": _e2p, "E2k": e2k_s, "E3": e3k, "E3b": e3r},
                      index=df_clean.index)
    if E1_NA_BILA_E3:
        Xn.loc[bool_np(Xn["E3"].eq(1)) | bool_np(Xn["E3b"].eq(1)), "E1"] = pd.NA
    Xn = terapkan_anomali(Xn)
    lr = dict(LR_UTAMA)
    lr["E2"] = 25.0 if exp == "WAWANCARA" else 10.0
    dipotong = []
    for k in Xn.columns:
        n_k, s_k = int(Xn[k].notna().sum()), int(bool_np(Xn[k].eq(1)).sum())
        if n_k and s_k:
            m_k = s_k / n_k
            maks = np.inf if m_k <= PRIOR_UTAMA else (1 - PRIOR_UTAMA) / (m_k - PRIOR_UTAMA)
            if lr[k] >= maks:
                lr[k] = round(0.95 * maks, 2)
                dipotong.append(k)
    simpan = INDIKATOR
    INDIKATOR = list(Xn.columns)
    try:
        kat, _ = kategori_dari_X(Xn, PRIOR_UTAMA, lr, AMBANG_SEDANG, AMBANG_TINGGI, SKEMA_UTAMA)
    finally:
        INDIKATOR = simpan
    return np.asarray(kat, dtype=object), dipotong

_ref_kunci = (TOGGLE_CEFIXIME_RULE, TOGGLE_EXPIRY_RULE, TOGGLE_E2K_RULE)
_semua = list(itertools.product(("WAWANCARA", "FORM_T012"), ("WAWANCARA", "FORM_T009"), ("WAWANCARA", "FORM_T005_T011")))
_hasil_sk = {k: kategori_skenario(*k) for k in _semua}
_kat = {k: v[0] for k, v in _hasil_sk.items()}
_ref = _kat[_ref_kunci]

_ada_t = "indeks_final" in globals() and "PAKAR" in globals() and bool(indeks_final)
_t_pos = {}
if _ada_t:
    for _kode, _idx in indeks_final.items():
        if _idx in df_clean.index:
            _t_pos[_kode] = df_clean.index.get_loc(_idx)

baris, baris_t = [], {}
for k in _semua:
    kat = _kat[k]
    hitung = {lab: int((kat == lab).sum()) for lab in ["RENDAH", "SEDANG", "TINGGI", "BELUM DAPAT DINILAI"]}
    cocok, tidak = [], []
    for kode, pos in _t_pos.items():
        if PAKAR.get(kode) in ("RENDAH", "SEDANG", "TINGGI"):
            (cocok if kat[pos] == PAKAR[kode] else tidak).append(kode)
    nama = f"C={SINGKAT[k[0]]} | ED={SINGKAT[k[1]]} | E2k={SINGKAT[k[2]]}"
    baris.append({"Skenario": nama + ("  (dipakai)" if k == _ref_kunci else ""),
                  "Saklar Cefixime": k[0], "Saklar Kedaluwarsa": k[1], "Saklar E2k": k[2],
                  "Rendah": hitung["RENDAH"], "Sedang": hitung["SEDANG"], "Tinggi": hitung["TINGGI"],
                  "% Sedang": hitung["SEDANG"] / len(df_clean) * 100, "% Tinggi": hitung["TINGGI"] / len(df_clean) * 100,
                  "Baris berubah vs dipakai": int((kat != _ref).sum()),
                  "T cocok": f"{len(cocok)} dari {len(cocok) + len(tidak)}" if _t_pos else "(T belum terpetakan)",
                  "T tidak cocok": ", ".join(tidak),
                  "LR dipotong": ", ".join(_hasil_sk[k][1])})
    baris_t[nama] = [kat[_t_pos[kd]] if kd in _t_pos else "" for kd in PAKAR] if _ada_t else []
tabel_diskrepansi = pd.DataFrame(baris)
print("Perbandingan delapan kombinasi saklar (prior, ambang, dan LR lain sama)")
display(tabel_diskrepansi.round({"% Sedang": 3, "% Tinggi": 3}))

if _ada_t:
    tabel_diskrepansi_T = pd.DataFrame(baris_t, index=list(PAKAR))
    tabel_diskrepansi_T.insert(0, "Penilaian pakar (form)", [PAKAR[k] for k in PAKAR])
    tabel_diskrepansi_T.insert(0, "Terpetakan", [k in _t_pos for k in PAKAR])
    print("\nKategori tiap transaksi T pada tiap skenario (kosong = belum terpetakan)")
    display(tabel_diskrepansi_T)
    print("Catatan: T001 'belum dapat ditentukan' tidak dihitung; T004 dan T012 baru ikut bila sudah terpetakan.")
else:
    tabel_diskrepansi_T = pd.DataFrame()
    print("\nTransaksi T belum terpetakan: jalankan sel pemetaan pakar dahulu untuk membandingkannya.")

with pd.ExcelWriter(BERKAS_DISKREPANSI, engine="openpyxl") as _p:
    tabel_diskrepansi.to_excel(_p, sheet_name="Skenario", index=False)
    if len(tabel_diskrepansi_T):
        tabel_diskrepansi_T.to_excel(_p, sheet_name="Transaksi_T")
print(f"Disimpan: {BERKAS_DISKREPANSI}")

> **Sel generator sampel berstrata sengaja dihapus dari versi ini.**
> Sampel validasi sudah dibuat dan dinilai pakar. Menjalankannya lagi akan membuat kunci baru yang tidak cocok dengan lembar pakar.

Membaca hasil penilaian pakar dan membandingkannya dengan model.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

BERKAS_ISI = "lembar_penilaian_pakar_terisi.xlsx"
BERKAS_KUNCI_BACA = "kunci_sampel.xlsx"
BERKAS_HASIL = "hasil_penilaian_pakar.xlsx"

URUT_P = {"RENDAH": 0, "SEDANG": 1, "TINGGI": 2}
KOLOM_N = "NILAI ANDA: seberapa perlu diperiksa?"
KOLOM_A = "ALASAN UTAMA"

def _ke_kategori(x):
    t = str(x).strip().casefold()
    if t.startswith("rendah"):
        return "RENDAH"
    if t.startswith("sedang"):
        return "SEDANG"
    if t.startswith("tinggi"):
        return "TINGGI"
    if t.startswith("tidak bisa"):
        return "TIDAK BISA DINILAI"
    return ""

isi = pd.read_excel(BERKAS_ISI, sheet_name="Penilaian", dtype=str).fillna("")
if "No" not in isi.columns or KOLOM_N not in isi.columns:
    raise ValueError("Kolom 'No' atau kolom nilai tidak ditemukan. Pakai lembar dari Bagian 5 tanpa mengubah judul kolom.")
isi["Nilai pakar"] = isi[KOLOM_N].map(_ke_kategori)
kosong = isi.loc[isi["Nilai pakar"] == "", "No"].tolist()
print(f"Baris di lembar: {len(isi)} | terisi: {int((isi['Nilai pakar'] != '').sum())} | belum terisi: {len(kosong)}")
if kosong:
    print("Belum terisi:", ", ".join(kosong[:40]) + (" ..." if len(kosong) > 40 else ""))

kunci = kunci_sampel if "kunci_sampel" in globals() else pd.read_excel(BERKAS_KUNCI_BACA, sheet_name="Kunci")
m = kunci.merge(isi[["No", "Nilai pakar", KOLOM_A]], left_on="ID lembar", right_on="No", how="left")
if m["No"].isna().any():
    print("PERINGATAN: ada ID di kunci yang tidak ditemukan di lembar:", m.loc[m["No"].isna(), "ID lembar"].tolist()[:10])
m["Nilai pakar"] = m["Nilai pakar"].fillna("")

dup = m[m["Duplikat uji ulang"] == "ya"]
rows = []
for _, r in dup.iterrows():
    a = m.loc[m["ID lembar"] == r["ID asli (bila duplikat)"], "Nilai pakar"]
    a = a.iloc[0] if len(a) else ""
    if r["Nilai pakar"] in URUT_P and a in URUT_P:
        rows.append({"ID asli": r["ID asli (bila duplikat)"], "ID ulang": r["ID lembar"], "Penilaian 1": a,
                     "Penilaian 2": r["Nilai pakar"], "Sama": a == r["Nilai pakar"],
                     "Selisih tingkat": abs(URUT_P[a] - URUT_P[r["Nilai pakar"]])})
kons = pd.DataFrame(rows)
print("\n=== Konsistensi pakar pada transaksi duplikat ===")
if len(kons):
    print(f"Pasangan dapat dibandingkan: {len(kons)} | sama persis: {int(kons['Sama'].sum())} ({kons['Sama'].mean()*100:.1f}%) | selisih maks: {int(kons['Selisih tingkat'].max())}")
    display(kons)
else:
    print("Belum ada pasangan duplikat yang terisi lengkap.")

u = m[(m["Duplikat uji ulang"] != "ya") & m["Nilai pakar"].isin(URUT_P)].copy()
u["p"] = u["Nilai pakar"].map(URUT_P)
print(f"\nTransaksi unik yang dapat dibandingkan: {len(u)} dari {int((m['Duplikat uji ulang'] != 'ya').sum())}")
print("Tidak bisa dinilai oleh pakar:", int((m[m['Duplikat uji ulang'] != 'ya']['Nilai pakar'] == 'TIDAK BISA DINILAI').sum()))

def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    pusat = (p + z * z / (2 * n)) / (1 + z * z / n)
    sebar = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return (pusat - sebar, pusat + sebar)

def kappa_berbobot(a, b, k=3):
    a, b = np.asarray(a), np.asarray(b)
    O = np.zeros((k, k))
    for x, y in zip(a, b):
        O[x, y] += 1
    O /= O.sum()
    E = np.outer(O.sum(1), O.sum(0))
    W = np.abs(np.subtract.outer(np.arange(k), np.arange(k))) / (k - 1)
    den = (W * E).sum()
    return np.nan if den == 0 else 1 - (W * O).sum() / den

ringkas, tabel_silang = [], {}
for nama, kol in [("Bayes", "Kategori Bayes"), ("Baseline", "Kategori Baseline")]:
    v = u[u[kol].isin(URUT_P)].copy()
    if v.empty:
        continue
    v["m"] = v[kol].map(URUT_P)
    cocok = int((v["m"] == v["p"]).sum())
    lo, hi = wilson(cocok, len(v))
    w = v["Bobot"].to_numpy(dtype=float)
    ringkas.append({
        "Metode": nama, "n": len(v),
        "Cocok persis (sampel)": f"{cocok} ({cocok/len(v)*100:.1f}%)",
        "IK 95% (Wilson)": f"{lo*100:.1f}–{hi*100:.1f}%",
        "Cocok persis (berbobot populasi)": f"{(w*(v['m']==v['p'])).sum()/w.sum()*100:.1f}%",
        "Selisih <=1 tingkat": f"{(v['m'].sub(v['p']).abs() <= 1).mean()*100:.1f}%",
        "Model lebih tinggi dari pakar": int((v["m"] > v["p"]).sum()),
        "Model lebih rendah dari pakar": int((v["m"] < v["p"]).sum()),
        "Kappa berbobot (linear)": round(kappa_berbobot(v["p"], v["m"]), 3),
    })
    tabel_silang[nama] = pd.crosstab(v["Nilai pakar"], v[kol]).reindex(index=list(URUT_P), columns=list(URUT_P), fill_value=0)
print("\n=== Ringkasan kesesuaian model dengan pakar ===")
if ringkas:
    display(pd.DataFrame(ringkas))
    for nama, t in tabel_silang.items():
        print(f"\nTabel silang {nama} (baris = pakar, kolom = model):")
        display(t)
else:
    print("Belum ada data yang dapat dibandingkan.")

print("\n=== Menurut strata ===")
if len(u):
    u["m"] = u["Kategori Bayes"].map(URUT_P)
    u["cocok"] = (u["m"] == u["p"]).astype(int)
    per_strata = u.groupby(["Strata", "Nama strata"]).agg(
        n=("p", "size"), Cocok_Bayes=("cocok", "sum"),
        Pakar_rata2=("p", "mean"), Model_rata2=("m", "mean")).round(2).reset_index()
    display(per_strata)
    print("\n=== Alasan utama pakar menurut kategori pakar ===")
    display(pd.crosstab(u[KOLOM_A].replace("", "(kosong)"), u["Nilai pakar"]))
    print("\n=== Transaksi yang berbeda >= 1 tingkat (periksa satu per satu) ===")
    beda = u[u["Kategori Bayes"].map(URUT_P).sub(u["p"]).abs() >= 1]
    display(beda[["ID lembar", "Strata", "Nilai pakar", "Kategori Bayes", "Posterior", "E1 (model)", "E2", "E3",
                  "Alasan E3", "Catatan Pendukung", KOLOM_A]].round({"Posterior": 4}))

with pd.ExcelWriter(BERKAS_HASIL, engine="openpyxl") as penulis:
    m.to_excel(penulis, sheet_name="Gabungan", index=False)
    if ringkas:
        pd.DataFrame(ringkas).to_excel(penulis, sheet_name="Ringkasan", index=False)
        for nama, t in tabel_silang.items():
            t.to_excel(penulis, sheet_name=f"Silang_{nama}")
    if len(kons):
        kons.to_excel(penulis, sheet_name="Konsistensi", index=False)
print(f"\nDisimpan: {BERKAS_HASIL}")
print("CATATAN: n kecil; baca sebagai kesesuaian dengan penilaian pakar pada sampel ini, bukan akurasi prediksi.")

**PERBANDINGAN DENGAN BASELINE**

Perbandingan model Bayesian dengan dua baseline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

BERKAS_EVAL = "hasil_evaluasi_akhir.xlsx"
GAMBAR_7_1 = "gambar_7_1_bayes_vs_baseline.png"

for _n in ["df_clean", "X", "INDIKATOR", "HARAPAN_PAKAR"]:
    if _n not in globals():
        raise NameError(
            f"Variabel {_n} belum ada. Jalankan Bagian 3 sampai selesai."
        )

for _k in [
    "Kategori Bayes",
    "Kategori Baseline",
    "Posterior Bayes"
]:
    if _k not in df_clean.columns:
        raise ValueError(
            f"Kolom '{_k}' belum ada di df_clean. Jalankan Bagian 3."
        )

URUT7 = {
    "RENDAH": 0,
    "SEDANG": 1,
    "TINGGI": 2
}

IND7 = list(INDIKATOR)

HARAP_FS = {
    frozenset(k): v
    for k, v in HARAPAN_PAKAR.items()
}

def _aturan_pakar(pola):

    aktif = [
        IND7[j]
        for j, v in enumerate(pola)
        if v == 1
    ]

    if all(v == -1 for v in pola):
        return "BELUM DAPAT DINILAI"

    if not aktif:
        return "RENDAH"

    fs = frozenset(aktif)

    if fs in HARAP_FS:
        return HARAP_FS[fs]

    return "SEDANG" if len(aktif) == 1 else "TINGGI"

_Xn = X[IND7].apply(
    pd.to_numeric,
    errors="coerce"
)

_kode = _Xn.fillna(-1).to_numpy(dtype=int)

_uniq, _inv = np.unique(
    _kode,
    axis=0,
    return_inverse=True
)

_inv = np.asarray(_inv).ravel()

_b2 = np.array(
    [_aturan_pakar(p) for p in _uniq],
    dtype=object
)

df_clean["Kategori Aturan Pakar"] = _b2[_inv]

JUMLAH_AKTIF = _Xn.eq(1).sum(axis=1).to_numpy()

def kappa_linear(a, b, w=None, k=3):

    a = np.asarray(a, dtype=int)
    b = np.asarray(b, dtype=int)

    w = (
        np.ones(len(a))
        if w is None
        else np.asarray(w, dtype=float)
    )

    O = np.zeros((k, k))

    np.add.at(O, (a, b), w)

    O /= O.sum()

    E = np.outer(
        O.sum(1),
        O.sum(0)
    )

    W = (
        np.abs(
            np.subtract.outer(
                np.arange(k),
                np.arange(k)
            )
        ) / (k - 1)
    )

    den = (W * E).sum()

    return np.nan if den == 0 else 1 - (W * O).sum() / den

def banding_pasangan(
    nama_a,
    kol_a,
    nama_b,
    kol_b,
    data
):

    d = data[
        data[kol_a].isin(URUT7)
        & data[kol_b].isin(URUT7)
    ]

    a = d[kol_a].map(URUT7).to_numpy()
    b = d[kol_b].map(URUT7).to_numpy()

    return {
        "Pasangan": f"{nama_a} vs {nama_b}",
        "n": len(d),

        "Kategori sama (%)": round(
            (a == b).mean() * 100, 4
        ),

        "Berbeda (baris)": int(
            (a != b).sum()
        ),

        "Pertama lebih tinggi": int(
            (a > b).sum()
        ),

        "Pertama lebih rendah": int(
            (a < b).sum()
        ),

        "Kappa berbobot (linear)": round(
            kappa_linear(a, b), 4
        )
    }

URUTAN_LABEL = [
    "RENDAH",
    "SEDANG",
    "TINGGI",
    "BELUM DAPAT DINILAI"
]

KOLOM_METODE = {
    "Bayesian": "Kategori Bayes",

    "Baseline 1 (jumlah indikator)": "Kategori Baseline",

    "Baseline 2 (aturan pakar)": "Kategori Aturan Pakar"
}

tabel_7_1 = pd.DataFrame({
    m: df_clean[k].value_counts().reindex(
        URUTAN_LABEL,
        fill_value=0
    )

    for m, k in KOLOM_METODE.items()
})

tabel_7_1_persen = (
    tabel_7_1 / len(df_clean) * 100
).round(3)

print("=== 7.1 Distribusi kategori menurut metode (baris) ===")

display(tabel_7_1)

print("Persen:")

display(tabel_7_1_persen)

tabel_7_2 = pd.DataFrame([

    banding_pasangan(
        "Bayesian",
        "Kategori Bayes",
        "Baseline 1",
        "Kategori Baseline",
        df_clean
    ),

    banding_pasangan(
        "Bayesian",
        "Kategori Bayes",
        "Baseline 2",
        "Kategori Aturan Pakar",
        df_clean
    ),

    banding_pasangan(
        "Baseline 1",
        "Kategori Baseline",
        "Baseline 2",
        "Kategori Aturan Pakar",
        df_clean
    )

])

print(
    "\n=== 7.2 Kesesuaian berpasangan antar metode "
    "(seluruh transaksi yang dapat dinilai) ==="
)

display(tabel_7_2)

print(
    "Baca: kesesuaian adalah KESAMAAN keputusan, bukan akurasi. "
    "Akurasi hanya dapat dinilai terhadap penilaian pakar (7B)."
)

def silang(kol_baris, kol_kolom):

    d = df_clean[
        df_clean[kol_baris].isin(URUT7)
        & df_clean[kol_kolom].isin(URUT7)
    ]

    return pd.crosstab(
        d[kol_baris],
        d[kol_kolom]
    ).reindex(
        index=list(URUT7),
        columns=list(URUT7),
        fill_value=0
    )

tabel_7_3 = {

    "Baseline 1 (baris) x Bayesian (kolom)":
        silang(
            "Kategori Baseline",
            "Kategori Bayes"
        ),

    "Baseline 2 (baris) x Bayesian (kolom)":
        silang(
            "Kategori Aturan Pakar",
            "Kategori Bayes"
        ),

    "Baseline 1 (baris) x Baseline 2 (kolom)":
        silang(
            "Kategori Baseline",
            "Kategori Aturan Pakar"
        )

}

print("\n=== 7.3 Tabel silang ===")

for nama, t in tabel_7_3.items():

    print(nama)

    display(t)

_label_pola = np.array(
    [
        "/".join(
            "NA" if v == -1 else str(v)
            for v in p
        )
        for p in _uniq
    ],
    dtype=object
)

pola_df = pd.DataFrame({

    "Pola (" + "/".join(IND7) + ")":
        _label_pola[_inv],

    "Posterior":
        df_clean["Posterior Bayes"].to_numpy(),

    "Bayesian":
        df_clean["Kategori Bayes"].to_numpy(),

    "Baseline 1":
        df_clean["Kategori Baseline"].to_numpy(),

    "Baseline 2":
        df_clean["Kategori Aturan Pakar"].to_numpy()

})

_kp = pola_df.columns[0]

tabel_7_4 = (

    pola_df.groupby(
        [_kp, "Bayesian", "Baseline 1", "Baseline 2"],
        dropna=False
    )

    .agg(
        Baris=("Posterior", "size"),
        Posterior=("Posterior", "mean")
    )

    .reset_index()

)

tabel_7_4 = tabel_7_4[
    (tabel_7_4["Bayesian"] != tabel_7_4["Baseline 1"])
    |
    (tabel_7_4["Bayesian"] != tabel_7_4["Baseline 2"])
]

tabel_7_4 = tabel_7_4.sort_values(
    "Baris",
    ascending=False
).reset_index(drop=True)

print(
    "\n=== 7.4 Pola indikator yang kategorinya berbeda antar metode ==="
)

if len(tabel_7_4):

    display(
        tabel_7_4.round({
            "Posterior": 4
        })
    )

else:

    print(
        "Tidak ada: ketiga metode memberi kategori sama pada semua pola."
    )

post = df_clean["Posterior Bayes"]

ok = post.notna()

resolusi = [

    [
        "Tingkat skor unik: Bayesian",
        int(post[ok].round(6).nunique())
    ],

    [
        "Tingkat skor unik: Baseline 1 (jumlah indikator)",
        int(
            pd.Series(JUMLAH_AKTIF)[
                ok.to_numpy()
            ].nunique()
        )
    ],

    [
        "Tingkat skor unik: Baseline 2 (kategori)",
        int(
            df_clean.loc[
                ok,
                "Kategori Aturan Pakar"
            ].nunique()
        )
    ],

    [
        "Korelasi Spearman: posterior vs jumlah indikator aktif",

        round(
            float(
                pd.Series(
                    post[ok].to_numpy()
                ).corr(
                    pd.Series(
                        JUMLAH_AKTIF[ok.to_numpy()]
                    ),
                    method="spearman"
                )
            ),
            4
        )
    ]

]

_sed = (
    df_clean["Kategori Baseline"].eq("SEDANG")
    & ok
)

if _sed.any():

    sk = post[_sed].round(6)

    resolusi += [

        [
            "Baseline 1 = Sedang: jumlah transaksi",
            int(_sed.sum())
        ],

        [
            "Baseline 1 = Sedang: tingkat skor Bayesian yang membedakannya",
            int(sk.nunique())
        ],

        [
            "Baseline 1 = Sedang: persen transaksi yang oleh Bayesian dinaikkan ke Tinggi",

            round(
                float(
                    df_clean.loc[
                        _sed,
                        "Kategori Bayes"
                    ].eq("TINGGI").mean() * 100
                ),
                3
            )
        ],

        [
            "Baseline 1 = Sedang: posterior min / median / maks",

            f"{sk.min():.4f} / {sk.median():.4f} / {sk.max():.4f}"
        ]

    ]

tabel_7_5 = pd.DataFrame(
    resolusi,
    columns=["Ukuran", "Nilai"]
)

print("\n=== 7.5 Daya pembeda (resolusi) ===")

display(tabel_7_5)

sebaran_kat = (

    df_clean[ok]

    .groupby("Kategori Bayes")["Posterior Bayes"]

    .agg([
        "size",
        "min",
        "median",
        "max"
    ])

    .round(4)

    .reindex([
        l
        for l in URUTAN_LABEL
        if l in set(
            df_clean.loc[
                ok,
                "Kategori Bayes"
            ]
        )
    ])

)

print(
    "Sebaran posterior di dalam setiap kategori Bayesian:"
)

display(sebaran_kat)

warna = {

    "RENDAH": "#8fbf8f",

    "SEDANG": "#e6c35c",

    "TINGGI": "#d9534f",

    "BELUM DAPAT DINILAI": "#bbbbbb"

}

fig, ax = plt.subplots(
    1,
    2,
    figsize=(12, 4.2)
)

bawah = np.zeros(
    len(KOLOM_METODE),
    dtype=float
)

for lab in URUTAN_LABEL[:3]:

    nilai = pd.to_numeric(
        tabel_7_1_persen.loc[lab],
        errors="coerce"
    ).fillna(0).to_numpy(dtype=float)

    ax[0].bar(
        range(len(KOLOM_METODE)),
        nilai,
        bottom=bawah,
        color=warna[lab],
        label=lab
    )

    bawah += nilai

ax[0].set_xticks(
    range(len(KOLOM_METODE))
)

ax[0].set_xticklabels(
    [
        m.replace(" (", "\n(")
        for m in KOLOM_METODE
    ],
    fontsize=8
)

ax[0].set_ylabel("% transaksi")

ax[0].set_title("(a) Distribusi kategori")

ax[0].legend(fontsize=8)

if _sed.any():

    for lab in ["SEDANG", "TINGGI"]:

        v = post[
            _sed
            & df_clean["Kategori Bayes"].eq(lab)
        ]

        if len(v):

            ax[1].hist(
                v,
                bins=40,
                color=warna[lab],
                alpha=0.85,
                label=f"Bayesian: {lab} (n={len(v):,})"
            )

    ax[1].set_yscale("log")

    ax[1].set_xlabel("Posterior Bayesian")

    ax[1].set_ylabel(
        "Jumlah transaksi (skala log)"
    )

    ax[1].set_title(
        "(b) Transaksi Baseline 1 = Sedang:\n"
        "bagaimana Bayesian membedakannya"
    )

    ax[1].legend(fontsize=8)

plt.tight_layout()

try:

    plt.savefig(
        GAMBAR_7_1,
        dpi=200,
        bbox_inches="tight"
    )

    print(
        f"Gambar disimpan: {GAMBAR_7_1}"
    )

except Exception as galat:

    print(
        f"Gambar tidak tersimpan: {galat}"
    )

plt.show()

HASIL_EVAL = {

    "7_1_Distribusi":
        tabel_7_1.reset_index().rename(
            columns={"index": "Kategori"}
        ),

    "7_2_Kesesuaian":
        tabel_7_2,

    "7_4_Pola_berbeda":
        tabel_7_4,

    "7_5_Resolusi":
        tabel_7_5

}

for _kunci, _nama in zip(

    tabel_7_3,

    [
        "7_3_B1_x_Bayes",
        "7_3_B2_x_Bayes",
        "7_3_B1_x_B2"
    ]

):

    HASIL_EVAL[_nama] = (
        tabel_7_3[_kunci].reset_index()
    )

print(
    "\nCATATAN: Baseline 2 memakai tabel keputusan yang sama "
    "dengan sumber kalibrasi parameter Bayesian. "
    "Karena itu kesesuaian Bayesian dengan Baseline 2 "
    "diharapkan tinggi; yang diuji adalah apakah probabilitas "
    "menambah daya pembeda."
)

Kecocokan model dengan penilaian pakar.

In [ ]:
import os
import math
import numpy as np
import pandas as pd
from IPython.display import display

_ULANG = os.path.exists("lembar_validasi_ulang_terisi.xlsx")
BERKAS_ISI_EVAL = "lembar_validasi_ulang_terisi.xlsx" if _ULANG else "lembar_penilaian_pakar_terisi.xlsx"
BERKAS_KUNCI_EVAL = "kunci_validasi_ulang.xlsx" if _ULANG else "kunci_sampel.xlsx"
N_BOOTSTRAP = 2000
SEED_BOOTSTRAP = 2026

if "Kategori Aturan Pakar" not in df_clean.columns:
    raise NameError("Jalankan Bagian 7A dahulu (kolom 'Kategori Aturan Pakar' belum ada).")

METODE = {"Bayesian": "Kategori Bayes", "Baseline 1": "Kategori Baseline", "Baseline 2": "Kategori Aturan Pakar"}

def wilson_ik(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    pusat = (p + z * z / (2 * n)) / (1 + z * z / n)
    sebar = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return (pusat - sebar, pusat + sebar)

def uji_tanda_eksak(b, c):
    """Uji tanda dua sisi: b = hanya metode 1 benar, c = hanya metode 2 benar."""
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    return float(min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n))

def ukuran_metode(p, m, w=None):
    """p = nilai pakar (0-2), m = nilai metode (0-2), w = bobot populasi (opsional)."""
    p, m = np.asarray(p, dtype=int), np.asarray(m, dtype=int)
    w = np.ones(len(p)) if w is None else np.asarray(w, dtype=float)
    n = len(p)
    k = int((p == m).sum())
    lo, hi = wilson_ik(k, n)
    pos, pred = p >= 1, m >= 1
    def _r(x, y):
        return round(float(x / y), 3) if y > 0 else np.nan
    sens = _r((w * (pos & pred)).sum(), (w * pos).sum())
    spes = _r((w * (~pos & ~pred)).sum(), (w * ~pos).sum())
    ppv = _r((w * (pos & pred)).sum(), (w * pred).sum())
    return {"n": n, "Cocok persis": f"{k} ({k / n * 100:.1f}%)", "IK 95% Wilson": f"{lo * 100:.1f}-{hi * 100:.1f}%",
            "Cocok persis (berbobot)": f"{(w * (p == m)).sum() / w.sum() * 100:.1f}%",
            "Selisih <=1 tingkat": f"{(np.abs(p - m) <= 1).mean() * 100:.1f}%",
            "MAE ordinal": round(float(np.abs(p - m).mean()), 3),
            "Model lebih tinggi": int((m > p).sum()), "Model lebih rendah": int((m < p).sum()),
            "Kappa berbobot (linear)": round(float(kappa_linear(p, m, w)), 3),
            "Sensitivitas (>=Sedang)": sens, "Spesifisitas (Rendah)": spes, "PPV (>=Sedang)": ppv}

HASIL_EVAL_B = {}

print("=== 7B(i). Transaksi pakar T001-T012 ===")
KONFLIK_T = {"T005": "E2k (tanggal kedaluwarsa janggal): wawancara netral vs form Sedang",
             "T011": "E2k (tanggal kedaluwarsa janggal): wawancara netral vs form Sedang",
             "T009": "E2 saja: wawancara Tinggi vs form Sedang",
             "T010": "E1 saja pada obat umum di RS: pakar Rendah, aturan jumlah besar = Sedang",
             "T012": "Cefixime besar ke apotek: wawancara Sedang vs form Tinggi"}
if "tabel_pakar" not in globals() or "indeks_final" not in globals():
    print("Hasil Bagian 4 belum ada; jalankan sel pemetaan T001-T012 terlebih dahulu.")
    tabel_7_6 = pd.DataFrame()
else:
    tp = tabel_pakar[["Kode", "Indeks", "Penilaian pakar"]].copy()
    for nama, kol in METODE.items():
        tp[nama] = [df_clean.at[i, kol] for i in tp["Indeks"]]
    tp["Posterior"] = [df_clean.at[i, "Posterior Bayes"] for i in tp["Indeks"]]
    tp["Catatan konflik elisitasi"] = tp["Kode"].map(KONFLIK_T).fillna("")
    _valid = tp["Penilaian pakar"].isin(URUT7).tolist()
    for nama in METODE:
        tp["Cocok " + nama] = pd.Series([(a == b) if (v and a in URUT7) else None
                                         for a, b, v in zip(tp[nama], tp["Penilaian pakar"], _valid)], index=tp.index, dtype=object)
    belum = [k for k in PAKAR if k not in indeks_final]
    tabel_7_6 = tp
    print("Transaksi terpetakan:", len(tp), "| belum terpetakan:", belum if belum else "tidak ada")
    display(tp.round({"Posterior": 4}))
    ev = tp[tp["Penilaian pakar"].isin(URUT7) & tp["Bayesian"].isin(URUT7)]
    if len(ev):
        rows = []
        for nama in METODE:
            ok_m = ev[nama].isin(URUT7)
            r = ukuran_metode(ev.loc[ok_m, "Penilaian pakar"].map(URUT7), ev.loc[ok_m, nama].map(URUT7))
            rows.append({"Metode": nama, **r})
        tabel_7_7 = pd.DataFrame(rows)
        print(f"\nRingkasan pada {len(ev)} transaksi yang dapat dibandingkan (studi kasus; n kecil, satu pakar):")
        display(tabel_7_7)
        salah = ev[ev["Cocok Bayesian"] == False]
        if len(salah):
            print("Transaksi yang tidak cocok dengan Bayesian:")
            display(salah[["Kode", "Penilaian pakar", "Bayesian", "Baseline 1", "Baseline 2", "Posterior", "Catatan konflik elisitasi"]]
                    .round({"Posterior": 4}))
        HASIL_EVAL_B["7_6_T_kasus"] = tabel_7_6
        HASIL_EVAL_B["7_7_T_ringkas"] = tabel_7_7
        print("CATATAN: dengan n sekecil ini selisih antar metode tidak bermakna secara statistik. Bagian ini dibaca sebagai "
              "studi kasus; ketidakcocokan dijelaskan oleh konflik elisitasi (kolom catatan), bukan oleh kesalahan hitung.")
    else:
        tabel_7_7 = pd.DataFrame()

print("\n=== 7B(ii). Lembar sampel berstrata yang diisi pakar ===")
tabel_7_8, tabel_7_9, tabel_7_10 = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
tabel_7_11, tabel_7_12, tabel_7_13 = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
tabel_7_13b, tabel_7_16, tabel_7_17 = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
data_sampel_pakar = pd.DataFrame()
STALE_PERSEN = np.nan
if not os.path.exists(BERKAS_ISI_EVAL):
    print(f"Berkas {BERKAS_ISI_EVAL} belum ada. Bagian ini aktif otomatis setelah lembar yang diisi pakar disimpan di folder yang sama.")
else:
    KOLOM_N7 = "NILAI ANDA: seberapa perlu diperiksa?"
    def _kat(x):
        t = str(x).strip().casefold()
        for kunci, hasil in [("rendah", "RENDAH"), ("sedang", "SEDANG"), ("tinggi", "TINGGI"), ("tidak bisa", "TIDAK BISA DINILAI")]:
            if t.startswith(kunci):
                return hasil
        return ""
    isi7 = pd.read_excel(BERKAS_ISI_EVAL, sheet_name="Penilaian", dtype=str).fillna("")
    isi7["Nilai pakar"] = isi7[KOLOM_N7].map(_kat)
    kunci7 = (pd.read_excel(BERKAS_KUNCI_EVAL, sheet_name="Kunci") if os.path.exists(BERKAS_KUNCI_EVAL)
              else kunci_sampel.copy())
    print(f"Lembar: {BERKAS_ISI_EVAL} | kunci: {BERKAS_KUNCI_EVAL}")
    KOLOM_A7 = "ALASAN UTAMA"
    KOLOM_Y7 = "Keyakinan Anda"
    for _kk in (KOLOM_A7, KOLOM_Y7):
        if _kk not in isi7.columns:
            isi7[_kk] = ""
    try:
        konf = pd.read_excel(BERKAS_ISI_EVAL, sheet_name="Konfirmasi_Aturan", dtype=str).fillna("")
    except Exception:
        konf = pd.DataFrame()
    if len(konf):
        _peta = {"K1": ("TOGGLE_CEFIXIME_RULE", {"sedang": "WAWANCARA", "tinggi": "FORM_T012"}),
                 "K2": ("TOGGLE_EXPIRY_RULE", {"tinggi": "WAWANCARA", "sedang": "FORM_T009"}),
                 "K3": ("TOGGLE_E2K_RULE", {"tidak berpengaruh": "WAWANCARA", "perlu klarifikasi": "FORM_T005_T011"})}
        _saran = []
        for _, r in konf.iterrows():
            kode = str(r.get("No", "")).strip()
            jawab = str(r.get("Jawaban Anda", "")).strip()
            sar = ""
            if kode in _peta and jawab:
                for kunci_j, nilai in _peta[kode][1].items():
                    if jawab.casefold().startswith(kunci_j):
                        sar = f"{_peta[kode][0]} = {nilai}"
            _saran.append(sar)
        konf["Saran pengaturan saklar"] = _saran
        tabel_7_16 = konf
        print("\nJawaban konfirmasi aturan oleh pakar:")
        display(konf[[c for c in ["No", "Pertanyaan", "Jawaban Anda", "Saran pengaturan saklar"] if c in konf.columns]])
        HASIL_EVAL_B["7_16_Konfirmasi_aturan"] = konf
    g = kunci7.merge(isi7[["No", "Nilai pakar", KOLOM_A7, KOLOM_Y7]], left_on="ID lembar", right_on="No", how="left")
    g["Nilai pakar"] = g["Nilai pakar"].fillna("")
    _kons = []
    for _, r in g[g["Duplikat uji ulang"] == "ya"].iterrows():
        a = g.loc[g["ID lembar"] == r["ID asli (bila duplikat)"], "Nilai pakar"]
        a = a.iloc[0] if len(a) else ""
        if r["Nilai pakar"] in URUT7 and a in URUT7:
            _kons.append({"ID asli": r["ID asli (bila duplikat)"], "Penilaian 1": a, "Penilaian 2": r["Nilai pakar"],
                          "Sama": a == r["Nilai pakar"], "Selisih tingkat": abs(URUT7[a] - URUT7[r["Nilai pakar"]])})
    tabel_7_13 = pd.DataFrame(_kons)
    if len(tabel_7_13):
        print(f"Konsistensi pakar pada {len(tabel_7_13)} transaksi duplikat: sama persis {int(tabel_7_13['Sama'].sum())} "
              f"({tabel_7_13['Sama'].mean() * 100:.1f}%), selisih maksimum {int(tabel_7_13['Selisih tingkat'].max())} tingkat")
        HASIL_EVAL_B["7_13_Konsistensi_pakar"] = tabel_7_13
        _pa = tabel_7_13["Penilaian 1"].map(URUT7).to_numpy()
        _pb = tabel_7_13["Penilaian 2"].map(URUT7).to_numpy()
        _lo, _hi = wilson_ik(int((_pa == _pb).sum()), len(_pa))
        tabel_7_13b = pd.DataFrame([{"Pasangan duplikat": len(_pa), "Sama persis (%)": round(float((_pa == _pb).mean() * 100), 1),
                                     "IK 95% Wilson": f"{_lo * 100:.1f}-{_hi * 100:.1f}%",
                                     "Kappa berbobot (linear)": round(float(kappa_linear(_pa, _pb)), 3),
                                     "Selisih 2 tingkat (pasangan)": int((np.abs(_pa - _pb) == 2).sum())}])
        print("Keterulangan pakar (intra-rater); kecocokan model dengan pakar tidak dapat diharapkan melebihi ini:")
        display(tabel_7_13b)
        HASIL_EVAL_B["7_13b_Keterulangan"] = tabel_7_13b
    else:
        print("Belum ada pasangan duplikat yang terisi lengkap.")
    g = g[(g["Duplikat uji ulang"] != "ya") & g["Nilai pakar"].isin(URUT7)].copy()
    print(f"Transaksi unik dengan penilaian pakar: {len(g)} (di luar 'tidak bisa dinilai' dan duplikat uji ulang)")
    if len(g) < 5:
        print("Terlalu sedikit untuk dianalisis.")
    else:
        idx = g["Indeks df_clean"].to_numpy()
        hilang = [i for i in idx if i not in df_clean.index]
        if hilang:
            raise KeyError(f"Indeks sampel tidak ada di df_clean (apakah df_clean diubah setelah sampel dibuat?): {hilang[:5]}")
        for nama, kol in METODE.items():
            g[nama] = df_clean.loc[idx, kol].to_numpy()
        g = g[g["Bayesian"].isin(URUT7)].copy()
        stale = float((g["Kategori Bayes"].to_numpy() != g["Bayesian"].to_numpy()).mean() * 100)
        STALE_PERSEN = stale
        if stale > 0:
            print(f"PERINGATAN: {stale:.1f}% sampel punya kategori Bayes berbeda dari saat lembar dibuat (aturan berubah setelah sampel). "
                  "Perbandingan memakai kategori TERKINI. Bila perubahan besar, buat sampel ulang setelah aturan final.")
        p = g["Nilai pakar"].map(URUT7).to_numpy()
        w = g["Bobot"].astype(float).to_numpy()
        data_sampel_pakar = pd.DataFrame({"Indeks df_clean": g["Indeks df_clean"].to_numpy(), "p": p, "Bobot": w,
                                          "Strata": g["Strata"].to_numpy()})
        rows = []
        for nama in METODE:
            ok_m = g[nama].isin(URUT7).to_numpy()
            rows.append({"Metode": nama, **ukuran_metode(p[ok_m], g.loc[ok_m, nama].map(URUT7).to_numpy(), w[ok_m])})
        tabel_7_8 = pd.DataFrame(rows)
        print("\nRingkasan kecocokan dengan pakar (bobot = bobot populasi sel strata):")
        display(tabel_7_8)
        sil = {}
        for nama in METODE:
            sil[nama] = pd.crosstab(g["Nilai pakar"], g[nama]).reindex(index=list(URUT7), columns=list(URUT7), fill_value=0)
            print(f"\nTabel silang pakar (baris) x {nama} (kolom):")
            display(sil[nama])
        pas = []
        rng = np.random.default_rng(SEED_BOOTSTRAP)
        bm = {nama: (g[nama].map(URUT7).to_numpy() == p) for nama in METODE}
        for lawan in ["Baseline 1", "Baseline 2"]:
            b = int((bm["Bayesian"] & ~bm[lawan]).sum())
            c = int((~bm["Bayesian"] & bm[lawan]).sum())
            sel = []
            n = len(g)
            for _ in range(N_BOOTSTRAP):
                s = rng.integers(0, n, n)
                ws = w[s]
                sel.append(((ws * bm["Bayesian"][s]).sum() - (ws * bm[lawan][s]).sum()) / ws.sum())
            pas.append({"Pembanding": lawan, "Hanya Bayesian benar": b, "Hanya pembanding benar": c,
                        "Uji tanda eksak (p)": round(uji_tanda_eksak(b, c), 4),
                        "Selisih cocok persis berbobot (Bayesian - pembanding)": round(float(np.mean(sel)) * 100, 2),
                        "IK 95% bootstrap (poin %)": f"{np.percentile(sel, 2.5) * 100:.1f} s.d. {np.percentile(sel, 97.5) * 100:.1f}"})
        tabel_7_9 = pd.DataFrame(pas)
        print("\nPerbandingan berpasangan pada transaksi yang sama:")
        display(tabel_7_9)
        g["Cocok Bayesian"] = bm["Bayesian"].astype(int)
        g["Cocok Baseline 1"] = bm["Baseline 1"].astype(int)
        g["p"] = p
        g["m"] = g["Bayesian"].map(URUT7)
        tabel_7_10 = (g.groupby(["Strata", "Nama strata"]).agg(n=("p", "size"), Cocok_Bayesian=("Cocok Bayesian", "sum"),
                                                              Cocok_Baseline1=("Cocok Baseline 1", "sum"),
                                                              Pakar_rata2=("p", "mean"), Bayesian_rata2=("m", "mean")).round(2).reset_index())
        print("\nMenurut strata:")
        display(tabel_7_10)
        idx = g["Indeks df_clean"].to_numpy()
        tabel_7_11 = pd.crosstab(g[KOLOM_A7].replace("", "(kosong)"), g["Nilai pakar"])
        print("\nAlasan utama pakar menurut penilaian pakar:")
        display(tabel_7_11)
        g["Selisih (Bayesian - pakar)"] = g["Bayesian"].map(URUT7) - p
        _f = df_clean.loc[idx]
        for _k in ["E1", "E2", "E2p", "E3", "E3b"]:
            if _k in X.columns:
                g[_k] = X.loc[idx, _k].to_numpy()
        for _k in ["Kategori Obat", "Jenis Sarana", "Jumlah Setelah Konversi", "Alasan E3", "Catatan Pendukung"]:
            if _k in _f.columns:
                g[_k] = _f[_k].to_numpy()
        g["Posterior"] = df_clean.loc[idx, "Posterior Bayes"].to_numpy()
        _kol = [c for c in ["ID lembar", "Strata", "Nilai pakar", "Bayesian", "Posterior", "E1", "E2", "E2p", "E3", "E3b", "Kategori Obat",
                            "Jenis Sarana", "Jumlah Setelah Konversi", "Alasan E3", KOLOM_A7] if c in g.columns]
        tabel_7_12 = g.reindex(g["Selisih (Bayesian - pakar)"].abs().sort_values(ascending=False, kind="stable").index)
        tabel_7_12 = tabel_7_12[tabel_7_12["Selisih (Bayesian - pakar)"] != 0][_kol].head(40)
        print("\nTransaksi dengan selisih terbesar antara Bayesian dan pakar (maks 40):")
        display(tabel_7_12.round({"Posterior": 4}))
        HASIL_EVAL_B["7_11_Alasan_pakar"] = tabel_7_11.reset_index()
        HASIL_EVAL_B["7_12_Selisih_terbesar"] = tabel_7_12
        if (g[KOLOM_Y7].astype(str).str.casefold().str.startswith("yakin")).any():
            gy = g[g[KOLOM_Y7].astype(str).str.casefold().str.startswith("yakin")]
            _ry = []
            for nama in METODE:
                ok_m = gy[nama].isin(URUT7).to_numpy()
                _ry.append({"Metode": nama, **ukuran_metode(gy.loc[ok_m, "Nilai pakar"].map(URUT7).to_numpy(),
                                                            gy.loc[ok_m, nama].map(URUT7).to_numpy(), gy.loc[ok_m, "Bobot"].astype(float).to_numpy())})
            tabel_7_17 = pd.DataFrame(_ry)
            print("\nHanya penilaian yang ditandai pakar 'Yakin':")
            display(tabel_7_17)
            HASIL_EVAL_B["7_17_Hanya_yakin"] = tabel_7_17
        HASIL_EVAL_B["7_8_Sampel_ringkas"] = tabel_7_8
        HASIL_EVAL_B["7_9_Sampel_berpasangan"] = tabel_7_9
        HASIL_EVAL_B["7_10_Sampel_strata"] = tabel_7_10
        for nama, t in sil.items():
            HASIL_EVAL_B["7_8_Silang_" + nama.replace(" ", "")] = t.reset_index()
        print("\nCATATAN: IK Wilson dan bootstrap mengasumsikan penilaian independen; sampel dinilai satu pakar, sehingga hasil adalah "
              "kesesuaian dengan penilaian pakar tersebut, bukan akurasi terhadap kebenaran mutlak.")

try:
    with pd.ExcelWriter(BERKAS_EVAL, engine="openpyxl") as penulis:
        for nama, t in {**HASIL_EVAL, **HASIL_EVAL_B}.items():
            t.to_excel(penulis, sheet_name=nama[:31], index=False)
    print(f"\nDisimpan: {BERKAS_EVAL} ({len(HASIL_EVAL) + len(HASIL_EVAL_B)} lembar)")
    try:
        from google.colab import files as _f
        _f.download(BERKAS_EVAL)
    except Exception:
        pass
except Exception as galat:
    print(f"Ekspor Excel gagal: {galat}")

Sensitivitas OAT terhadap kecocokan dengan pakar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

GAMBAR_7_2 = "gambar_7_2_oat_kecocokan_pakar.png"

for _n in ["KAT_SKENARIO", "INVERS", "tabel_oat_semua", "df_clean", "kappa_linear", "ukuran_metode", "URUT7"]:
    if _n not in globals():
        raise NameError(f"{_n} belum ada. Jalankan Bagian 3 (OAT), 7A, dan 7B dahulu.")

def _kategori_baris(kat_pola, indeks):
    pos = df_clean.index.get_indexer(indeks)
    return np.asarray(kat_pola, dtype=object)[INVERS[pos]]

def _ukur(kat, p, w):
    ok = np.array([k in URUT7 for k in kat])
    if ok.sum() == 0:
        return None
    m = np.array([URUT7[k] for k in kat[ok]])
    r = ukuran_metode(p[ok], m, w[ok])
    return {"n": int(ok.sum()),
            "Cocok persis (%)": round(float((m == p[ok]).mean() * 100), 1),
            "Cocok persis berbobot (%)": round(float((w[ok] * (m == p[ok])).sum() / w[ok].sum() * 100), 1),
            "Kappa berbobot": r["Kappa berbobot (linear)"],
            "Sensitivitas (>=Sedang)": r["Sensitivitas (>=Sedang)"], "Spesifisitas": r["Spesifisitas (Rendah)"],
            "Model lebih tinggi": r["Model lebih tinggi"], "Model lebih rendah": r["Model lebih rendah"]}

sumber = {}
if "data_sampel_pakar" in globals() and len(data_sampel_pakar):
    sumber["Sampel pakar"] = (data_sampel_pakar["Indeks df_clean"].to_numpy(), data_sampel_pakar["p"].to_numpy(dtype=int),
                              data_sampel_pakar["Bobot"].to_numpy(dtype=float))
if "tabel_7_6" in globals() and len(tabel_7_6):
    _t = tabel_7_6[tabel_7_6["Penilaian pakar"].isin(URUT7)]
    if len(_t):
        sumber["T001-T012"] = (_t["Indeks"].to_numpy(), _t["Penilaian pakar"].map(URUT7).to_numpy(dtype=int), np.ones(len(_t)))
if not sumber:
    print("Belum ada data pakar (sampel terisi atau T-kasus). Bagian ini dilewati.")
    tabel_7_14 = pd.DataFrame()
else:
    skenario_valid = tabel_oat_semua[tabel_oat_semua["Keterangan"].fillna("") == ""]
    baris = []
    for nama_sumber, (idx, p, w) in sumber.items():
        for _, r in skenario_valid.iterrows():
            kat_pola = KAT_SKENARIO.get((r["Kode"], r["Skema"]))
            if kat_pola is None:
                continue
            u = _ukur(_kategori_baris(kat_pola, idx), p, w)
            if u:
                baris.append({"Data pakar": nama_sumber, "Skema": r["Skema"], "Kode": r["Kode"], "Skenario": r["Skenario"],
                              "Jenis": r["Jenis"], **u})
    tabel_7_14 = pd.DataFrame(baris)
    ref = tabel_7_14[tabel_7_14["Kode"] == "S0"].set_index(["Data pakar", "Skema"])
    for kol in ["Cocok persis (%)", "Kappa berbobot"]:
        tabel_7_14["Selisih " + kol + " vs S0"] = [
            round(float(r[kol] - ref.loc[(r["Data pakar"], r["Skema"]), kol]), 3) if (r["Data pakar"], r["Skema"]) in ref.index else np.nan
            for _, r in tabel_7_14.iterrows()]
    pemb = []
    for nama_sumber, (idx, p, w) in sumber.items():
        for nama, kol in [("Baseline 1", "Kategori Baseline"), ("Baseline 2", "Kategori Aturan Pakar")]:
            u = _ukur(df_clean.loc[idx, kol].to_numpy(dtype=object), p, w)
            if u:
                pemb.append({"Data pakar": nama_sumber, "Skema": "-", "Kode": nama, "Skenario": nama, "Jenis": "Baseline", **u})
    tabel_7_14b = pd.DataFrame(pemb)

    for nama_sumber in sumber:
        d = tabel_7_14[tabel_7_14["Data pakar"] == nama_sumber]
        print(f"\n=== 7.14 OAT terhadap kecocokan dengan pakar: {nama_sumber} (n = {d['n'].iloc[0]}) ===")
        for skema in d["Skema"].unique():
            print(f"\nSkema {skema}")
            kol_t = ["Kode", "Skenario", "Jenis", "Cocok persis (%)", "Cocok persis berbobot (%)", "Kappa berbobot",
                     "Sensitivitas (>=Sedang)", "Spesifisitas", "Model lebih tinggi", "Model lebih rendah",
                     "Selisih Cocok persis (%) vs S0"]
            display(d[d["Skema"] == skema][kol_t])
        print("Pembanding baseline:")
        display(tabel_7_14b[tabel_7_14b["Data pakar"] == nama_sumber][["Skenario", "Cocok persis (%)", "Cocok persis berbobot (%)",
                                                                     "Kappa berbobot", "Sensitivitas (>=Sedang)", "Spesifisitas"]])

    ring = []
    for (ds, sk), d in tabel_7_14.groupby(["Data pakar", "Skema"]):
        oat = d[d["Jenis"] == "OAT"]
        s0 = d[d["Kode"] == "S0"].iloc[0]
        terbaik = oat.sort_values(["Kappa berbobot", "Cocok persis (%)"], ascending=False).iloc[0]
        ring.append({"Data pakar": ds, "Skema": sk, "n": int(s0["n"]),
                     "Cocok persis S0 (%)": s0["Cocok persis (%)"],
                     "Rentang OAT (%)": f"{oat['Cocok persis (%)'].min()} - {oat['Cocok persis (%)'].max()}",
                     "Rentang kappa OAT": f"{oat['Kappa berbobot'].min()} - {oat['Kappa berbobot'].max()}",
                     "Skenario OAT paling cocok": terbaik["Skenario"], "Cocok persis skenario itu (%)": terbaik["Cocok persis (%)"],
                     "Skenario OAT tanpa perubahan kecocokan": int((oat["Selisih Cocok persis (%) vs S0"].abs() < 1e-9).sum()),
                     "Jumlah skenario OAT": len(oat)})
    tabel_7_15 = pd.DataFrame(ring)
    print("\n=== 7.15 Ringkasan: seberapa peka kecocokan dengan pakar terhadap parameter ===")
    display(tabel_7_15)
    print("Baca: bila rentang OAT sempit dan jauh di bawah yang diharapkan, ketidakcocokan dengan pakar BUKAN akibat pilihan nilai "
          "prior/LR/ambang, melainkan struktur aturan (indikator dan definisinya) atau cara pakar menilai.")

    ds_gambar = "Sampel pakar" if "Sampel pakar" in sumber else list(sumber)[0]
    sk_gambar = SKEMA_UTAMA if SKEMA_UTAMA in tabel_7_14["Skema"].unique() else tabel_7_14["Skema"].iloc[0]
    d = tabel_7_14[(tabel_7_14["Data pakar"] == ds_gambar) & (tabel_7_14["Skema"] == sk_gambar)].copy()
    d = d.sort_values("Cocok persis (%)")
    fig, ax = plt.subplots(figsize=(8.5, 0.34 * len(d) + 1.6))
    warna = ["#d9534f" if k == "S0" else "#7a9cc6" for k in d["Kode"]]
    ax.barh(d["Kode"] + "  " + d["Skenario"].str.slice(0, 38), d["Cocok persis (%)"], color=warna)
    for _, rb in tabel_7_14b[tabel_7_14b["Data pakar"] == ds_gambar].iterrows():
        ax.axvline(rb["Cocok persis (%)"], ls="--", lw=1, color="#555555")
        ax.text(rb["Cocok persis (%)"], len(d) - 0.4, rb["Skenario"], rotation=90, va="top", ha="right", fontsize=7, color="#555555")
    ax.set_xlabel(f"Cocok persis dengan pakar (%), {ds_gambar}, skema {sk_gambar}")
    ax.set_title("Sensitivitas OAT terhadap kecocokan dengan pakar (merah = referensi S0)", fontsize=10)
    plt.tight_layout()
    try:
        plt.savefig(GAMBAR_7_2, dpi=200, bbox_inches="tight")
        print(f"Gambar disimpan: {GAMBAR_7_2}")
    except Exception as galat:
        print(f"Gambar tidak tersimpan: {galat}")
    plt.show()
    try:
        with pd.ExcelWriter("hasil_evaluasi_akhir_oat_pakar.xlsx", engine="openpyxl") as penulis:
            tabel_7_14.to_excel(penulis, sheet_name="7_14_OAT_vs_pakar", index=False)
            tabel_7_14b.to_excel(penulis, sheet_name="7_14b_Baseline", index=False)
            tabel_7_15.to_excel(penulis, sheet_name="7_15_Ringkasan", index=False)
        print("Disimpan: hasil_evaluasi_akhir_oat_pakar.xlsx")
    except Exception as galat:
        print(f"Ekspor Excel gagal: {galat}")

**PENCOCOKAN DENGAN VALIDASI PAKAR**

Pencocokan jawaban konfirmasi aturan pakar dengan model.

In [ ]:
import os
import numpy as np
import pandas as pd
from IPython.display import display

BERKAS_ISI_7D = "lembar_validasi_ulang_terisi.xlsx"
BERKAS_HASIL_7D = "pencocokan_konfirmasi_aturan.xlsx"
for _n in ["df_clean", "X", "kategori_obat", "sarana_cf", "bool_np"]:
    if _n not in globals():
        raise NameError(f"{_n} belum ada. Jalankan Bagian 3 (model) dahulu.")
if "Kategori Bayes" not in df_clean.columns:
    raise NameError("Kolom 'Kategori Bayes' belum ada di df_clean. Jalankan Bagian 3 dahulu.")

if "tabel_7_16" in globals() and len(tabel_7_16):
    konf7d = tabel_7_16.copy()
elif os.path.exists(BERKAS_ISI_7D):
    konf7d = pd.read_excel(BERKAS_ISI_7D, sheet_name="Konfirmasi_Aturan", dtype=str).fillna("")
else:
    konf7d = pd.DataFrame()
    print(f"Belum ada jawaban konfirmasi ({BERKAS_ISI_7D} tidak ditemukan). Bagian 7D dilewati.")

tabel_7d = pd.DataFrame()
if len(konf7d):
    kat = df_clean["Kategori Bayes"].astype(str).str.upper().to_numpy()
    obat = kategori_obat.astype("string")
    ob = lambda *nama: bool_np(obat.isin(list(nama)))
    rs = bool_np(sarana_cf.eq("rumah sakit"))
    ap_kl = bool_np(sarana_cf.isin(["apotek", "klinik"]))
    satu = lambda k: bool_np(X[k].eq(1))
    e1, e3, e3b = satu("E1"), satu("E3"), satu("E3b")
    e2 = satu("E2") | (satu("E2p") if "E2p" in X.columns else np.zeros(len(df_clean), dtype=bool))
    e2p = satu("E2p") if "E2p" in X.columns else np.zeros(len(df_clean), dtype=bool)
    e2k_aktif = bool_np(df_clean["E2k_rev"].eq(1)) if "E2k_rev" in df_clean.columns else np.zeros(len(df_clean), dtype=bool)
    anomali = bool_np(df_clean["Temuan data anomali"]) if "Temuan data anomali" in df_clean.columns else np.zeros(len(df_clean), dtype=bool)
    biasa = ~ob("Ketamine", "Meropenem")
    e1_mentah = bool_np(df_clean["E1_rev"].eq(1))
    e2_mentah = bool_np(df_clean["E2_rev"].eq(1))
    SKENARIO = {
        "K1": ob("Cefixime") & ap_kl & e3b & ~e2 & ~e3,
        "K2": biasa & ~rs & ~ap_kl & e2 & ~e1 & ~e3 & ~e3b,
        "K2a": biasa & rs & e2 & ~e1_mentah & ~e3 & ~e3b,
        "K2b": biasa & ap_kl & e2p & ~e1 & ~e3 & ~e3b,
        "K3": e2k_aktif & ~e1 & ~e2 & ~e3 & ~e3b,
        "K4": ob("Cefixime", "TBC") & rs & e1_mentah & ~e2_mentah & ~e3 & ~e3b,
        "K5": ob("Ketamine") & rs & e1 & ~e2 & ~e3 & ~e3b,
        "K6": ob("Ketamine") & rs & e1 & e2 & ~e3 & ~e3b,
    }
    def _peta_jawaban(teks):
        t = str(teks).strip().casefold()
        for kunci, hasil in [("tidak berpengaruh", "RENDAH"), ("perlu klarifikasi", "SEDANG"), ("rendah", "RENDAH"),
                             ("aman", "RENDAH"), ("sedang", "SEDANG"), ("tinggi", "TINGGI")]:
            if t.startswith(kunci):
                return hasil
        return ""
    SARAN = {
        "K1": "TOGGLE_CEFIXIME_RULE: WAWANCARA = Sedang (E3b), FORM_T012 = Tinggi (E3)",
        "K2": "TOGGLE_EXPIRY_RULE dan LR E2 (tujuan selain apotek/klinik)",
        "K2a": "E2_RS_BIASA_AMAN: False = Sedang di RS (sesuai konfirmasi pakar)",
        "K2b": "E2_SPLIT_SARANA dan LR E2p (apotek/klinik)",
        "K3": "TOGGLE_E2K_RULE: WAWANCARA = netral, FORM_T005_T011 = Sedang",
        "K4": "E1_RS_BIASA_AMAN: True = Rendah di RS, False = Sedang lewat E1",
        "K5": "OBAT_RS_DIKAJI dan E3 (Ketamine ke RS): saat ini hanya E1 (Sedang)",
        "K6": "OBAT_RS_DIKAJI, LR E1 dan LR E2 (E1 + E2 = Tinggi)",
    }
    baris = []
    for _, r in konf7d.iterrows():
        kode = str(r.get("No", "")).strip()
        if kode not in SKENARIO:
            continue
        jawab = str(r.get("Jawaban Anda", "")).strip()
        pakar = _peta_jawaban(jawab)
        m = SKENARIO[kode] & ~anomali
        n = int(m.sum())
        if n:
            hit = pd.Series(kat[m]).value_counts()
            modus = hit.index[0]
            persen = float((kat[m] == pakar).mean() * 100) if pakar else np.nan
            sebaran = "; ".join(f"{k} {v:,}" for k, v in hit.items())
        else:
            modus, persen, sebaran = "", np.nan, ""
        if not jawab:
            status = "Belum dijawab"
        elif not n:
            status = "Tidak ada transaksi pada skenario ini"
        elif not pakar:
            status = "Jawaban tidak terbaca"
        elif modus == pakar:
            status = "Sesuai"
        else:
            status = "TIDAK SESUAI"
        baris.append({"No": kode, "Pertanyaan": str(r.get("Pertanyaan", ""))[:110], "Jawaban pakar": jawab,
                      "Kategori pakar": pakar, "Transaksi pada skenario": n, "Kategori model (terbanyak)": modus,
                      "% baris sesuai pakar": round(persen, 1) if persen == persen else np.nan,
                      "Sebaran kategori model": sebaran, "Status": status,
                      "Perlu dicek di": SARAN[kode] if status in ("TIDAK SESUAI", "Tidak ada transaksi pada skenario ini") else ""})
    tabel_7d = pd.DataFrame(baris)
    print("Pencocokan jawaban konfirmasi aturan pakar dengan kategori model (konfigurasi saat ini)")
    display(tabel_7d)
    if len(tabel_7d):
        n_ok = int((tabel_7d["Status"] == "Sesuai").sum())
        n_nilai = int(tabel_7d["Status"].isin(["Sesuai", "TIDAK SESUAI"]).sum())
        print(f"\nAturan yang sesuai: {n_ok} dari {n_nilai} yang bisa dinilai.")
        for _, r in tabel_7d[tabel_7d["Status"] == "TIDAK SESUAI"].iterrows():
            print(f"- {r['No']}: pakar {r['Kategori pakar']}, model {r['Kategori model (terbanyak)']} -> cek {r['Perlu dicek di']}")
        try:
            tabel_7d.to_excel(BERKAS_HASIL_7D, index=False)
            print(f"Disimpan: {BERKAS_HASIL_7D}")
        except Exception as galat:
            print("Gagal menyimpan berkas 7D:", galat)
        if "HASIL_EVAL_B" in globals():
            HASIL_EVAL_B["7_18_Pencocokan_konfirmasi"] = tabel_7d

Ringkasan dua validasi pakar (V1 dan V2).

In [ ]:
import os
import numpy as np
import pandas as pd
from IPython.display import display

VALIDASI_7E = [
    ("Validasi 1", "lembar_penilaian_pakar_terisi.xlsx", "kunci_sampel.xlsx"),
    ("Validasi 2", "lembar_validasi_ulang_terisi.xlsx", "kunci_validasi_ulang.xlsx"),
]
BERKAS_HASIL_7E = "ringkasan_dua_validasi.xlsx"
KOLOM_NILAI_7E = "NILAI ANDA: seberapa perlu diperiksa?"
for _n in ["df_clean", "wilson_ik", "ukuran_metode", "URUT7", "METODE"]:
    if _n not in globals():
        raise NameError(f"{_n} belum ada. Jalankan Bagian 3 dan 7B dahulu.")

def _kat_7e(x):
    t = str(x).strip().casefold()
    for kunci, hasil in [("rendah", "RENDAH"), ("sedang", "SEDANG"), ("tinggi", "TINGGI")]:
        if t.startswith(kunci):
            return hasil
    return ""

baris_7e, baris_ulang_7e = [], []
for nama_v, berkas_isi, berkas_kunci in VALIDASI_7E:
    if not (os.path.exists(berkas_isi) and os.path.exists(berkas_kunci)):
        print(f"{nama_v}: dilewati ({berkas_isi} atau {berkas_kunci} tidak ditemukan).")
        continue
    isi = pd.read_excel(berkas_isi, sheet_name="Penilaian", dtype=str).fillna("")
    kunci = pd.read_excel(berkas_kunci, sheet_name="Kunci")
    isi["Nilai pakar"] = isi[KOLOM_NILAI_7E].map(_kat_7e) if KOLOM_NILAI_7E in isi.columns else ""
    g = kunci.merge(isi[["No", "Nilai pakar"]], left_on="ID lembar", right_on="No", how="left")
    g["Nilai pakar"] = g["Nilai pakar"].fillna("")
    pa, pb = [], []
    for _, r in g[g["Duplikat uji ulang"] == "ya"].iterrows():
        a = g.loc[g["ID lembar"] == r["ID asli (bila duplikat)"], "Nilai pakar"]
        a = a.iloc[0] if len(a) else ""
        if r["Nilai pakar"] in URUT7 and a in URUT7:
            pa.append(URUT7[a]); pb.append(URUT7[r["Nilai pakar"]])
    if pa:
        pa, pb = np.array(pa), np.array(pb)
        lo, hi = wilson_ik(int((pa == pb).sum()), len(pa))
        baris_ulang_7e.append({"Validasi": nama_v, "Pasangan duplikat": len(pa),
                               "Sama persis (%)": round(float((pa == pb).mean() * 100), 1),
                               "IK 95% Wilson": f"{lo * 100:.1f}-{hi * 100:.1f}%",
                               "Kappa berbobot (linear)": round(float(kappa_linear(pa, pb)), 3) if "kappa_linear" in globals() else np.nan})
    g = g[(g["Duplikat uji ulang"] != "ya") & g["Nilai pakar"].isin(URUT7)].copy()
    ada = g["Indeks df_clean"].isin(df_clean.index)
    if (~ada).any():
        print(f"{nama_v}: {int((~ada).sum())} indeks sampel tidak ada di df_clean dan dilewati.")
    g = g[ada]
    if len(g) < 5:
        print(f"{nama_v}: terlalu sedikit penilaian ({len(g)}).")
        continue
    idx = g["Indeks df_clean"].to_numpy()
    p = g["Nilai pakar"].map(URUT7).to_numpy()
    w = g["Bobot"].astype(float).to_numpy() if "Bobot" in g.columns else np.ones(len(g))
    berubah = float((g["Kategori Bayes"].astype(str).to_numpy() != df_clean.loc[idx, "Kategori Bayes"].astype(str).to_numpy()).mean() * 100)
    for nama_m, kol in METODE.items():
        kat = df_clean.loc[idx, kol].astype(str).to_numpy()
        ok = np.array([k in URUT7 for k in kat])
        if ok.sum() < 5:
            continue
        m = np.array([URUT7[k] for k in kat[ok]])
        r = ukuran_metode(p[ok], m, w[ok])
        k_cocok = int((m == p[ok]).sum())
        lo, hi = wilson_ik(k_cocok, int(ok.sum()))
        baris_7e.append({"Validasi": nama_v, "Metode": nama_m, "n": int(ok.sum()),
                         "Cocok persis": f"{k_cocok} ({k_cocok / ok.sum() * 100:.1f}%)",
                         "IK 95% Wilson": f"{lo * 100:.1f}-{hi * 100:.1f}%",
                         "Kappa berbobot (linear)": r["Kappa berbobot (linear)"],
                         "Sensitivitas (>=Sedang)": r["Sensitivitas (>=Sedang)"],
                         "Spesifisitas": r["Spesifisitas (Rendah)"],
                         "Model lebih tinggi": r["Model lebih tinggi"], "Model lebih rendah": r["Model lebih rendah"],
                         "% sampel berubah kategori sejak lembar dibuat": round(berubah, 1)})

tabel_7e = pd.DataFrame(baris_7e)
tabel_7e_ulang = pd.DataFrame(baris_ulang_7e)
if len(tabel_7e):
    print("Kecocokan dengan pakar pada setiap validasi (kategori model TERKINI)")
    display(tabel_7e)
    if len(tabel_7e_ulang):
        print("Keterulangan pakar pada pasangan duplikat")
        display(tabel_7e_ulang)
    print("Catatan: validasi 1 dan 2 memakai sampel, formulir, dan aturan model yang berbeda; jangan digabung menjadi satu angka.")
    try:
        with pd.ExcelWriter(BERKAS_HASIL_7E, engine="openpyxl") as _pen:
            tabel_7e.to_excel(_pen, sheet_name="Kecocokan", index=False)
            tabel_7e_ulang.to_excel(_pen, sheet_name="Keterulangan", index=False)
        print(f"Disimpan: {BERKAS_HASIL_7E}")
    except Exception as galat:
        print("Gagal menyimpan 7E:", galat)
    if "HASIL_EVAL_B" in globals():
        HASIL_EVAL_B["7_19_Dua_validasi"] = tabel_7e
else:
    print("Belum ada lembar validasi yang dapat dibaca.")

Konsistensi pakar: penilaian per transaksi dibandingkan jawaban aturan.

In [ ]:
import os
import numpy as np
import pandas as pd
from IPython.display import display

BERKAS_ISI_7F = "lembar_validasi_ulang_terisi.xlsx"
BERKAS_KUNCI_7F = "kunci_validasi_ulang.xlsx"
BERKAS_HASIL_7F = "konsistensi_pakar_aturan.xlsx"
KOLOM_NILAI_7F = "NILAI ANDA: seberapa perlu diperiksa?"
tabel_7f = pd.DataFrame()
if "SKENARIO" not in globals() or "konf7d" not in globals() or not len(konf7d):
    print("Jalankan Bagian 7D lebih dahulu (butuh jawaban konfirmasi aturan). Bagian 7F dilewati.")
elif not (os.path.exists(BERKAS_ISI_7F) and os.path.exists(BERKAS_KUNCI_7F)):
    print(f"{BERKAS_ISI_7F} atau {BERKAS_KUNCI_7F} tidak ditemukan. Bagian 7F dilewati.")
else:
    _isi = pd.read_excel(BERKAS_ISI_7F, sheet_name="Penilaian", dtype=str).fillna("")
    _isi["Nilai pakar"] = _isi[KOLOM_NILAI_7F].map(_kat_7e) if "_kat_7e" in globals() else ""
    _kunci = pd.read_excel(BERKAS_KUNCI_7F, sheet_name="Kunci")
    _g = _kunci.merge(_isi[["No", "Nilai pakar"]], left_on="ID lembar", right_on="No", how="left")
    _g = _g[(_g["Duplikat uji ulang"] != "ya") & _g["Nilai pakar"].isin(["RENDAH", "SEDANG", "TINGGI"])]
    _g = _g[_g["Indeks df_clean"].isin(df_clean.index)]
    _pos = df_clean.index.get_indexer(_g["Indeks df_clean"].to_numpy())
    _nilai = _g["Nilai pakar"].to_numpy()
    _jawab = {str(r["No"]).strip(): _peta_jawaban(r.get("Jawaban Anda", "")) for _, r in konf7d.iterrows()}
    _semua = SKENARIO.copy()
    _X1 = np.column_stack([bool_np(X[k].eq(1)) for k in X.columns])
    _X0 = np.column_stack([bool_np(X[k].eq(0)) for k in X.columns])
    _semua["Tanpa indikator aktif"] = (~_X1.any(axis=1)) & _X0.any(axis=1)
    _jawab["Tanpa indikator aktif"] = "RENDAH"
    baris = []
    for kode, mask in _semua.items():
        sel = np.asarray(mask)[_pos]
        n = int(sel.sum())
        if not n:
            continue
        nilai = _nilai[sel]
        ans = _jawab.get(kode, "")
        baris.append({"Aturan": kode, "Jawaban pakar pada aturan": ans, "Transaksi sampel": n,
                      "Penilaian per transaksi: Rendah": int((nilai == "RENDAH").sum()),
                      "Sedang": int((nilai == "SEDANG").sum()), "Tinggi": int((nilai == "TINGGI").sum()),
                      "% sama dengan jawaban aturan": round(float((nilai == ans).mean() * 100), 1) if ans else np.nan,
                      "Kategori model": ", ".join(sorted(set(df_clean["Kategori Bayes"].astype(str).to_numpy()[_g["Indeks df_clean"].map(df_clean.index.get_loc).to_numpy()[sel]])))})
    tabel_7f = pd.DataFrame(baris)
    if len(tabel_7f):
        print("Penilaian per transaksi oleh pakar dibanding jawabannya sendiri pada aturan yang sama")
        display(tabel_7f)
        _tot = int(tabel_7f["Transaksi sampel"].sum())
        _ok = float((tabel_7f["% sama dengan jawaban aturan"].fillna(0) * tabel_7f["Transaksi sampel"]).sum() / _tot) if _tot else np.nan
        print(f"\nTransaksi sampel yang masuk skenario aturan: {_tot}; penilaian per transaksi sama dengan jawaban aturan pakar: {_ok:.1f}%")
        print("Bila angka ini rendah, ketidakcocokan model dengan pakar sebagian besar berasal dari ketidakkonsistenan penilaian pakar, bukan dari model.")
        try:
            tabel_7f.to_excel(BERKAS_HASIL_7F, index=False)
            print(f"Disimpan: {BERKAS_HASIL_7F}")
        except Exception as galat:
            print("Gagal menyimpan 7F:", galat)
        if "HASIL_EVAL_B" in globals():
            HASIL_EVAL_B["7_20_Konsistensi_pakar_aturan"] = tabel_7f
    else:
        print("Tidak ada transaksi sampel pada skenario aturan.")

Ringkasan angka utama (jumlah transaksi, kategori, validasi) untuk pemeriksaan cepat.

In [ ]:
import numpy as np
import pandas as pd

_KELUAR = []

def _cetak(judul, isi=None, maks=70):
    teks = f"\n=== {judul} ==="
    if isi is not None:
        if isinstance(isi, pd.DataFrame):
            isi = isi.head(maks).round(4).to_string()
        elif isinstance(isi, pd.Series):
            isi = isi.head(maks).to_string()
        teks += "\n" + str(isi)
    print(teks)
    _KELUAR.append(teks)

def _tampil(nama, judul, maks=70):
    """Cetak variabel notebook (DataFrame atau dict DataFrame) bila ada."""
    if nama not in globals():
        _cetak(judul, f"(variabel {nama} tidak ada)")
        return
    obj = globals()[nama]
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, pd.DataFrame):
                _cetak(f"{judul} [{k}]", v, maks)
    elif isinstance(obj, pd.DataFrame):
        _cetak(judul, obj, maks)
    else:
        _cetak(judul, obj)

_perlu = ["df_clean", "X", "INDIKATOR", "POLA_UNIK", "JUMLAH_POLA", "POST_REF", "KAT_REF", "FREK", "LR_UTAMA", "PRIOR_UTAMA"]
_hilang = [v for v in _perlu if v not in globals()]
if _hilang:
    raise NameError(f"Jalankan dulu semua bagian di atas; variabel belum ada: {_hilang}")

_n = len(df_clean)
_cetak("0. Ringkasan", f"baris={_n:,} | prior={PRIOR_UTAMA} | LR={LR_UTAMA} | indikator={INDIKATOR} | "
                       f"E3_KUAT={sorted(globals().get('E3_KUAT', []))} | skema utama={globals().get('SKEMA_UTAMA')}"
       f" | saklar: Cefixime={globals().get('TOGGLE_CEFIXIME_RULE')}, kedaluwarsa={globals().get('TOGGLE_EXPIRY_RULE')}, "
       f"E2k={globals().get('TOGGLE_E2K_RULE')} | E1 PBF Rendah={globals().get('E1_PBF_RENDAH')} | v3 RS biasa aman: E1={globals().get('E1_RS_BIASA_AMAN')}, E2={globals().get('E2_RS_BIASA_AMAN')} | E2 dipecah sarana={globals().get('E2_SPLIT_SARANA')} | E1 vial per obat={globals().get('E1_VIAL_PER_OBAT')}")

_cetak("A1. Kategori Obat", df_clean["Kategori Obat"].value_counts(dropna=False).head(30))
_cetak("A2. Jenis Sarana", df_clean["Jenis Sarana"].value_counts(dropna=False).head(30))
if "Kategori Satuan" in df_clean.columns:
    _cetak("A3. Kategori Satuan", df_clean["Kategori Satuan"].value_counts(dropna=False))
if "Tanggal Penyaluran" in df_clean.columns:
    _th = pd.to_datetime(df_clean["Tanggal Penyaluran"], errors="coerce").dt.year
    _cetak("A4. Tahun penyaluran", _th.value_counts(dropna=False).sort_index())
_kosong = (df_clean.isna().mean() * 100).round(2)
_cetak("A5. Kolom dengan nilai kosong > 0 (%)", _kosong[_kosong > 0].sort_values(ascending=False))
if "Jumlah Setelah Konversi" in df_clean.columns:
    _cetak("A6. Jumlah Setelah Konversi (ringkasan)", pd.to_numeric(df_clean["Jumlah Setelah Konversi"], errors="coerce").describe())

_baris = []
for _k in X.columns:
    _m = FREK.get(_k, np.nan)
    _maks = np.inf if (not _m or _m != _m or _m <= PRIOR_UTAMA) else (1 - PRIOR_UTAMA) / (_m - PRIOR_UTAMA)
    _baris.append({"Indikator": _k, "Dinilai": int(X[_k].notna().sum()), "Bernilai 1": int((X[_k] == 1).fillna(False).sum()),
                   "NA": int(X[_k].isna().sum()), "m": round(float(_m), 5) if _m == _m else np.nan,
                   "LR dipakai": LR_UTAMA.get(_k), "LR maks layak": round(_maks, 2) if np.isfinite(_maks) else "tak terbatas"})
_cetak("B1. Frekuensi indikator dan kelayakan LR", pd.DataFrame(_baris))
if "Status E2_rev" in df_clean.columns:
    _cetak("B1b. Status E2 (masa simpan)", df_clean["Status E2_rev"].value_counts(dropna=False))
_tampil("batas_e1", "B2. Batas IQR E1 per Kategori Satuan")
_tampil("batas_e3", "B3. Batas IQR E3 per Kategori Obat x Satuan")
if "Catatan Pendukung" in df_clean.columns:
    _cp = df_clean["Catatan Pendukung"].astype("string")
    _cetak("B4. Catatan pendukung", pd.Series({"Faktur bermasalah": int(_cp.str.contains("Faktur bermasalah", na=False).sum()),
                                               "Lokasi bermasalah/tidak terbaca": int(_cp.str.contains("Lokasi", na=False).sum()),
                                               "Kedaluwarsa <= penyaluran": int(_cp.str.contains("Kedaluwarsa <=", na=False).sum()),
                                               "Tanggal kedaluwarsa kosong": int(_cp.str.contains("kedaluwarsa kosong", na=False).sum()),
                                               "Cefixime/Orlistat besar ke RS": int(_cp.str.contains("Cefixime/Orlistat", na=False).sum()),
                                               "Ketamin besar ke RS": int(_cp.str.contains("Ketamin jumlah besar ke RS", na=False).sum())}))
for _kol in ["E5 kosong", "E5 kota-provinsi tidak cocok", "E5 lokasi bermasalah (final)"]:
    if _kol in df_clean.columns:
        _cetak(f"B5. {_kol}", f"{int(df_clean[_kol].fillna(0).sum()):,} baris")

_tabel = pd.DataFrame({"Pola (" + "/".join(INDIKATOR) + ")": [" / ".join("NA" if v < 0 else str(int(v)) for v in b) for b in POLA_UNIK],
                       "Baris": JUMLAH_POLA, "Posterior": np.round(POST_REF, 4), "Kategori": KAT_REF}).sort_values("Baris", ascending=False)
_cetak("C1. Posterior per pola", _tabel, 40)

_HARAP = dict(HARAPAN_PAKAR)
_b = []
for _i, _pola in enumerate(POLA_UNIK):
    _aktif = tuple(k for _j, k in enumerate(INDIKATOR) if _pola[_j] == 1)
    if _aktif in _HARAP:
        _b.append({"Aktif": "+".join(_aktif), "Baris": int(JUMLAH_POLA[_i]), "Posterior": round(float(POST_REF[_i]), 4),
                   "Model": KAT_REF[_i], "Pakar": _HARAP[_aktif], "Sesuai": KAT_REF[_i] == _HARAP[_aktif]})
_cek = pd.DataFrame(_b)
_cetak("C2. Konsistensi dengan jawaban pakar", _cek)
if len(_cek):
    _t = int(_cek["Baris"].sum()); _o = int(_cek.loc[_cek["Sesuai"], "Baris"].sum())
    _cetak("C2b. Baris sesuai pakar", f"{_o:,} dari {_t:,} ({_o / _t * 100:.1f}%)")

_d = pd.Series({lab: int(JUMLAH_POLA[KAT_REF == lab].sum()) for lab in ["RENDAH", "SEDANG", "TINGGI", "BELUM DAPAT DINILAI"]})
_cetak("C3. Sebaran kategori Bayes", pd.DataFrame({"Baris": _d, "%": (_d / _n * 100).round(3)}))
if "Kategori Baseline" in df_clean.columns:
    _cetak("C3b. Sebaran kategori baseline", df_clean["Kategori Baseline"].value_counts(dropna=False))
    _cetak("C4. Tabel silang baseline x Bayes", pd.crosstab(df_clean["Kategori Baseline"], df_clean["Kategori Bayes"]))
    _cetak("C4b. Kesesuaian", f"sama: {float((df_clean['Kategori Baseline'] == df_clean['Kategori Bayes']).mean() * 100):.4f}% | "
                              f"tingkat skor unik Bayes: {df_clean['Posterior Bayes'].round(6).nunique()}")

_kol_e3 = "E3 semua pantauan" if "E3 semua pantauan" in df_clean.columns else "E3_rev"
if "ATURAN_E3" in globals() and _kol_e3 in df_clean.columns:
    _e3 = df_clean[_kol_e3].eq(1).fillna(False).to_numpy(dtype=bool)
    _mon = df_clean["Kategori Obat"].isin(list(ATURAN_E3)).to_numpy(dtype=bool)
    _besar = df_clean["Jumlah Besar E3"].eq(1).fillna(False).to_numpy(dtype=bool) if "Jumlah Besar E3" in df_clean.columns else np.zeros(_n, bool)
    _g = pd.DataFrame({"Obat": df_clean["Kategori Obat"].astype("string"), "Sarana": df_clean["Jenis Sarana"].astype("string"),
                       "E3=1": _e3, "Jumlah besar": _besar, "Pantauan": _mon})
    _cetak("D1. Obat pantauan: baris, jumlah besar, E3=1",
           _g[_g["Pantauan"]].groupby("Obat").agg(Baris=("E3=1", "size"), Jumlah_besar=("Jumlah besar", "sum"), E3_1=("E3=1", "sum")))
    _cetak("D2. E3=1 menurut obat x sarana", pd.crosstab(_g.loc[_g["E3=1"], "Obat"], _g.loc[_g["E3=1"], "Sarana"]))
    _rs = _g["Pantauan"].to_numpy() & _g["Jumlah besar"].to_numpy() & (_g["Sarana"].str.casefold() == "rumah sakit").fillna(False).to_numpy(dtype=bool)
    if _rs.any():
        _cetak("D3. Obat pantauan jumlah besar ke RS: kategori Bayes", pd.crosstab(_g.loc[_rs, "Obat"], df_clean.loc[_rs, "Kategori Bayes"]))
_tt = df_clean[df_clean["Kategori Bayes"].isin(["SEDANG", "TINGGI"])]
_cetak("D4. Sedang/Tinggi menurut Kategori Obat",
       pd.crosstab(_tt["Kategori Obat"], _tt["Kategori Bayes"]).assign(Total=lambda t: t.sum(axis=1)).sort_values("Total", ascending=False))
_cetak("D5. Sedang/Tinggi menurut Jenis Sarana",
       pd.crosstab(_tt["Jenis Sarana"], _tt["Kategori Bayes"]).assign(Total=lambda t: t.sum(axis=1)).sort_values("Total", ascending=False))
_kolom_top = [c for c in ["Nama PBF", "Kategori Obat", "Jenis Sarana", "Jumlah Setelah Konversi", "Posterior Bayes", "Kategori Bayes",
                          "Alasan E3", "Catatan Pendukung"] if c in df_clean.columns]
_cetak("D6. 10 transaksi posterior tertinggi", df_clean.sort_values("Posterior Bayes", ascending=False)[_kolom_top].head(10), 10)

_tampil("tabel_ambang", "E1. Ambang efektif per prior")
_tampil("tabel_oat", "E2. OAT per skema", 30)
_tampil("tabel_sapu_prior", "E3. Sapuan prior", 60)
_tampil("tabel_sapu_lr", "E4. Sapuan LR", 80)
_tampil("tabel_grid", "E5. Grid prior x pengali LR", 40)
_tampil("tabel_sapu_ambang", "E6. Sapuan ambang", 40)
_tampil("tabel_mc_ringkas", "E7. Monte Carlo ringkas", 30)
_tampil("tabel_mc_param", "E8. Monte Carlo korelasi parameter", 30)
_tampil("tabel_mc_pola", "E9. Monte Carlo stabilitas pola", 40)
_tampil("tabel_definisi", "E10. Uji definisi indikator", 30)
_tampil("tabel_ringkas", "E11. Tabel ringkas akhir", 60)

_tampil("tabel_pakar", "F1. Pakar vs model (T001-T012)", 20)
_tampil("ringkas_lr", "F2. Rentang LR konsisten dengan penilaian pakar", 20)
_tampil("tabel_diskrepansi", "G1. Diskrepansi elisitasi: saklar Cefixime x kedaluwarsa x E2k", 20)
_tampil("tabel_diskrepansi_T", "G2. Kategori transaksi T menurut skenario", 20)
_tampil("ringkas_temuan_anonim", "H1. Temuan data anomali menurut PBF (nama disamarkan)", 40)
if "Temuan data anomali" in df_clean.columns:
    _an = df_clean["Temuan data anomali"].fillna(False).to_numpy(dtype=bool)
    _cetak("H2. Temuan data anomali (dikeluarkan dari penilaian risiko)",
           f"{int(_an.sum())} transaksi; kategori Bayes: " + str(df_clean.loc[_an, "Kategori Bayes"].value_counts().to_dict()))
if "tabel_pakar" not in globals():
    _cetak("F0. Pemetaan pakar", "belum ada hasil: isi INDEKS_T atau CIRI_T di Bagian 5 lalu jalankan ulang sel itu")

_tampil("tabel_7_1", "I1. Distribusi kategori: Bayesian, Baseline 1, Baseline 2")
_tampil("tabel_7_2", "I2. Kesesuaian berpasangan antar metode")
_tampil("tabel_7_3", "I3. Tabel silang antar metode")
_tampil("tabel_7_4", "I4. Pola indikator yang kategorinya berbeda", 40)
_tampil("tabel_7_5", "I5. Daya pembeda (resolusi)")
_tampil("sebaran_kat", "I5b. Sebaran posterior di dalam kategori Bayesian")
_tampil("tabel_7_6", "I6. T001-T012: Bayesian, Baseline 1, Baseline 2 vs pakar", 20)
_tampil("tabel_7_7", "I7. T001-T012: ringkasan ukuran", 10)
_tampil("tabel_7_8", "I8. Lembar sampel pakar: ringkasan ukuran", 10)
_tampil("tabel_7_9", "I9. Lembar sampel pakar: Bayesian vs baseline berpasangan", 10)
_tampil("tabel_7_10", "I10. Lembar sampel pakar: menurut strata", 30)
_tampil("tabel_7_11", "I11. Lembar sampel pakar: alasan utama menurut penilaian pakar", 30)
_tampil("tabel_7_12", "I12. Lembar sampel pakar: selisih terbesar Bayesian vs pakar", 40)
_tampil("tabel_7_13", "I13. Konsistensi pakar pada transaksi duplikat", 30)
_cetak("I13b. Persen sampel yang kategori Bayes-nya berubah sejak lembar dibuat", str(globals().get("STALE_PERSEN")))
_tampil("tabel_7_13b", "I13c. Keterulangan pakar (intra-rater)", 5)
_tampil("tabel_7_16", "I13d. Konfirmasi aturan oleh pakar", 15)
_tampil("tabel_7_17", "I13e. Hanya penilaian yang diyakini pakar", 5)
_tampil("tabel_7_14", "I14. OAT: kecocokan dengan pakar per skenario", 80)
_tampil("tabel_7_14b", "I14b. Baseline: kecocokan dengan pakar", 10)
_tampil("tabel_7_15", "I15. OAT: ringkasan kepekaan kecocokan dengan pakar", 10)
if "indeks_final" in globals():
    _cetak("I0. Status pemetaan T", "terpetakan: " + ", ".join(sorted(indeks_final)) + " | belum: " +
           (", ".join(k for k in PAKAR if k not in indeks_final) or "tidak ada"))

try:
    with open("cek_data.txt", "w", encoding="utf-8") as _f:
        _f.write("\n".join(_KELUAR))
    print("\nTersimpan: cek_data.txt")
    try:
        from google.colab import files
        files.download("cek_data.txt")
    except Exception:
        pass
except Exception as _e:
    print(f"Gagal menyimpan txt: {_e}")

**TABEL DAN GAMBAR YANG DI BUTUHKAN UNTUK JURNAL**

Tabel dan gambar yang dipakai dalam artikel, dihitung langsung dari data.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

_ringkas = []

def _bandingkan(nama, aktual, harapan):
    """aktual dan harapan: dict {kunci: nilai}. Mencetak tabel Cek."""
    baris = []
    for k, h in harapan.items():
        a = aktual.get(k, np.nan)
        try:
            ok = (abs(float(a) - float(h)) < 0.00051)
        except Exception:
            ok = (str(a) == str(h))
        baris.append({"Butir": k, "Hasil kode": a, "Di artikel": h, "Cek": "SESUAI" if ok else "BEDA"})
    t = pd.DataFrame(baris)
    n_ok = int((t["Cek"] == "SESUAI").sum())
    print(f"-> {nama}: {n_ok} dari {len(t)} butir sesuai dengan artikel")
    if n_ok < len(t):
        display(t[t["Cek"] == "BEDA"])
    _ringkas.append((nama, len(t), n_ok))

_KAT = ["RENDAH", "SEDANG", "TINGGI"]

print("=== TABEL 1. Indikator model dan frekuensinya ===")
_t1 = pd.DataFrame([{"Indikator": k, "LR": LR_UTAMA.get(k), "Dinilai": int(X[k].notna().sum()),
                     "Bernilai 1": int((X[k] == 1).fillna(False).sum()),
                     "m": round(float((X[k] == 1).sum() / max(X[k].notna().sum(), 1)), 4)} for k in X.columns])
display(_t1)
_h1 = {"E1": (154929, 1991), "E2": (56326, 633), "E2p": (98374, 355), "E2k": (155016, 317), "E3": (20400, 1184), "E3b": (141045, 5387)}
_a = {}; _h = {}
for _, r in _t1.iterrows():
    if r["Indikator"] in _h1:
        _a[r["Indikator"] + " dinilai"] = r["Dinilai"]; _a[r["Indikator"] + " bernilai 1"] = r["Bernilai 1"]
        _h[r["Indikator"] + " dinilai"] = _h1[r["Indikator"]][0]; _h[r["Indikator"] + " bernilai 1"] = _h1[r["Indikator"]][1]
_bandingkan("Tabel 1", _a, _h)

print("\n=== TABEL 2. Posterior menurut pola indikator (sepuluh pola terbanyak) ===")
_t2 = pd.DataFrame({"Pola (" + "/".join(INDIKATOR) + ")": [" / ".join("NA" if v < 0 else str(int(v)) for v in b) for b in POLA_UNIK],
                    "Baris": JUMLAH_POLA, "Posterior": np.round(POST_REF, 4), "Kategori": KAT_REF})
_t2 = _t2[_t2["Kategori"] != "BELUM DAPAT DINILAI"].sort_values("Baris", ascending=False).reset_index(drop=True)
display(_t2.head(10))
print(f"Jumlah pola yang dapat dinilai: {len(_t2)} (artikel: 41); tingkat skor unik: {len(set(np.round(POST_REF[~np.isnan(POST_REF)], 6)))} (artikel: 41)")
_a = {"jumlah pola dinilai": len(_t2)}; _h = {"jumlah pola dinilai": 41}
_ar = [(83770, .1289), (44650, .1261), (9251, .1033), (7752, .1056), (5837, .1317), (5017, .5081), (1362, .3731), (1066, .7749), (424, .5463), (376, .4333)]
for i, (b_, p_) in enumerate(_ar, 1):
    if i <= len(_t2):
        _a[f"pola {i} baris"] = int(_t2.loc[i - 1, "Baris"]); _a[f"pola {i} posterior"] = float(_t2.loc[i - 1, "Posterior"])
    _h[f"pola {i} baris"] = b_; _h[f"pola {i} posterior"] = p_
_bandingkan("Tabel 2", _a, _h)

_tt = df_clean[df_clean["Kategori Bayes"].isin(["SEDANG", "TINGGI"])]
_t3 = pd.crosstab(_tt["Kategori Obat"], _tt["Kategori Bayes"]).reindex(columns=["SEDANG", "TINGGI"], fill_value=0)
_t3["Total"] = _t3.sum(axis=1)
print("\n=== TABEL 3. Sedang dan Tinggi menurut obat ===")
display(_t3.sort_values("Total", ascending=False))
_h3 = {"Cefixime": (4430, 100), "TBC": (1912, 238), "Ketamine": (1564, 207), "Orlistat": (10, 1002), "Meropenem": (326, 15)}
_a = {}; _h = {}
for o, (s_, t_) in _h3.items():
    _a[o + " sedang"] = int(_t3.loc[o, "SEDANG"]) if o in _t3.index else np.nan; _h[o + " sedang"] = s_
    _a[o + " tinggi"] = int(_t3.loc[o, "TINGGI"]) if o in _t3.index else np.nan; _h[o + " tinggi"] = t_
_bandingkan("Tabel 3", _a, _h)

_t4 = pd.crosstab(_tt["Jenis Sarana"], _tt["Kategori Bayes"]).reindex(columns=["SEDANG", "TINGGI"], fill_value=0)
_t4["Total"] = _t4.sum(axis=1)
print("\n=== TABEL 4. Sedang dan Tinggi menurut jenis sarana ===")
display(_t4.sort_values("Total", ascending=False))
_h4 = {"Rumah Sakit": (2138, 18), "Apotek": (5113, 1172), "Klinik": (785, 285), "PBF": (85, 82), "TUJUAN PENYALURAN TIDAK JELAS": (54, 2)}
_a = {}; _h = {}
for o, (s_, t_) in _h4.items():
    _a[o + " sedang"] = int(_t4.loc[o, "SEDANG"]) if o in _t4.index else np.nan; _h[o + " sedang"] = s_
    _a[o + " tinggi"] = int(_t4.loc[o, "TINGGI"]) if o in _t4.index else np.nan; _h[o + " tinggi"] = t_
_bandingkan("Tabel 4", _a, _h)

print("\n=== TABEL 5. Ringkasan kecocokan dengan pakar ===")
for _nm, _v in (("T001-T012 (tabel_7_7)", "tabel_7_7"), ("Sampel pakar n=60 (tabel_7_8)", "tabel_7_8")):
    if _v in globals():
        print(_nm); display(globals()[_v][[c for c in ["Metode", "n", "Cocok persis", "Kappa berbobot (linear)"] if c in globals()[_v].columns]])
    else:
        print(_nm, ": belum ada (jalankan Bagian 7 dengan lembar pakar terisi)")
_a = {}; _h = {}
if "tabel_7_7" in globals() and "tabel_7_8" in globals():
    _s7, _s8 = globals()["tabel_7_7"].set_index("Metode"), globals()["tabel_7_8"].set_index("Metode")
    _h.update({"T001-T012 Bayesian cocok": "9 (100.0%)", "Sampel Bayesian n": 60, "Sampel Bayesian cocok": "15 (25.0%)",
               "Sampel Baseline 1 cocok": "12 (20.0%)", "Sampel Baseline 2 cocok": "15 (25.0%)"})
    _a.update({"T001-T012 Bayesian cocok": _s7.loc["Bayesian", "Cocok persis"], "Sampel Bayesian n": _s8.loc["Bayesian", "n"],
               "Sampel Bayesian cocok": _s8.loc["Bayesian", "Cocok persis"], "Sampel Baseline 1 cocok": _s8.loc["Baseline 1", "Cocok persis"],
               "Sampel Baseline 2 cocok": _s8.loc["Baseline 2", "Cocok persis"]})
    for _k in list(_a): _a[_k] = str(_a[_k]) if "cocok" in _k else _a[_k]
    _kap = {"Sampel Bayesian kappa": (round(float(_s8.loc["Bayesian", "Kappa berbobot (linear)"]), 3), -0.027),
            "Sampel Baseline 1 kappa": (round(float(_s8.loc["Baseline 1", "Kappa berbobot (linear)"]), 3), -0.035)}
    for _k, (_x, _y) in _kap.items(): _a[_k] = _x; _h[_k] = _y
    _bandingkan("Tabel 5 (validasi 2 dan T001-T012)", _a, _h)
print("Catatan: baris 'Validasi 1' di artikel (n = 95; Bayesian 20 = 21,1%; Baseline 1 11 = 11,6%) dihitung dengan aturan model lama dan "
      "tidak dihitung ulang oleh notebook ini.")

print("\n=== GAMBAR 1. Sebaran kategori ===")
_g1 = globals()["tabel_7_1"].reindex(["RENDAH", "SEDANG", "TINGGI"])
_bay = _g1["Bayesian"].astype(int).tolist(); _b1 = _g1["Baseline 1 (jumlah indikator)"].astype(int).tolist()
display(pd.DataFrame({"Bayesian": _bay, "Baseline 1": _b1}, index=_KAT))
_x = np.arange(3); _w = .38
_fig, _ax = plt.subplots(figsize=(3.4, 2.3), dpi=200)
_r1 = _ax.bar(_x - _w / 2, _bay, _w, label="Bayesian", color="#1f4e79")
_r2 = _ax.bar(_x + _w / 2, _b1, _w, label="Baseline 1 (jumlah indikator)", color="#9aa5b1")
_ax.set_yscale("log"); _ax.set_xticks(_x); _ax.set_xticklabels(["Rendah", "Sedang", "Tinggi"])
_ax.set_ylabel("Jumlah transaksi (skala log)")
for _r in list(_r1) + list(_r2):
    _ax.text(_r.get_x() + _r.get_width() / 2, max(_r.get_height(), 1) * 1.12, f"{int(_r.get_height()):,}".replace(",", "."), ha="center", fontsize=6)
_ax.set_ylim(10, 1e6); _ax.legend(fontsize=6, frameon=False); _ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig("gambar1_sebaran_kategori.png"); plt.show()
_bandingkan("Gambar 1", dict(zip(["Bayes R", "Bayes S", "Bayes T", "Base1 R", "Base1 S", "Base1 T"], _bay + _b1)),
            dict(zip(["Bayes R", "Bayes S", "Bayes T", "Base1 R", "Base1 S", "Base1 T"], [151892, 8242, 1562, 151892, 9741, 63])))

print("\n=== GAMBAR 2. OAT skema A (persen transaksi berubah) ===")
_o = tabel_oat_semua[(tabel_oat_semua["Skema"] == "A") & (tabel_oat_semua["Jenis"] == "OAT")][["Skenario", "Baris berubah vs S0", "% berubah vs S0"]]
_o = _o[_o["% berubah vs S0"] > 0].sort_values("% berubah vs S0", ascending=False).head(8)
display(_o.reset_index(drop=True))
_oo = _o.sort_values("% berubah vs S0")
_fig, _ax = plt.subplots(figsize=(3.4, 2.0), dpi=200)
_ax.barh(_oo["Skenario"], _oo["% berubah vs S0"], color="#1f4e79")
for _i, _v in enumerate(_oo["% berubah vs S0"]): _ax.text(_v + .03, _i, f"{_v:.2f}".replace(".", ","), va="center", fontsize=6.5)
_ax.set_xlabel("Transaksi berubah (%)"); _ax.set_xlim(0, max(1.5, float(_oo["% berubah vs S0"].max()) * 1.25))
_ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.savefig("gambar2_oat_skema_A.png"); plt.show()
_hh = {"LR E1 = 2.5": 1.218, "Ambang Tinggi = 0.8": 0.927, "Prior = 0.05": 0.923, "Ambang Sedang = 0.4": 0.901,
       "LR E3 = 10.0": 0.729, "LR E2 = 25.0": 0.380, "LR E2p = 10.0": 0.198, "Ambang Tinggi = 0.6": 0.118}
_aa = {k: float(_o.set_index("Skenario")["% berubah vs S0"].get(k, np.nan)) for k in _hh}
_bandingkan("Gambar 2", _aa, _hh)

print("\n=== ANGKA UTAMA DI TEKS (bagian IV.A, IV.B, IV.E) ===")
_n = len(df_clean); _c = df_clean["Kategori Bayes"].value_counts()
_a = {"Rendah": int(_c.get("RENDAH", 0)), "Sedang": int(_c.get("SEDANG", 0)), "Tinggi": int(_c.get("TINGGI", 0)),
      "Belum dapat dinilai": int(_c.get("BELUM DAPAT DINILAI", 0)), "Tingkat skor unik": int(df_clean["Posterior Bayes"].nunique())}
_h = {"Rendah": 151892, "Sedang": 8242, "Tinggi": 1562, "Belum dapat dinilai": 32, "Tingkat skor unik": 41}
try:
    _p = globals()["tabel_7_2"].set_index("Pasangan")
    _a["Sama dg baseline 1 (%)"] = round(float(_p.loc["Bayesian vs Baseline 1", "Kategori sama (%)"]), 2); _h["Sama dg baseline 1 (%)"] = 99.07
    _a["Sama dg baseline 2 (%)"] = round(float(_p.loc["Bayesian vs Baseline 2", "Kategori sama (%)"]), 2); _h["Sama dg baseline 2 (%)"] = 100.0
except Exception as _e:
    print("tabel_7_2 tidak terbaca:", _e)
try:
    _mc = globals()["tabel_mc_ringkas"]
    _mc = pd.concat(list(_mc.values())) if isinstance(_mc, dict) else _mc
    _m = _mc.set_index("Skema")
    _a["MC A median % berubah"] = round(float(_m.loc["A", "Median % berubah vs referensi"]), 2); _h["MC A median % berubah"] = 2.35
    _kol = [k for k in _m.columns if "95% undian" in k][0]
    _a["MC A % stabil >=95% undian"] = round(float(_m.loc["A", _kol]), 2); _h["MC A % stabil >=95% undian"] = 93.94
except Exception as _e:
    print("tabel_mc_ringkas tidak terbaca:", _e)
_bandingkan("Angka teks", _a, _h)

print("\n=== RINGKASAN CEK ARTIKEL ===")
_rr = pd.DataFrame(_ringkas, columns=["Bagian", "Butir dicek", "Sesuai"]); _rr["Beda"] = _rr["Butir dicek"] - _rr["Sesuai"]
display(_rr)
print("SEMUA SESUAI" if int(_rr["Beda"].sum()) == 0 else f"Ada {int(_rr['Beda'].sum())} butir BEDA: kirim output sel ini ke Claude.")

# **Tabel suplemen artikel (S1–S4)**

Jalankan **paling akhir**, setelah Bagian 3 (model, termasuk Monte Carlo 6g dan ekspor 7) dan Bagian 7E. Sel ini hanya membaca objek yang sudah ada dan menulis `tabel_suplemen_artikel.xlsx`:

| Sheet | Isi |
|---|---|
| S1_Batas_IQR | Kuartil dan batas atas per kelompok E1 dan E3 |
| S2_Aturan_Kalibrasi | Frekuensi indikator, p0 dan p1 per prior |
| S3_Confusion_Validasi | Matriks pakar × model, sensitivitas, spesifisitas, kecocokan berbobot, dalam satu tingkat |
| S4_Stabilitas_Kategori | Stabilitas Monte Carlo per kategori referensi dan per pola |

Penyusunan tabel suplemen.

In [ ]:
import os
import numpy as np
import pandas as pd
from IPython.display import display

BERKAS_SUPLEMEN = "tabel_suplemen_artikel.xlsx"

_wajib = ["tabel_mc_pola", "tabel_mc_ringkas", "df_clean", "URUT7", "VALIDASI_7E", "KOLOM_NILAI_7E", "_kat_7e"]
_hilang = [n for n in _wajib if n not in globals()]
if _hilang:
    raise NameError(f"Belum ada: {_hilang}. Jalankan Bagian 3 (sampai ekspor), Bagian 7B dan 7E dahulu.")

lembar_suplemen = {}

for _nama, _var in [("S1_Batas_IQR_E1", "batas_e1"), ("S1_Batas_IQR_E3", "batas_e3")]:
    if _var in globals():
        lembar_suplemen[_nama] = globals()[_var]
    else:
        print(f"{_var} tidak ditemukan; {_nama} dilewati.")

for _nama, _var in [("S2_Frekuensi_Indikator", "tabel_frek"), ("S2_Kalibrasi_Prior", "tabel_kalibrasi"),
                    ("S2_Parameter_Referensi", "tabel_ref")]:
    if _var in globals():
        lembar_suplemen[_nama] = globals()[_var]
    else:
        print(f"{_var} tidak ditemukan; {_nama} dilewati.")

KEBALIK = {v: k for k, v in URUT7.items()}
URUT_LABEL = [KEBALIK[i] for i in sorted(KEBALIK)]
baris_conf, baris_ukuran = [], []
for nama_v, berkas_isi, berkas_kunci in VALIDASI_7E:
    if not (os.path.exists(berkas_isi) and os.path.exists(berkas_kunci)):
        print(f"{nama_v}: dilewati (berkas tidak ditemukan).")
        continue
    isi = pd.read_excel(berkas_isi, sheet_name="Penilaian", dtype=str).fillna("")
    kunci = pd.read_excel(berkas_kunci, sheet_name="Kunci")
    isi["Nilai pakar"] = isi[KOLOM_NILAI_7E].map(_kat_7e)
    g = kunci.merge(isi[["No", "Nilai pakar"]], left_on="ID lembar", right_on="No", how="left")
    g["Nilai pakar"] = g["Nilai pakar"].fillna("")
    g = g[(g["Duplikat uji ulang"] != "ya") & g["Nilai pakar"].isin(URUT7)]
    g = g[g["Indeks df_clean"].isin(df_clean.index)].copy()
    g["Model"] = df_clean.loc[g["Indeks df_clean"].to_numpy(), "Kategori Bayes"].astype(str).to_numpy()
    g = g[g["Model"].isin(URUT7)]
    if len(g) < 5:
        print(f"{nama_v}: terlalu sedikit penilaian ({len(g)}).")
        continue
    w = g["Bobot"].astype(float).to_numpy() if "Bobot" in g.columns else np.ones(len(g))
    p = g["Nilai pakar"].map(URUT7).to_numpy()
    m = g["Model"].map(URUT7).to_numpy()
    for a in URUT_LABEL:
        for b in URUT_LABEL:
            sel = (g["Nilai pakar"] == a).to_numpy() & (g["Model"] == b).to_numpy()
            baris_conf.append({"Validasi": nama_v, "Pakar": a, "Model": b,
                               "n": int(sel.sum()), "n berbobot": float(w[sel].sum())})
    rendah_p = p == URUT7[URUT_LABEL[0]]
    rendah_m = m == URUT7[URUT_LABEL[0]]
    def _rasio(a, b): return float(a / b) if b > 0 else np.nan
    baris_ukuran.append({
        "Validasi": nama_v, "n": int(len(g)),
        "Cocok persis": int((p == m).sum()),
        "Cocok persis (%)": round(float((p == m).mean() * 100), 1),
        "Cocok berbobot (%)": round(float(w[p == m].sum() / w.sum() * 100), 1),
        "Dalam satu tingkat (%)": round(float((np.abs(p - m) <= 1).mean() * 100), 1),
        "Model lebih tinggi": int((m > p).sum()), "Model lebih rendah": int((m < p).sum()),
        "Sensitivitas >=Sedang (tak berbobot)": round(_rasio((~rendah_p & ~rendah_m).sum(), (~rendah_p).sum()), 3),
        "Spesifisitas Rendah (tak berbobot)": round(_rasio((rendah_p & rendah_m).sum(), rendah_p.sum()), 3),
        "Sensitivitas >=Sedang (berbobot)": round(_rasio(w[~rendah_p & ~rendah_m].sum(), w[~rendah_p].sum()), 3),
        "Spesifisitas Rendah (berbobot)": round(_rasio(w[rendah_p & rendah_m].sum(), w[rendah_p].sum()), 3),
    })
if baris_conf:
    lembar_suplemen["S3_Confusion_Validasi"] = pd.DataFrame(baris_conf)
    lembar_suplemen["S3_Ukuran_Validasi"] = pd.DataFrame(baris_ukuran)
    for nama_v in pd.DataFrame(baris_conf)["Validasi"].unique():
        t = pd.DataFrame(baris_conf)
        t = t[t["Validasi"] == nama_v].pivot(index="Pakar", columns="Model", values="n").reindex(index=URUT_LABEL, columns=URUT_LABEL)
        print(f"Matriks {nama_v} (baris = pakar, kolom = model)")
        display(t)
    display(pd.DataFrame(baris_ukuran))

kol_stab = "Stabilitas (% undian = referensi)"
kol_pola = [c for c in tabel_mc_pola.columns if c.startswith("Pola (")][0]
baris_kat = []
for skema, t in tabel_mc_pola.groupby("Skema"):
    for kat, tk in t.groupby("Kategori referensi"):
        n = float(tk["Jumlah Baris"].sum())
        baris_kat.append({"Skema": skema, "Kategori referensi": kat, "Jumlah transaksi": int(n),
                          "Stabilitas rata-rata berbobot (%)": round(float((tk[kol_stab] * tk["Jumlah Baris"]).sum() / n), 1) if n > 0 else np.nan,
                          "Stabilitas terendah antar pola (%)": round(float(tk[kol_stab].min()), 1)})
tabel_stab_kat = pd.DataFrame(baris_kat)
lembar_suplemen["S4_Stabilitas_Kategori"] = tabel_stab_kat
lembar_suplemen["S4_Stabilitas_Pola"] = tabel_mc_pola
lembar_suplemen["S4_MC_Ringkasan"] = tabel_mc_ringkas
print("Stabilitas Monte Carlo per kategori referensi")
display(tabel_stab_kat)

try:
    with pd.ExcelWriter(BERKAS_SUPLEMEN, engine="openpyxl") as penulis:
        for nama, tabel in lembar_suplemen.items():
            tabel.to_excel(penulis, sheet_name=nama[:31], index=False)
    print(f"Disimpan: {BERKAS_SUPLEMEN} ({len(lembar_suplemen)} sheet)")
except Exception as galat:
    print(f"Ekspor Excel gagal ({galat}); menyimpan sebagai CSV.")
    for nama, tabel in lembar_suplemen.items():
        tabel.to_csv(f"suplemen_{nama}.csv", index=False)